# Final figures — self-contained Meta-control pipeline

This notebook follows:
1. `rnn_data_prep.ipynb`
2. `rnn_training.ipynb`
3. `meta_control_grid_search.ipynb`

All required project paths are under `~/Downloads/Meta-control`.

Before plotting, the notebook regenerates the RNN-until-contact predictions,
collision-stage labels, and mixed-model row-level table needed by Part I.
All figures are saved as 300 dpi PNGs under
`~/Downloads/Meta-control/final_figures`.


In [ ]:
# ============================================================
# Global figure output + text standard
# ============================================================
from pathlib import Path

HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"
import re
import warnings
import pandas as pd
import matplotlib.pyplot as plt

FINAL_FIG_DIR = BASE_DIR / "final_figures"
FINAL_FIG_DIR.mkdir(parents=True, exist_ok=True)

TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_SIZE = 14
BASE_FONT_SIZE = 14

plt.rcParams.update({
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "legend.title_fontsize": LEGEND_SIZE,
    "figure.dpi": 120,
})

_original_final_plt_show = plt.show
_final_figure_name_counts = {}
_final_figure_manifest = []
FIGURE_PREFIX = "figure"

def _clean_figure_filename(title):
    name = str(title).strip().lower()
    name = (
        name
        .replace("meta-control", "meta_control")
        .replace("−", "minus")
        .replace("–", "-")
    )
    name = re.sub(r"[^a-z0-9]+", "_", name).strip("_")
    return name or "figure"

def set_figure_prefix(prefix):
    global FIGURE_PREFIX
    FIGURE_PREFIX = _clean_figure_filename(prefix)

def _next_figure_path(fig, fig_num):
    explicit_name = getattr(fig, "_final_filename", None)
    if explicit_name:
        filename = str(explicit_name)
        if not filename.lower().endswith(".png"):
            filename += ".png"
        return FINAL_FIG_DIR / filename

    title = next(
        (
            ax.get_title()
            for ax in fig.axes
            if hasattr(ax, "get_title") and ax.get_title().strip()
        ),
        f"figure_{fig_num}",
    )
    stem = f"{FIGURE_PREFIX}_{_clean_figure_filename(title)}"
    count = _final_figure_name_counts.get(stem, 0) + 1
    _final_figure_name_counts[stem] = count
    if count > 1:
        stem = f"{stem}_{count}"
    return FINAL_FIG_DIR / f"{stem}.png"

def _auto_save_final_show(*args, **kwargs):
    for fig_num in plt.get_fignums():
        fig = plt.figure(fig_num)
        if getattr(fig, "_final_already_saved", False):
            continue

        output_path = _next_figure_path(fig, fig_num)
        fig.savefig(
            output_path,
            dpi=300,
            bbox_inches="tight",
        )
        fig._final_already_saved = True
        _final_figure_manifest.append({
            "order": len(_final_figure_manifest) + 1,
            "title": next(
                (
                    ax.get_title()
                    for ax in fig.axes
                    if hasattr(ax, "get_title") and ax.get_title().strip()
                ),
                "",
            ),
            "path": str(output_path),
        })
        print(f"Saved: {output_path}")

    return _original_final_plt_show(*args, **kwargs)

plt.show = _auto_save_final_show

print("All figures will be saved to:", FINAL_FIG_DIR)


# RNN analysis data generation

This section regenerates the three RNN-analysis inputs used by Part I directly
from the trained checkpoint and `rnn_testing_data`:

- `rnn_analysis/rnn_predictions_until_contact.csv`
- `rnn_analysis/collision_labels/rnn_labeled_container_or_line_circle_phases.csv`
- `rnn_analysis/mixed_model/mixed_model_row_level_data.csv`

No separate RNN-analysis notebook is required.


In [ ]:
# ============================================================
# 1. CPU-only setup and best-epoch checkpoint configuration
# ============================================================

import os

# Hide GPUs before importing torch so this notebook is CPU-only.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

from pathlib import Path
import json
import shutil
import time

import numpy as np
import pandas as pd
from PIL import Image

import torch
from torch import nn

import matplotlib.pyplot as plt
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D


HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"


CHECKPOINT_PATH = BASE_DIR / "rnn_training_results" / "lambda_0_2_offset0_everypoint_sliding15_nonfreeze_windows20_epochs150" / "best_int_checkpoint_rnn_model.pt"

TEST_DATA_ROOT = BASE_DIR / "rnn_testing_data"
TEST_EXP_FOLDERS = ["exp1", "exp2"]

DEFAULT_FRAME_CACHE_DIR = BASE_DIR / "compressed_frame_cache_100x128"
DEFAULT_BALL_CACHE_DIR = BASE_DIR / "ball_position_cache"

ANALYSIS_ROOT = BASE_DIR / "rnn_analysis"
OUTPUT_DIR = ANALYSIS_ROOT / "optional_overlay_plots"

# Remove old PNGs so the folder contains only results from this run.
RESET_OUTPUT_DIR = False

# Plot every 10th predicted point plus the terminal-contact point.
# The model still evaluates every integer horizon, +1 at a time.
DEFAULT_OVERLAY_STRIDE = 10


# Terminal-contact geometry.
# The predicted coordinates are ball-center coordinates.
# Stop when the predicted center is within 20 px of the goal or 40 px of the ground.
GOAL_CONTACT_DISTANCE_PX = 20.0
GROUND_CONTACT_DISTANCE_PX = 40.0
DEFAULT_GOAL_HALF_WIDTH_PX = 40.0   # 80-pixel-wide goal box
DEFAULT_GOAL_HALF_HEIGHT_PX = 20.0  # 40-pixel-high goal box


# This is not a model stopping rule. It only prevents an accidental
# infinite loop if a scene's predictions never approach either target.
# If reached, the notebook raises an error and does not label the final
# evaluated point as a valid prediction endpoint.
MAX_PREDICTION_HORIZON = 10000

DEVICE = torch.device("cpu")
CPU_THREADS = min(4, os.cpu_count() or 1)
torch.set_num_threads(CPU_THREADS)

if not CHECKPOINT_PATH.exists():
    raise FileNotFoundError(
        "The requested checkpoint was not found:\n"
        f"  {CHECKPOINT_PATH}\n"
        "This notebook intentionally does not fall back to another run."
    )

if not TEST_DATA_ROOT.exists():
    raise FileNotFoundError(f"Missing testing-data folder: {TEST_DATA_ROOT}")

try:
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=False,
    )
except TypeError:
    # Compatibility with older torch versions.
    checkpoint = torch.load(CHECKPOINT_PATH, map_location="cpu")

checkpoint_config = checkpoint.get("config", {})

EXPECTED_CHECKPOINT_EPOCH = 150
EXPECTED_BEST_TEST_TOTAL_LOSS = 0.0005792553347419016

loaded_epoch = checkpoint.get("epoch")
loaded_best_loss = checkpoint.get("best_test_total_loss")

if loaded_epoch != EXPECTED_CHECKPOINT_EPOCH:
    raise RuntimeError(
        "The selected best checkpoint is not the expected completed-run model.\n"
        f"Expected epoch: {EXPECTED_CHECKPOINT_EPOCH}\n"
        f"Loaded epoch: {loaded_epoch}"
    )

if loaded_best_loss is None or not np.isclose(
    float(loaded_best_loss),
    EXPECTED_BEST_TEST_TOTAL_LOSS,
    rtol=1e-7,
    atol=1e-12,
):
    raise RuntimeError(
        "The selected checkpoint's best held-out loss does not match "
        "the completed training notebook.\n"
        f"Expected: {EXPECTED_BEST_TEST_TOTAL_LOSS}\n"
        f"Loaded: {loaded_best_loss}"
    )

N_HISTORY = int(checkpoint_config.get("N_HISTORY", 15))
IMAGE_H = int(checkpoint_config.get("IMAGE_H", 128))
IMAGE_W = int(checkpoint_config.get("IMAGE_W", 100))
HIDDEN_CHANNELS = int(checkpoint_config.get("HIDDEN_CHANNELS", 96))
TIME_EMBED_DIM = int(checkpoint_config.get("TIME_EMBED_DIM", 32))
OVERLAY_STRIDE = int(
    checkpoint_config.get("OVERLAY_STRIDE", DEFAULT_OVERLAY_STRIDE)
)

ORIGINAL_FRAME_WIDTH = float(
    checkpoint_config.get("ORIGINAL_FRAME_WIDTH", 800.0)
)
ORIGINAL_FRAME_HEIGHT = float(
    checkpoint_config.get("ORIGINAL_FRAME_HEIGHT", 1024.0)
)
COORD_SCALE = np.array(
    [ORIGINAL_FRAME_WIDTH, ORIGINAL_FRAME_HEIGHT],
    dtype=np.float32,
)
ERR_MAG_PIXEL_SCALE_FOR_PLOTS = float(np.mean(COORD_SCALE))

if "GLOBAL_MAX_FUTURE_OFFSET" not in checkpoint_config:
    raise KeyError(
        "Checkpoint config is missing GLOBAL_MAX_FUTURE_OFFSET. "
        "That value is required to reproduce the model's time normalization."
    )
GLOBAL_MAX_FUTURE_OFFSET = float(
    max(1, int(checkpoint_config["GLOBAL_MAX_FUTURE_OFFSET"]))
)

def existing_config_path(key, default_path):
    raw = checkpoint_config.get(key)
    if raw:
        candidate = Path(raw).expanduser()
        if candidate.exists():
            return candidate
    return Path(default_path)

FRAME_CACHE_DIR = existing_config_path(
    "FRAME_CACHE_DIR",
    DEFAULT_FRAME_CACHE_DIR,
)
BALL_CACHE_DIR = existing_config_path(
    "BALL_CACHE_DIR",
    DEFAULT_BALL_CACHE_DIR,
)

if N_HISTORY != 15:
    raise RuntimeError(
        f"The checkpoint reports N_HISTORY={N_HISTORY}, not 15. "
        "This notebook is intended for the trained sliding-15 model."
    )

if not FRAME_CACHE_DIR.exists():
    raise FileNotFoundError(f"Missing frame-cache folder: {FRAME_CACHE_DIR}")

ANALYSIS_ROOT.mkdir(parents=True, exist_ok=True)
if RESET_OUTPUT_DIR and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Python/torch CPU setup")
print("----------------------")
print("torch version:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("device:", DEVICE)
print("CPU threads:", CPU_THREADS)
print()
print("Checkpoint:", CHECKPOINT_PATH)
print("Checkpoint epoch:", checkpoint.get("epoch", "unknown"))
print(
    "Verified completed-run best checkpoint:",
    loaded_epoch == EXPECTED_CHECKPOINT_EPOCH,
)
print("Best test loss:", checkpoint.get("best_test_total_loss", "unknown"))
print("N_HISTORY:", N_HISTORY)
print("GLOBAL_MAX_FUTURE_OFFSET:", GLOBAL_MAX_FUTURE_OFFSET)
print("Frame cache:", FRAME_CACHE_DIR)
print("Ball cache:", BALL_CACHE_DIR)
print("Testing data:", TEST_DATA_ROOT)
print("Output folder:", OUTPUT_DIR)

In [ ]:
# ============================================================
# 2. Scene discovery and cache helpers
# ============================================================

def scene_rel_for_cache(scene_dir):
    path = Path(scene_dir)
    try:
        return path.resolve().relative_to(BASE_DIR.resolve())
    except Exception:
        return path


def safe_scene_id(scene_dir):
    return (
        str(scene_rel_for_cache(scene_dir))
        .replace("/", "__")
        .replace("\\", "__")
        .replace(":", "")
    )


def compressed_frame_cache_path(scene_dir):
    return FRAME_CACHE_DIR / (
        f"{safe_scene_id(scene_dir)}"
        f"__frames_{IMAGE_H}x{IMAGE_W}_rgb_uint8.npy"
    )


def ball_cache_path(scene_dir):
    return BALL_CACHE_DIR / (
        f"{safe_scene_id(scene_dir)}"
        "__ball_positions_from_frames.csv"
    )


def get_frame_files(scene_dir):
    frames_dir = Path(scene_dir) / "frames"
    files = sorted(frames_dir.glob("frame_*.png"))
    if not files:
        files = sorted(frames_dir.glob("*.png"))
    return files


def load_true_positions(scene_dir):
    """
    Prefer the trained central ball-position cache used during training.
    Fall back to simulation_dataset.csv only if the central cache is absent.
    """
    central_path = ball_cache_path(scene_dir)

    if central_path.exists():
        df = pd.read_csv(central_path)
    else:
        simulation_path = Path(scene_dir) / "simulation_dataset.csv"
        if not simulation_path.exists():
            raise FileNotFoundError(
                f"No ball-position cache or simulation CSV for {scene_dir}"
            )
        df = pd.read_csv(simulation_path)

    required = {"ball_x", "ball_y"}
    missing = required - set(df.columns)
    if missing:
        raise RuntimeError(
            f"Position table for {scene_dir} is missing: {sorted(missing)}"
        )

    if "frame_index" in df.columns:
        df = df.sort_values("frame_index")
    elif "frame" in df.columns:
        df = df.sort_values("frame")

    df = df.dropna(subset=["ball_x", "ball_y"]).reset_index(drop=True)
    return df



def first_finite_scene_value(
    df,
    candidate_names,
    default=None,
    required=False,
):
    """
    Return the first finite numeric value found among candidate columns.
    """
    for column in candidate_names:
        if column not in df.columns:
            continue

        values = pd.to_numeric(
            df[column],
            errors="coerce",
        )
        values = values[np.isfinite(values)]

        if len(values) > 0:
            return float(values.iloc[0])

    if required:
        raise KeyError(
            "Could not find a finite value for any of these columns:\n"
            f"  {candidate_names}\n"
            f"Available columns:\n"
            f"  {df.columns.tolist()}"
        )

    return default


def as_xy(value, label="position"):
    """
    Parse the same x-y formats used by the trained scene generator.
    """
    original = value

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if isinstance(value, dict):
        for key in (
            "position",
            "pos",
            "point",
            "location",
        ):
            if key in value:
                value = value[key]
                break

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if (
        isinstance(value, (list, tuple))
        and len(value) >= 2
        and isinstance(value[0], (int, float))
        and isinstance(value[1], (int, float))
    ):
        return float(value[0]), float(value[1])

    raise ValueError(
        f"Could not parse {label} as x,y. "
        f"Got: {original!r}"
    )


def find_scene_json_path(scene_dir, scene_df):
    """
    Locate the actual scene-configuration JSON.

    The trained testing generator copied the original scene JSON into
    each testing scene directory. Prefer that local copy so this remains
    valid even if the original source path later moves.
    """
    scene_dir = Path(scene_dir)

    excluded_names = {
        "metadata.json",
        "static_scene_settings.json",
    }

    local_candidates = [
        path
        for path in sorted(scene_dir.glob("*.json"))
        if path.name not in excluded_names
    ]

    # Prefer a JSON whose stem matches the scene folder name.
    exact_local = [
        path
        for path in local_candidates
        if path.stem == scene_dir.name
    ]
    if exact_local:
        return exact_local[0]

    if len(local_candidates) == 1:
        return local_candidates[0]

    # metadata.json records the copied JSON as json_file.
    metadata_path = scene_dir / "metadata.json"
    if metadata_path.exists():
        with open(metadata_path, "r") as file:
            metadata = json.load(file)

        for key in ("json_file", "output_json", "source_json"):
            raw_path = metadata.get(key)
            if not raw_path:
                continue

            candidate = Path(raw_path).expanduser()
            if candidate.exists():
                return candidate

            local_by_name = scene_dir / candidate.name
            if local_by_name.exists():
                return local_by_name

    # static_scene_settings.json also records JSON paths.
    settings_path = scene_dir / "static_scene_settings.json"
    if settings_path.exists():
        with open(settings_path, "r") as file:
            settings = json.load(file)

        for key in ("output_json", "source_json", "json_file"):
            raw_path = settings.get(key)
            if not raw_path:
                continue

            candidate = Path(raw_path).expanduser()
            if candidate.exists():
                return candidate

            local_by_name = scene_dir / candidate.name
            if local_by_name.exists():
                return local_by_name

    # Last fallback: source_json repeated in simulation_dataset.csv.
    if "source_json" in scene_df.columns:
        source_values = (
            scene_df["source_json"]
            .dropna()
            .astype(str)
            .str.strip()
        )

        for raw_path in source_values:
            if not raw_path:
                continue

            candidate = Path(raw_path).expanduser()
            if candidate.exists():
                return candidate

            local_by_name = scene_dir / candidate.name
            if local_by_name.exists():
                return local_by_name

    raise FileNotFoundError(
        "Could not locate the scene-configuration JSON for:\n"
        f"  {scene_dir}\n"
        f"Local JSON candidates were:\n"
        f"  {[str(path) for path in local_candidates]}"
    )


def get_json_bottom_border(scene_data):
    """
    Parse the bottom-border endpoints exactly as the generator did.
    """
    args = scene_data.get("bottom_border_args")

    if isinstance(args, list) and len(args) >= 2:
        return (
            np.array(
                as_xy(
                    args[0],
                    "bottom_border_args[0]",
                ),
                dtype=np.float64,
            ),
            np.array(
                as_xy(
                    args[1],
                    "bottom_border_args[1]",
                ),
                dtype=np.float64,
            ),
        )

    return (
        np.array([10.0, 990.0], dtype=np.float64),
        np.array([790.0, 990.0], dtype=np.float64),
    )


def load_scene_geometry(scene_dir):
    """
    Load geometry using the same inputs as the trained generator.

    Goal:
      original goal position from JSON goal_args
      + goal_y_offset_applied from simulation_dataset.csv

    Ground:
      finite segment from JSON bottom_border_args

    The physical goal is an 80 x 40 axis-aligned rectangle.
    """
    scene_dir = Path(scene_dir)
    csv_path = scene_dir / "simulation_dataset.csv"

    if not csv_path.exists():
        raise FileNotFoundError(
            f"Missing scene data CSV: {csv_path}"
        )

    scene_df = pd.read_csv(csv_path)
    scene_json_path = find_scene_json_path(
        scene_dir,
        scene_df,
    )

    with open(scene_json_path, "r") as file:
        scene_data = json.load(file)

    if "goal_args" not in scene_data:
        raise KeyError(
            f"{scene_json_path} does not contain goal_args."
        )

    original_goal_x, original_goal_y = as_xy(
        scene_data["goal_args"],
        "goal_args",
    )

    goal_y_offset = first_finite_scene_value(
        scene_df,
        ["goal_y_offset_applied"],
        default=0.0,
    )

    goal_center = np.array(
        [
            original_goal_x,
            original_goal_y + goal_y_offset,
        ],
        dtype=np.float64,
    )

    ground_start, ground_end = get_json_bottom_border(
        scene_data
    )

    return {
        "scene_json_path": scene_json_path,
        "goal_y_offset_applied": float(goal_y_offset),
        "goal_center": goal_center,
        "goal_half_width": DEFAULT_GOAL_HALF_WIDTH_PX,
        "goal_half_height": DEFAULT_GOAL_HALF_HEIGHT_PX,
        "ground_start": ground_start,
        "ground_end": ground_end,
    }



def point_to_segment_distance(point, segment_start, segment_end):
    """
    Euclidean distance from a point to a finite line segment.
    """
    point = np.asarray(point, dtype=np.float64)
    segment_start = np.asarray(segment_start, dtype=np.float64)
    segment_end = np.asarray(segment_end, dtype=np.float64)

    segment_vector = segment_end - segment_start
    segment_length_squared = float(
        np.dot(segment_vector, segment_vector)
    )

    if segment_length_squared <= 0:
        return float(np.linalg.norm(point - segment_start))

    projection = float(
        np.dot(point - segment_start, segment_vector)
        / segment_length_squared
    )
    projection = float(np.clip(projection, 0.0, 1.0))

    closest_point = (
        segment_start + projection * segment_vector
    )
    return float(np.linalg.norm(point - closest_point))


def distance_to_axis_aligned_rectangle(
    point,
    rectangle_center,
    half_width,
    half_height,
):
    """
    Distance from a point to an axis-aligned rectangle.

    Returns 0 when the point lies inside the rectangle.
    """
    point = np.asarray(point, dtype=np.float64)
    rectangle_center = np.asarray(
        rectangle_center,
        dtype=np.float64,
    )

    delta = np.abs(point - rectangle_center)
    outside_x = max(float(delta[0] - half_width), 0.0)
    outside_y = max(float(delta[1] - half_height), 0.0)

    return float(np.hypot(outside_x, outside_y))


def predicted_terminal_event(predicted_center_xy, scene_geometry):
    """
    Check whether a predicted ball center touches the goal or ground.

    Goal contact:
        distance from center to the 80 x 40 goal rectangle <= 20 px

    Ground contact:
        distance from center to the finite ground segment <= 40 px

    Goal is checked first if both contacts occur at the same frame.
    """
    goal_distance = distance_to_axis_aligned_rectangle(
        point=predicted_center_xy,
        rectangle_center=scene_geometry["goal_center"],
        half_width=scene_geometry["goal_half_width"],
        half_height=scene_geometry["goal_half_height"],
    )

    if goal_distance <= GOAL_CONTACT_DISTANCE_PX:
        return "goal"

    ground_distance = point_to_segment_distance(
        point=predicted_center_xy,
        segment_start=scene_geometry["ground_start"],
        segment_end=scene_geometry["ground_end"],
    )

    if ground_distance <= GROUND_CONTACT_DISTANCE_PX:
        return "ground"

    return None


def truncate_predictions_at_first_contact(
    checkpoint_offsets,
    predicted_position_px,
    predicted_error_px,
    scene_geometry,
):
    """
    Keep offset 0 and all predictions through the first positive-offset
    predicted goal/ground contact. Predictions after contact are discarded.
    """
    terminal_index = None
    terminal_event = None

    for prediction_index in range(1, len(checkpoint_offsets)):
        event = predicted_terminal_event(
            predicted_position_px[prediction_index],
            scene_geometry,
        )

        if event is not None:
            terminal_index = prediction_index
            terminal_event = event
            break

    if terminal_index is None:
        terminal_index = len(checkpoint_offsets) - 1

    keep_through = terminal_index + 1

    truncated_offsets = checkpoint_offsets[:keep_through]
    truncated_positions = predicted_position_px[:keep_through]
    truncated_errors = predicted_error_px[:keep_through]

    terminal_offset = (
        int(checkpoint_offsets[terminal_index])
        if terminal_event is not None
        else None
    )

    return {
        "checkpoint_offsets": truncated_offsets,
        "predicted_position_px": truncated_positions,
        "predicted_error_px": truncated_errors,
        "terminal_event": terminal_event,
        "terminal_offset": terminal_offset,
    }

def find_test_scenes():
    scenes = []

    for exp_name in TEST_EXP_FOLDERS:
        exp_dir = TEST_DATA_ROOT / exp_name
        if not exp_dir.exists():
            print(f"Warning: missing test folder {exp_dir}")
            continue

        for scene_dir in sorted(exp_dir.iterdir()):
            if not scene_dir.is_dir():
                continue
            if not (scene_dir / "frames").exists():
                continue

            frame_cache = compressed_frame_cache_path(scene_dir)
            if not frame_cache.exists():
                raise FileNotFoundError(
                    "Missing compressed-frame cache for test scene:\n"
                    f"  scene: {scene_dir}\n"
                    f"  expected cache: {frame_cache}"
                )

            cached_shape = np.load(frame_cache, mmap_mode="r").shape
            positions = load_true_positions(scene_dir)
            n_frames = min(len(positions), int(cached_shape[0]))

            # Need the 15 input frames plus at least one future frame.
            if n_frames <= N_HISTORY:
                print(
                    f"Skipping too-short scene {scene_dir.name}: "
                    f"{n_frames} frames"
                )
                continue

            scenes.append(scene_dir)

    if not scenes:
        raise RuntimeError("No usable testing scenes were found.")

    print(f"Usable testing scenes: {len(scenes)}")
    if len(scenes) != 106:
        print(
            "Warning: the trained testing set was expected to contain "
            f"106 scenes, but {len(scenes)} usable scenes were found."
        )

    return scenes


test_scene_dirs = find_test_scenes()

print("\nFirst few scenes:")
for scene_dir in test_scene_dirs[:8]:
    print(" ", scene_dir)

In [ ]:
# ============================================================
# 3. Exact trained layers, reusable encoding, and checkpoint load
# ============================================================

class InTCheckpointRNN(nn.Module):
    def __init__(self, hidden_channels=96, time_embed_dim=32):
        super().__init__()

        self.hidden_channels = hidden_channels
        self.time_embed_dim = time_embed_dim

        self.input_conv = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=5, stride=2, padding=2),
            nn.ReLU(inplace=True),
            nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(
                64,
                hidden_channels,
                kernel_size=3,
                stride=1,
                padding=1,
            ),
            nn.ReLU(inplace=True),
        )

        self.recurrent_conv = nn.Conv2d(
            hidden_channels,
            hidden_channels,
            kernel_size=3,
            stride=1,
            padding=1,
            groups=hidden_channels,
            bias=False,
        )

        self.gate_conv = nn.Conv2d(
            hidden_channels * 2,
            hidden_channels,
            kernel_size=1,
        )
        self.hidden_bias = nn.Parameter(
            torch.zeros(1, hidden_channels, 1, 1)
        )

        self.attn_conv = nn.Conv2d(
            hidden_channels,
            1,
            kernel_size=1,
        )

        self.scene_mlp = nn.Sequential(
            nn.Linear(hidden_channels, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.05),
            nn.Linear(192, 192),
            nn.ReLU(inplace=True),
        )

        self.time_mlp = nn.Sequential(
            nn.Linear(4, time_embed_dim),
            nn.ReLU(inplace=True),
            nn.Linear(time_embed_dim, time_embed_dim),
            nn.ReLU(inplace=True),
        )

        decoder_in = 192 + time_embed_dim
        self.checkpoint_decoder = nn.Sequential(
            nn.Linear(decoder_in, 192),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.05),
            nn.Linear(192, 96),
            nn.ReLU(inplace=True),
        )

        self.position_head = nn.Linear(96, 2)
        self.error_head = nn.Linear(96, 1)

    def _attention_pool(self, hidden):
        batch, channels, height, width = hidden.shape
        logits = self.attn_conv(hidden).view(
            batch,
            1,
            height * width,
        )
        weights = torch.softmax(logits, dim=-1)
        hidden_flat = hidden.view(
            batch,
            channels,
            height * width,
        )
        return (hidden_flat * weights).sum(dim=-1)

    def _time_features(self, checkpoint_times):
        """
        Use the same four time features used during training.

        The original implementation capped tau at 1.5. That ceiling is
        removed here because it would make all longer horizons identical,
        preventing a genuine +1, +2, +3, ... search beyond that point.

        This does not change predictions in the trained range; it only
        permits extrapolation to longer horizons.
        """
        tau = checkpoint_times.clamp(min=0.0)

        return torch.stack(
            [
                tau,
                tau ** 2,
                torch.sin(np.pi * tau),
                torch.cos(np.pi * tau),
            ],
            dim=-1,
        )

    def encode_history(self, x):
        """
        Encode the 15-frame history once and return its scene code.
        """
        batch, n_frames, channels, height, width = x.shape

        hidden = None

        for frame_index in range(n_frames):
            z_t = self.input_conv(x[:, frame_index])

            if hidden is None:
                hidden = torch.zeros_like(z_t)

            candidate = torch.tanh(
                z_t
                + self.recurrent_conv(hidden)
                + self.hidden_bias
            )
            gate = torch.sigmoid(
                self.gate_conv(
                    torch.cat([z_t, hidden], dim=1)
                )
            )
            hidden = (
                gate * candidate
                + (1.0 - gate) * hidden
            )

        pooled = self._attention_pool(hidden)
        return self.scene_mlp(pooled)

    def decode_horizons(
        self,
        scene_code,
        checkpoint_times,
    ):
        """
        Decode one or more requested horizons from an encoded scene.
        """
        time_features = self._time_features(checkpoint_times)
        time_code = self.time_mlp(time_features)

        n_checkpoints = checkpoint_times.shape[1]
        expanded_scene_code = scene_code.unsqueeze(1).expand(
            scene_code.shape[0],
            n_checkpoints,
            scene_code.shape[-1],
        )

        decoder_input = torch.cat(
            [expanded_scene_code, time_code],
            dim=-1,
        )
        decoded = self.checkpoint_decoder(decoder_input)

        predicted_positions = self.position_head(decoded)
        predicted_error_raw = self.error_head(decoded)

        return predicted_positions, predicted_error_raw

    def forward(
        self,
        x,
        checkpoint_times,
        return_hidden=False,
    ):
        if return_hidden:
            raise ValueError(
                "return_hidden is not used in this inference notebook."
            )

        scene_code = self.encode_history(x)
        return self.decode_horizons(
            scene_code,
            checkpoint_times,
        )


def positive_error_magnitude(predicted_error_raw):
    return nn.functional.softplus(
        predicted_error_raw
    ).squeeze(-1)


model = InTCheckpointRNN(
    hidden_channels=HIDDEN_CHANNELS,
    time_embed_dim=TIME_EMBED_DIM,
).to(DEVICE)

state_dict = checkpoint.get(
    "model_state_dict",
    checkpoint,
)

# Handle checkpoints saved from torch.nn.DataParallel.
if state_dict and all(
    key.startswith("module.")
    for key in state_dict
):
    state_dict = {
        key.removeprefix("module."): value
        for key, value in state_dict.items()
    }

model.load_state_dict(state_dict, strict=True)
model.eval()

parameter_count = sum(
    parameter.numel()
    for parameter in model.parameters()
)

print(model)
print("\nParameter count:", parameter_count)
print("Checkpoint loaded strictly.")
print("History is encoded once per scene.")
print("Future horizons are evaluated one at a time: +1, +2, +3, ...")
print("Upper time clamp removed for genuine long-horizon extrapolation.")

In [ ]:
# ============================================================
# 4. Increment horizon by +1 until predicted terminal contact
# ============================================================

def load_first_history_tensor(scene_dir):
    cache_path = compressed_frame_cache_path(scene_dir)
    frames = np.load(cache_path, mmap_mode="r")

    expected_tail = (3, IMAGE_H, IMAGE_W)
    if (
        frames.ndim != 4
        or tuple(frames.shape[1:]) != expected_tail
    ):
        raise RuntimeError(
            f"Unexpected frame-cache shape for {scene_dir}:\n"
            f"  found: {frames.shape}\n"
            f"  expected: (T, {expected_tail[0]}, "
            f"{expected_tail[1]}, {expected_tail[2]})"
        )

    if frames.shape[0] < N_HISTORY:
        raise RuntimeError(
            f"{scene_dir} has fewer than {N_HISTORY} cached frames."
        )

    history = np.array(
        frames[:N_HISTORY],
        dtype=np.float32,
        copy=True,
    )
    history /= 255.0

    return torch.from_numpy(
        history
    ).unsqueeze(0).to(DEVICE)


def predict_scene_until_contact(scene_dir):
    """
    Encode the first 15 frames once.

    Then query exactly one increasingly longer horizon at a time:
    +1, +2, +3, ...

    The recorded scene length is used only for plotting the available
    ground-truth trajectory. It never limits the predicted horizon.
    """
    true_df = load_true_positions(scene_dir)
    scene_geometry = load_scene_geometry(scene_dir)

    cache_path = compressed_frame_cache_path(scene_dir)
    cached_frame_count = int(
        np.load(cache_path, mmap_mode="r").shape[0]
    )

    true_frame_count = min(
        len(true_df),
        cached_frame_count,
    )
    true_df = true_df.iloc[
        :true_frame_count
    ].reset_index(drop=True)

    input_last_frame = N_HISTORY - 1

    if true_frame_count <= input_last_frame:
        raise RuntimeError(
            f"{scene_dir} does not contain the full "
            f"{N_HISTORY}-frame history."
        )

    history_tensor = load_first_history_tensor(scene_dir)

    predicted_positions = []
    predicted_errors = []
    predicted_offsets = []

    with torch.inference_mode():
        # The expensive image history is encoded only once.
        scene_code = model.encode_history(history_tensor)

        for future_offset in range(
            1,
            MAX_PREDICTION_HORIZON + 1,
        ):
            normalized_time = (
                float(future_offset)
                / GLOBAL_MAX_FUTURE_OFFSET
            )

            time_tensor = torch.tensor(
                [[normalized_time]],
                dtype=torch.float32,
                device=DEVICE,
            )

            (
                predicted_position_norm,
                predicted_error_raw,
            ) = model.decode_horizons(
                scene_code,
                time_tensor,
            )

            predicted_position_px = (
                predicted_position_norm[0, 0]
                .cpu()
                .numpy()
                * COORD_SCALE
            )

            predicted_error_px = float(
                positive_error_magnitude(
                    predicted_error_raw
                )[0, 0].cpu()
                * ERR_MAG_PIXEL_SCALE_FOR_PLOTS
            )

            if not np.all(
                np.isfinite(predicted_position_px)
            ):
                raise RuntimeError(
                    "The RNN produced a non-finite position for "
                    f"{scene_dir.name} at horizon +{future_offset}."
                )

            if not np.isfinite(predicted_error_px):
                raise RuntimeError(
                    "The RNN produced a non-finite error for "
                    f"{scene_dir.name} at horizon +{future_offset}."
                )

            predicted_offsets.append(future_offset)
            predicted_positions.append(
                predicted_position_px.astype(
                    np.float64,
                    copy=False,
                )
            )
            predicted_errors.append(
                predicted_error_px
            )

            terminal_event = predicted_terminal_event(
                predicted_position_px,
                scene_geometry,
            )

            if terminal_event is not None:
                return {
                    "scene_dir": Path(scene_dir),
                    "true_df": true_df,
                    "input_last_frame": input_last_frame,
                    "true_final_frame": true_frame_count - 1,
                    "scene_geometry": scene_geometry,
                    "checkpoint_offsets": np.asarray(
                        predicted_offsets,
                        dtype=np.int64,
                    ),
                    "predicted_position_px": np.asarray(
                        predicted_positions,
                        dtype=np.float64,
                    ),
                    "predicted_error_px": np.asarray(
                        predicted_errors,
                        dtype=np.float64,
                    ),
                    "terminal_event": terminal_event,
                    "terminal_offset": future_offset,
                }

    # This point is not accepted as a valid endpoint.
    # The error prevents the notebook from silently calling the last
    # attempted horizon a final prediction.
    raise RuntimeError(
        f"No predicted goal or ground contact was found for "
        f"{scene_dir.name} through horizon "
        f"+{MAX_PREDICTION_HORIZON}.\n"
        "Increase MAX_PREDICTION_HORIZON if you intentionally want "
        "to extrapolate even farther."
    )


def select_overlay_indices(offsets):
    """
    Display every OVERLAY_STRIDE-th prediction and always display
    the terminal-contact prediction.
    """
    if len(offsets) == 0:
        return np.array([], dtype=int)

    selected = [
        index
        for index, offset in enumerate(offsets)
        if offset % OVERLAY_STRIDE == 0
    ]

    terminal_index = len(offsets) - 1
    if terminal_index not in selected:
        selected.append(terminal_index)

    return np.asarray(
        sorted(set(selected)),
        dtype=int,
    )


def load_background_image(
    scene_dir,
    input_last_frame,
):
    frame_path = (
        Path(scene_dir)
        / "frames"
        / f"frame_{input_last_frame:04d}.png"
    )

    if not frame_path.exists():
        frame_files = get_frame_files(scene_dir)
        if not frame_files:
            raise FileNotFoundError(
                f"No high-resolution PNG frames found for "
                f"{scene_dir}"
            )

        frame_path = frame_files[
            min(
                input_last_frame,
                len(frame_files) - 1,
            )
        ]

    return Image.open(frame_path).convert("RGB")


def draw_until_contact_overlay(result, output_path):
    scene_dir = result["scene_dir"]
    true_df = result["true_df"]
    input_last_frame = result["input_last_frame"]
    true_final_frame = result["true_final_frame"]
    terminal_event = result["terminal_event"]
    terminal_offset = result["terminal_offset"]

    offsets = result["checkpoint_offsets"]
    all_predicted_xy = result["predicted_position_px"]
    all_predicted_error = result["predicted_error_px"]

    shown_indices = select_overlay_indices(offsets)
    predicted_xy = all_predicted_xy[shown_indices]
    predicted_error = all_predicted_error[shown_indices]
    shown_offsets = offsets[shown_indices]

    image = load_background_image(
        scene_dir,
        input_last_frame,
    )

    current_true_xy = true_df.iloc[
        input_last_frame
    ][["ball_x", "ball_y"]].to_numpy(dtype=float)

    # Ground truth is shown only where recorded data exist.
    true_trajectory = true_df.iloc[
        input_last_frame : true_final_frame + 1
    ][["ball_x", "ball_y"]].to_numpy(dtype=float)

    # The model trajectory starts from the true current center and
    # ends only at the first predicted terminal contact.
    prediction_path = np.vstack(
        [current_true_xy, predicted_xy]
    )

    colors = plt.cm.viridis(
        np.linspace(
            0.05,
            0.95,
            len(predicted_xy),
        )
    )

    fig, axis = plt.subplots(figsize=(6, 7.5))
    axis.imshow(image, zorder=0)

    if len(true_trajectory) >= 2:
        axis.plot(
            true_trajectory[:, 0],
            true_trajectory[:, 1],
            linewidth=2.3,
            alpha=0.95,
            color="red",
            zorder=1,
            label="Recorded true trajectory",
        )

    if len(prediction_path) >= 2:
        path_segments = np.stack(
            [
                prediction_path[:-1],
                prediction_path[1:],
            ],
            axis=1,
        )

        line_collection = LineCollection(
            path_segments,
            colors=colors,
            linewidths=1.8,
            alpha=0.75,
            zorder=3,
        )
        axis.add_collection(line_collection)

    # Dashed predicted-error circles.
    for (
        (x_coord, y_coord),
        radius,
        color,
    ) in zip(
        predicted_xy,
        predicted_error,
        colors,
    ):
        if np.isfinite(radius) and radius > 0:
            circle = plt.Circle(
                (x_coord, y_coord),
                float(radius),
                fill=False,
                linestyle="--",
                linewidth=1.0,
                alpha=0.70,
                color=color,
                zorder=2,
            )
            axis.add_patch(circle)

    axis.scatter(
        predicted_xy[:, 0],
        predicted_xy[:, 1],
        s=34,
        c=colors,
        edgecolors="black",
        linewidths=0.4,
        zorder=4,
        label="RNN prediction",
    )

    # The final point is always a real predicted terminal contact.
    axis.scatter(
        predicted_xy[-1, 0],
        predicted_xy[-1, 1],
        marker="*",
        s=130,
        facecolors="none",
        edgecolors="black",
        linewidths=1.3,
        zorder=6,
        label=f"Predicted {terminal_event} contact",
    )

    axis.annotate(
        "+0",
        xy=current_true_xy,
        xytext=(5, -8),
        textcoords="offset points",
        fontsize=8,
        color="black",
        zorder=5,
    )

    for index, (
        (x_coord, y_coord),
        future_offset,
    ) in enumerate(
        zip(predicted_xy, shown_offsets)
    ):
        should_label = (
            index == 0
            or index == len(predicted_xy) - 1
            or future_offset % 50 == 0
        )

        if should_label:
            axis.annotate(
                f"+{int(future_offset)}",
                xy=(x_coord, y_coord),
                xytext=(5, 5),
                textcoords="offset points",
                fontsize=7,
                color="black",
                zorder=5,
            )

    error_handle = Line2D(
        [0],
        [0],
        linestyle="--",
        linewidth=1.0,
        color="gray",
        label="Predicted error radius",
    )

    handles, labels = (
        axis.get_legend_handles_labels()
    )
    handles.append(error_handle)
    labels.append("Predicted error radius")

    axis.legend(
        handles,
        labels,
        loc="lower right",
        framealpha=0.90,
        fontsize=8,
    )

    axis.set_xlim(0, ORIGINAL_FRAME_WIDTH)
    axis.set_ylim(ORIGINAL_FRAME_HEIGHT, 0)
    axis.set_title(
        f"{scene_dir.name}: pure till-the-end abstraction\n"
        f"stopped at predicted {terminal_event} contact "
        f"(+{terminal_offset} frames)",
        fontsize=10,
    )
    axis.axis("off")

    fig.tight_layout()
    fig.savefig(
        output_path,
        dpi=150,
        bbox_inches="tight",
    )
    plt.close(fig)

In [ ]:

# ============================================================
# 5. Run the trained RNN until predicted contact and save every
#    predicted point used by the analyses
# ============================================================

from IPython.display import display
import re


def scene_key_underscore(value):
    """Use scene_1_0 style consistently with the earlier labeling notebook."""
    value = str(value).replace("\\", "/").strip()
    value = Path(value).stem
    value = value.replace("__hybrid_predictions", "")

    if value.startswith("rnn_testing_data__"):
        value = value[len("rnn_testing_data__"):]

    value = value.replace("__", "/")
    value = Path(value).name

    match = re.match(r"^scene(\d+)\.(\d+)$", value)
    if match:
        return f"scene_{match.group(1)}_{match.group(2)}"

    match = re.match(r"^scene(\d+)_(\d+)$", value)
    if match:
        return f"scene_{match.group(1)}_{match.group(2)}"

    match = re.match(r"^scene_(\d+)\.(\d+)$", value)
    if match:
        return f"scene_{match.group(1)}_{match.group(2)}"

    return value

PREDICTION_CSV = ANALYSIS_ROOT / "rnn_predictions_until_contact.csv"
TERMINAL_SUMMARY_CSV = ANALYSIS_ROOT / "rnn_terminal_summary.csv"

# Set False to reuse the saved prediction CSV on a later analysis-only run.
RERUN_RNN_PREDICTIONS = True

if RERUN_RNN_PREDICTIONS or not PREDICTION_CSV.exists():
    prediction_rows = []
    terminal_rows = []

    for scene_index, scene_dir in enumerate(test_scene_dirs, start=1):
        result = predict_scene_until_contact(scene_dir)

        offsets = np.asarray(result["checkpoint_offsets"], dtype=int)
        positions = np.asarray(result["predicted_position_px"], dtype=float)
        errors = np.asarray(result["predicted_error_px"], dtype=float)

        if not (len(offsets) == len(positions) == len(errors)):
            raise RuntimeError(
                f"{scene_dir.name}: prediction arrays have different lengths."
            )

        expected_offsets = np.arange(1, int(result["terminal_offset"]) + 1)
        if not np.array_equal(offsets, expected_offsets):
            raise RuntimeError(
                f"{scene_dir.name}: offsets are not consecutive from +1 "
                "through terminal contact."
            )

        scene_name = scene_dir.name
        scene_key = scene_key_underscore(scene_name) if "scene_key_underscore" in globals() else scene_name
        experiment = scene_dir.parent.name
        input_last_frame = int(result["input_last_frame"])
        scene_total_frames = int(len(result["true_df"]))

        for point_index, (offset, xy, error_px) in enumerate(
            zip(offsets, positions, errors),
            start=1,
        ):
            prediction_rows.append({
                "scene_index": int(scene_index),
                "experiment": experiment,
                "scene": str(scene_dir),
                "scene_name": scene_name,
                "scene_key": scene_key,
                "point_id": int(point_index),
                "future_offset": int(offset),
                "checkpoint_offset": int(offset),
                "target_frame": int(input_last_frame + offset),
                "input_last_frame": input_last_frame,
                "scene_total_frames": scene_total_frames,
                "pred_x": float(xy[0]),
                "pred_y": float(xy[1]),
                "predicted_error_px": float(error_px),
                "terminal_event": str(result["terminal_event"]),
                "terminal_offset": int(result["terminal_offset"]),
                "is_terminal_prediction": int(offset == result["terminal_offset"]),
            })

        terminal_rows.append({
            "scene_index": int(scene_index),
            "experiment": experiment,
            "scene": str(scene_dir),
            "scene_name": scene_name,
            "scene_key": scene_key,
            "scene_total_frames": scene_total_frames,
            "terminal_event": str(result["terminal_event"]),
            "terminal_offset": int(result["terminal_offset"]),
            "n_predicted_points": int(len(offsets)),
        })

        print(
            f"[{scene_index:03d}/{len(test_scene_dirs):03d}] "
            f"{scene_name}: {result['terminal_event']} at "
            f"+{result['terminal_offset']}"
        )

    pred_df = pd.DataFrame(prediction_rows)
    terminal_summary_df = pd.DataFrame(terminal_rows)

    pred_df.to_csv(PREDICTION_CSV, index=False)
    terminal_summary_df.to_csv(TERMINAL_SUMMARY_CSV, index=False)
else:
    pred_df = pd.read_csv(PREDICTION_CSV)
    terminal_summary_df = pd.read_csv(TERMINAL_SUMMARY_CSV)

# Normalize the scene key after loading as well.
pred_df["scene_key"] = pred_df["scene_name"].astype(str).map(
    lambda value: scene_key_underscore(value)
)
terminal_summary_df["scene_key"] = terminal_summary_df["scene_name"].astype(str).map(
    lambda value: scene_key_underscore(value)
)

print("\nSaved prediction rows:", len(pred_df))
print("Scenes:", pred_df["scene_key"].nunique())
print("Prediction CSV:", PREDICTION_CSV)
print("Terminal summary:", TERMINAL_SUMMARY_CSV)
display(terminal_summary_df.head())


In [ ]:

# ============================================================
# 6. Apply the same collision-circle labeling logic as the
#    earlier container-circle notebook
#
# Container scenes:
#   base radius = 0.5 * sqrt(endpoint_length^2 + width^2)
#   active radius = 2 * base radius
#
# Line scenes:
#   the line segment itself is the circle diameter
# ============================================================

from pathlib import Path
import json
import math
import re
import numpy as np
import pandas as pd

LABEL_ROOT = ANALYSIS_ROOT / "collision_labels"
LABEL_ROOT.mkdir(parents=True, exist_ok=True)

CONTAINER_CSV_PATH = LABEL_ROOT / "container_circle_summary_endpoint_defined.csv"
LINE_CSV_PATH = LABEL_ROOT / "line_circle_summary_endpoint_defined.csv"
LABELED_CSV_PATH = LABEL_ROOT / "rnn_labeled_container_or_line_circle_phases.csv"

ORDER_COL_CANDIDATES = [
    "target_frame",
    "future_offset",
    "checkpoint_offset",
    "checkpoint_index",
    "input_last_frame",
]

def scene_key_underscore(value):
    """Use scene_1_0 style consistently."""
    s = str(value).replace("\\", "/").strip()
    s = Path(s).stem
    s = s.replace("__hybrid_predictions", "")
    if s.startswith("rnn_testing_data__"):
        s = s[len("rnn_testing_data__"):]
    s = s.replace("__", "/")
    s = Path(s).name

    import re

    m = re.match(r"^scene(\d+)\.(\d+)$", s)
    if m:
        return f"scene_{m.group(1)}_{m.group(2)}"

    m = re.match(r"^scene(\d+)_(\d+)$", s)
    if m:
        return f"scene_{m.group(1)}_{m.group(2)}"

    m = re.match(r"^scene_(\d+)\.(\d+)$", s)
    if m:
        return f"scene_{m.group(1)}_{m.group(2)}"

    return s

def as_xy(value):
    """Parse x,y from nested list/tuple/dict conventions used by the scene JSONs."""
    original = value

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if isinstance(value, dict):
        for key in ("position", "pos", "point", "location", "center"):
            if key in value:
                value = value[key]
                break

    while isinstance(value, list) and len(value) == 1:
        value = value[0]

    if isinstance(value, (list, tuple)) and len(value) >= 2:
        try:
            return float(value[0]), float(value[1])
        except Exception:
            pass

    raise ValueError(f"Could not parse x/y from: {original!r}")

def angle_to_radians(angle):
    try:
        a = float(angle)
    except Exception:
        return 0.0

    if abs(a) > 2 * math.pi:
        return a * math.pi / 180.0

    return a

def endpoints_from_center_length_angle(cx, cy, length, angle):
    theta = angle_to_radians(angle)
    half = float(length) / 2.0
    dx = half * math.cos(theta)
    dy = half * math.sin(theta)

    return (
        (float(cx) - dx, float(cy) - dy),
        (float(cx) + dx, float(cy) + dy),
    )

def finalize_container_record(scene_dir, idx, x1, y1, x2, y2, width=0.0, angle=0.0, source=""):
    """
    Create endpoint-defined container geometry.

    Important:
    - The base circle fully encircles the rectangular container:
        base_diameter = sqrt(endpoint_length^2 + width^2)
        base_radius   = base_diameter / 2
    - The ACTIVE circle used for both labeling and plotting has doubled radius:
        container_circle_radius = 2 * base_radius
    """
    x1, y1, x2, y2 = map(float, [x1, y1, x2, y2])

    try:
        width = float(width)
        if not np.isfinite(width):
            width = 0.0
    except Exception:
        width = 0.0

    try:
        angle = float(angle)
    except Exception:
        angle = 0.0

    endpoint_length = float(math.hypot(x2 - x1, y2 - y1))
    mid_x = (x1 + x2) / 2.0
    mid_y = (y1 + y2) / 2.0

    base_circle_diameter = float(math.hypot(endpoint_length, width))
    base_circle_radius = base_circle_diameter / 2.0

    # FIX: doubled radius for BOTH labeling and plotting.
    active_circle_radius = 2.0 * base_circle_radius
    active_circle_diameter = 2.0 * active_circle_radius

    return {
        "scene": str(scene_dir),
        "scene_name": Path(scene_dir).name,
        "scene_key": scene_key_underscore(Path(scene_dir).name),
        "container_id": int(idx),
        "container_x1": x1,
        "container_y1": y1,
        "container_x2": x2,
        "container_y2": y2,
        "container_mid_x": mid_x,
        "container_mid_y": mid_y,
        "container_width": float(width),
        "container_length_endpoint_distance": endpoint_length,
        "container_angle": float(angle),

        # Base encircling circle before doubling.
        "container_base_circle_diameter": base_circle_diameter,
        "container_base_circle_radius": base_circle_radius,

        # Active doubled circle used downstream.
        "container_circle_diameter": active_circle_diameter,
        "container_circle_radius": active_circle_radius,
        "container_radius_multiplier": 2.0,

        "container_endpoint_source": source,
    }

def parse_container_entry(c, idx, scene_dir, source="container_args"):
    """Return endpoint-defined container record from raw or already-parsed container formats."""
    while isinstance(c, list) and len(c) == 1:
        c = c[0]

    if isinstance(c, dict):
        if "raw" in c and c.get("raw") is not None:
            try:
                return parse_container_entry(
                    c.get("raw"),
                    c.get("id", idx),
                    scene_dir,
                    source=f"{source}.raw",
                )
            except Exception:
                pass

        endpoint_key_pairs = [
            ("point_a", "point_b"),
            ("a", "b"),
            ("p1", "p2"),
            ("start", "end"),
            ("endpoint_a", "endpoint_b"),
        ]

        for a_key, b_key in endpoint_key_pairs:
            if a_key in c and b_key in c:
                x1, y1 = as_xy(c[a_key])
                x2, y2 = as_xy(c[b_key])
                width = c.get("width", c.get("w", 0.0))
                angle = c.get("angle", 0.0)
                return finalize_container_record(
                    scene_dir,
                    idx,
                    x1,
                    y1,
                    x2,
                    y2,
                    width,
                    angle,
                    source=f"{source}:explicit_endpoints",
                )

        cx, cy = as_xy(c)
        width = float(c.get("width", c.get("w", 0.0)))
        length = float(c.get("length", c.get("l", 0.0)))
        angle = float(c.get("angle", 0.0))
        (x1, y1), (x2, y2) = endpoints_from_center_length_angle(cx, cy, length, angle)

        return finalize_container_record(
            scene_dir,
            c.get("id", idx),
            x1,
            y1,
            x2,
            y2,
            width,
            angle,
            source=f"{source}:center_length_angle",
        )

    if isinstance(c, (list, tuple)):
        if len(c) >= 2 and isinstance(c[0], (list, tuple, dict)) and isinstance(c[1], (list, tuple, dict)):
            try:
                x2_test, y2_test = as_xy(c[1])
                x1, y1 = as_xy(c[0])
                x2, y2 = x2_test, y2_test
                width = float(c[2]) if len(c) > 2 and isinstance(c[2], (int, float)) else 0.0
                angle = float(c[3]) if len(c) > 3 and isinstance(c[3], (int, float)) else 0.0

                return finalize_container_record(
                    scene_dir,
                    idx,
                    x1,
                    y1,
                    x2,
                    y2,
                    width,
                    angle,
                    source=f"{source}:list_explicit_endpoints",
                )
            except Exception:
                pass

        if len(c) >= 1:
            cx, cy = as_xy(c[0])
            width = float(c[1]) if len(c) > 1 else 0.0
            length = float(c[2]) if len(c) > 2 else 0.0
            angle = float(c[3]) if len(c) > 3 else 0.0
            (x1, y1), (x2, y2) = endpoints_from_center_length_angle(cx, cy, length, angle)

            return finalize_container_record(
                scene_dir,
                idx,
                x1,
                y1,
                x2,
                y2,
                width,
                angle,
                source=f"{source}:list_center_length_angle",
            )

    raise ValueError(f"Unknown container format: {c!r}")

def parse_container_columns(payload, scene_dir, source="static_columns"):
    """Parse columns like container_0_x, container_0_y, container_0_width, container_0_length, container_0_angle."""
    rows = []
    keys = list(payload.keys())

    import re

    ids = sorted({
        int(m.group(1))
        for k in keys
        for m in [re.match(r"container_(\d+)_(x|y|width|length|angle)$", str(k))]
        if m
    })

    for idx in ids:
        try:
            cx = float(payload.get(f"container_{idx}_x"))
            cy = float(payload.get(f"container_{idx}_y"))
            width = float(payload.get(f"container_{idx}_width", 0.0))
            length = float(payload.get(f"container_{idx}_length", 0.0))
            angle = float(payload.get(f"container_{idx}_angle", 0.0))
            (x1, y1), (x2, y2) = endpoints_from_center_length_angle(cx, cy, length, angle)

            rows.append(
                finalize_container_record(
                    scene_dir,
                    idx,
                    x1,
                    y1,
                    x2,
                    y2,
                    width,
                    angle,
                    source=source,
                )
            )
        except Exception as e:
            print(f"Warning: failed to parse container_{idx} static columns for {scene_dir}: {e}")

    return rows

def raw_scene_json_paths(scene_dir):
    excluded = {
        "static_scene_settings.json",
        "metadata.json",
        "batch_metadata.json",
        "frame_count_summary.json",
        "generation_uniqueness_summary.json",
    }

    paths = [p for p in Path(scene_dir).glob("*.json") if p.name not in excluded]
    preferred = [p for p in paths if p.stem == Path(scene_dir).name]

    return preferred + [p for p in paths if p not in preferred]

def load_scene_containers(scene_dir):
    """Load endpoint-defined containers for one rendered scene."""
    scene_dir = Path(scene_dir)
    rows = []

    static_path = scene_dir / "static_scene_settings.json"

    if static_path.exists():
        try:
            with open(static_path, "r") as f:
                static = json.load(f)

            if static.get("containers_json"):
                parsed = json.loads(static["containers_json"])
                for i, c in enumerate(parsed):
                    rows.append(
                        parse_container_entry(
                            c,
                            i,
                            scene_dir,
                            source="static_scene_settings.containers_json",
                        )
                    )

            if not rows:
                rows.extend(
                    parse_container_columns(
                        static,
                        scene_dir,
                        source="static_scene_settings.columns",
                    )
                )

            if not rows and static.get("full_scene_json"):
                raw = json.loads(static["full_scene_json"])
                for i, c in enumerate(raw.get("container_args", []) or []):
                    rows.append(
                        parse_container_entry(
                            c,
                            i,
                            scene_dir,
                            source="static_scene_settings.full_scene_json.container_args",
                        )
                    )
        except Exception as e:
            print(f"Warning: failed to parse {static_path}: {e}")

    if not rows:
        csv_path = scene_dir / "simulation_dataset.csv"

        if csv_path.exists():
            try:
                first = pd.read_csv(csv_path, nrows=1).iloc[0].to_dict()

                if first.get("containers_json"):
                    parsed = json.loads(first["containers_json"])
                    for i, c in enumerate(parsed):
                        rows.append(
                            parse_container_entry(
                                c,
                                i,
                                scene_dir,
                                source="simulation_dataset.containers_json",
                            )
                        )

                if not rows:
                    rows.extend(
                        parse_container_columns(
                            first,
                            scene_dir,
                            source="simulation_dataset.columns",
                        )
                    )

                if not rows and first.get("full_scene_json"):
                    raw = json.loads(first["full_scene_json"])
                    for i, c in enumerate(raw.get("container_args", []) or []):
                        rows.append(
                            parse_container_entry(
                                c,
                                i,
                                scene_dir,
                                source="simulation_dataset.full_scene_json.container_args",
                            )
                        )
            except Exception as e:
                print(f"Warning: failed to parse {csv_path}: {e}")

    if not rows:
        for jp in raw_scene_json_paths(scene_dir):
            try:
                with open(jp, "r") as f:
                    raw = json.load(f)

                for i, c in enumerate(raw.get("container_args", []) or []):
                    rows.append(
                        parse_container_entry(
                            c,
                            i,
                            scene_dir,
                            source=f"raw_scene_json:{jp.name}",
                        )
                    )

                if rows:
                    break
            except Exception as e:
                print(f"Warning: failed to parse raw scene json {jp}: {e}")

    if not rows:
        return pd.DataFrame(columns=[
            "scene",
            "scene_name",
            "scene_key",
            "container_id",
            "container_x1",
            "container_y1",
            "container_x2",
            "container_y2",
            "container_mid_x",
            "container_mid_y",
            "container_width",
            "container_length_endpoint_distance",
            "container_angle",
            "container_base_circle_diameter",
            "container_base_circle_radius",
            "container_circle_diameter",
            "container_circle_radius",
            "container_radius_multiplier",
            "container_endpoint_source",
        ])

    out = pd.DataFrame(rows)

    out = out[
        np.isfinite(out["container_circle_radius"]) &
        (out["container_circle_radius"] > 0)
    ].copy()

    out = out.drop_duplicates(subset=["scene", "container_id"], keep="first")

    return out

def present_columns(df, candidates):
    return [c for c in candidates if c in df.columns]

def label_one_scene(group, scene_key=None):
    group = group.copy()

    if scene_key is None:
        if "scene_key" in group.columns and len(group):
            scene_key = group["scene_key"].iloc[0]
        else:
            raise KeyError("scene_key is required to label one scene.")

    if "scene_key" not in group.columns:
        group["scene_key"] = scene_key

    sort_cols = present_columns(group, ORDER_COL_CANDIDATES)

    if sort_cols:
        group = group.sort_values(sort_cols).copy()
    else:
        group = group.sort_index().copy()

    circles = container_df[container_df["scene_key"] == scene_key].copy()

    if len(circles) == 0:
        group["inside_any_container_circle"] = False
        group["container_id_hit"] = np.nan
        group["container_distance_to_center"] = np.nan
        group["container_radius_hit"] = np.nan
        group["circle_episode"] = 1
        group["circle_phase"] = "between_or_before_circles"
        group["circle_label"] = "before_c1"
        group["transition_event"] = "no_container_circles_found"

        return group

    labels = []
    phases = []
    episodes = []
    transition_events = []
    hit_container_ids = []
    hit_distances = []
    hit_radii = []
    inside_flags = []

    current_episode = 0
    active_container_id = None
    was_inside = False

    circle_centers = circles[["container_mid_x", "container_mid_y"]].to_numpy(dtype=float)

    # This is now the doubled radius from finalize_container_record().
    circle_radii = circles["container_circle_radius"].to_numpy(dtype=float)

    circle_ids = circles["container_id"].to_numpy(dtype=int)

    for _, row in group.iterrows():
        px = float(row["pred_x"])
        py = float(row["pred_y"])

        d = np.sqrt(
            (px - circle_centers[:, 0]) ** 2 +
            (py - circle_centers[:, 1]) ** 2
        )

        inside_idx = np.where(np.isfinite(d) & (d <= circle_radii))[0]

        if len(inside_idx) == 0:
            label = f"before_c{current_episode + 1}"
            phase = "between_or_before_circles"
            episode = current_episode + 1
            event = "leave_circle" if was_inside else "none"
            hit_id = np.nan
            hit_dist = np.nan
            hit_rad = np.nan
            inside = False

            active_container_id = None
            was_inside = False

        else:
            best_i = inside_idx[np.argmin(d[inside_idx])]
            hit_id = int(circle_ids[best_i])
            hit_dist = float(d[best_i])
            hit_rad = float(circle_radii[best_i])
            inside = True

            if active_container_id is None or (not was_inside) or hit_id != active_container_id:
                current_episode += 1
                active_container_id = hit_id
                event = "enter_circle"
            else:
                event = "none"

            label = f"c{current_episode}"
            phase = "inside_circle"
            episode = current_episode
            was_inside = True

        labels.append(label)
        phases.append(phase)
        episodes.append(episode)
        transition_events.append(event)
        hit_container_ids.append(hit_id)
        hit_distances.append(hit_dist)
        hit_radii.append(hit_rad)
        inside_flags.append(inside)

    group["inside_any_container_circle"] = inside_flags
    group["container_id_hit"] = hit_container_ids
    group["container_distance_to_center"] = hit_distances
    group["container_radius_hit"] = hit_radii
    group["circle_episode"] = episodes
    group["circle_phase"] = phases
    group["circle_label"] = labels
    group["transition_event"] = transition_events

    return group

def get_line_args_from_json_any(data):
    """
    Match line-key conventions used by the render/generation scripts.
    """
    for key in [
        "line_args",
        "lines_args",
        "line_arg",
        "lines",
        "Line_args",
        "Line",
    ]:
        if key in data:
            val = data[key]
            if val is None:
                return [], key
            if isinstance(val, list):
                return val, key
            return [val], key

    return [], None

def parse_line_arg_any(line_arg, idx):
    """
    Parse one line into endpoint coordinates.

    Supported forms:
      dict with point_a/point_b, a/b, p1/p2, start/end, endpoint_a/endpoint_b
      list [[x1,y1], [x2,y2], angle?]
      list [x1, y1, x2, y2, angle?]
    """
    original = line_arg

    while isinstance(line_arg, list) and len(line_arg) == 1:
        line_arg = line_arg[0]

    if isinstance(line_arg, dict):
        endpoint_key_pairs = [
            ("point_a", "point_b"),
            ("a", "b"),
            ("p1", "p2"),
            ("start", "end"),
            ("endpoint_a", "endpoint_b"),
        ]

        for a_key, b_key in endpoint_key_pairs:
            if a_key in line_arg and b_key in line_arg:
                x1, y1 = as_xy(line_arg[a_key])
                x2, y2 = as_xy(line_arg[b_key])
                angle = float(line_arg.get("angle", 0.0))
                return {
                    "id": int(line_arg.get("id", idx)),
                    "point_a": (x1, y1),
                    "point_b": (x2, y2),
                    "angle": angle,
                    "raw": original,
                }

        raise ValueError(f"Line dict missing endpoint keys: {original!r}")

    if isinstance(line_arg, (list, tuple)):
        if (
            len(line_arg) >= 2
            and isinstance(line_arg[0], (list, tuple, dict))
            and isinstance(line_arg[1], (list, tuple, dict))
        ):
            x1, y1 = as_xy(line_arg[0])
            x2, y2 = as_xy(line_arg[1])
            angle = (
                float(line_arg[2])
                if len(line_arg) > 2 and isinstance(line_arg[2], (int, float))
                else 0.0
            )
            return {
                "id": int(idx),
                "point_a": (x1, y1),
                "point_b": (x2, y2),
                "angle": angle,
                "raw": original,
            }

        if len(line_arg) >= 4:
            x1 = float(line_arg[0])
            y1 = float(line_arg[1])
            x2 = float(line_arg[2])
            y2 = float(line_arg[3])
            angle = (
                float(line_arg[4])
                if len(line_arg) > 4 and isinstance(line_arg[4], (int, float))
                else 0.0
            )
            return {
                "id": int(idx),
                "point_a": (x1, y1),
                "point_b": (x2, y2),
                "angle": angle,
                "raw": original,
            }

    raise ValueError(f"Unknown line format: {original!r}")

def finalize_line_circle_record(scene_dir, idx, x1, y1, x2, y2, angle=0.0, source=""):
    """
    For line scenes:
      circle center = midpoint of line endpoints
      circle diameter = endpoint-to-endpoint distance
      circle radius = diameter / 2

    This intentionally does NOT double radius, because the requested rule is:
      draw a circle using this line as diameter.
    """
    x1, y1, x2, y2 = map(float, [x1, y1, x2, y2])

    try:
        angle = float(angle)
    except Exception:
        angle = 0.0

    line_length = float(math.hypot(x2 - x1, y2 - y1))
    mid_x = (x1 + x2) / 2.0
    mid_y = (y1 + y2) / 2.0

    return {
        "scene": str(scene_dir),
        "scene_name": Path(scene_dir).name,
        "scene_key": scene_key_underscore(Path(scene_dir).name),
        "line_id": int(idx),
        "line_x1": x1,
        "line_y1": y1,
        "line_x2": x2,
        "line_y2": y2,
        "line_mid_x": mid_x,
        "line_mid_y": mid_y,
        "line_length_endpoint_distance": line_length,
        "line_angle": angle,
        "line_circle_diameter": line_length,
        "line_circle_radius": line_length / 2.0,
        "line_endpoint_source": source,
    }

def parse_lines_json_string(x, scene_dir, source):
    rows = []

    if x is None:
        return rows

    if isinstance(x, float) and pd.isna(x):
        return rows

    if isinstance(x, str):
        if x.strip() == "" or x.strip() == "[]":
            return rows
        parsed = json.loads(x)
    else:
        parsed = x

    if not isinstance(parsed, list):
        parsed = [parsed]

    for i, line_item in enumerate(parsed):
        pl = parse_line_arg_any(line_item, i)
        x1, y1 = pl["point_a"]
        x2, y2 = pl["point_b"]
        rows.append(
            finalize_line_circle_record(
                scene_dir=scene_dir,
                idx=pl["id"],
                x1=x1,
                y1=y1,
                x2=x2,
                y2=y2,
                angle=pl.get("angle", 0.0),
                source=source,
            )
        )

    return rows

def parse_line_columns(payload, scene_dir, source):
    """
    Parse static columns:
      line_0_point_a_x, line_0_point_a_y, line_0_point_b_x, line_0_point_b_y, line_0_angle
    """
    import re

    keys = list(payload.keys())

    ids = sorted({
        int(m.group(1))
        for k in keys
        for m in [re.match(r"line_(\d+)_point_(a|b)_(x|y)$", str(k))]
        if m
    })

    rows = []

    for idx in ids:
        try:
            x1 = float(payload[f"line_{idx}_point_a_x"])
            y1 = float(payload[f"line_{idx}_point_a_y"])
            x2 = float(payload[f"line_{idx}_point_b_x"])
            y2 = float(payload[f"line_{idx}_point_b_y"])
            angle = float(payload.get(f"line_{idx}_angle", 0.0))

            rows.append(
                finalize_line_circle_record(
                    scene_dir=scene_dir,
                    idx=idx,
                    x1=x1,
                    y1=y1,
                    x2=x2,
                    y2=y2,
                    angle=angle,
                    source=source,
                )
            )
        except Exception as e:
            print(f"Warning: failed to parse line_{idx} static columns for {scene_dir}: {e}")

    return rows

def raw_scene_json_paths_for_lines(scene_dir):
    excluded = {
        "static_scene_settings.json",
        "metadata.json",
        "batch_metadata.json",
        "frame_count_summary.json",
        "generation_uniqueness_summary.json",
    }

    scene_dir = Path(scene_dir)

    paths = [p for p in scene_dir.glob("*.json") if p.name not in excluded]
    preferred = [p for p in paths if p.stem == scene_dir.name]

    return preferred + [p for p in paths if p not in preferred]

def load_scene_lines(scene_dir):
    """
    Load endpoint-defined line circles for one rendered scene.

    Priority:
      1. static_scene_settings.json
      2. simulation_dataset.csv first row
      3. raw scene JSON
    """
    scene_dir = Path(scene_dir)
    rows = []

    static_path = scene_dir / "static_scene_settings.json"

    if static_path.exists():
        try:
            with open(static_path, "r") as f:
                static = json.load(f)

            if static.get("lines_json"):
                rows.extend(
                    parse_lines_json_string(
                        static["lines_json"],
                        scene_dir,
                        source="static_scene_settings.lines_json",
                    )
                )

            if not rows:
                rows.extend(
                    parse_line_columns(
                        static,
                        scene_dir,
                        source="static_scene_settings.columns",
                    )
                )

            if not rows and static.get("full_scene_json"):
                raw = json.loads(static["full_scene_json"])
                line_args, line_key = get_line_args_from_json_any(raw)

                for i, line_arg in enumerate(line_args):
                    pl = parse_line_arg_any(line_arg, i)
                    x1, y1 = pl["point_a"]
                    x2, y2 = pl["point_b"]
                    rows.append(
                        finalize_line_circle_record(
                            scene_dir=scene_dir,
                            idx=pl["id"],
                            x1=x1,
                            y1=y1,
                            x2=x2,
                            y2=y2,
                            angle=pl.get("angle", 0.0),
                            source=f"static_scene_settings.full_scene_json.{line_key}",
                        )
                    )

        except Exception as e:
            print(f"Warning: failed to parse line metadata from {static_path}: {e}")

    if not rows:
        csv_path = scene_dir / "simulation_dataset.csv"

        if csv_path.exists():
            try:
                first = pd.read_csv(csv_path, nrows=1).iloc[0].to_dict()

                if first.get("lines_json"):
                    rows.extend(
                        parse_lines_json_string(
                            first["lines_json"],
                            scene_dir,
                            source="simulation_dataset.lines_json",
                        )
                    )

                if not rows:
                    rows.extend(
                        parse_line_columns(
                            first,
                            scene_dir,
                            source="simulation_dataset.columns",
                        )
                    )

                if not rows and first.get("full_scene_json"):
                    raw = json.loads(first["full_scene_json"])
                    line_args, line_key = get_line_args_from_json_any(raw)

                    for i, line_arg in enumerate(line_args):
                        pl = parse_line_arg_any(line_arg, i)
                        x1, y1 = pl["point_a"]
                        x2, y2 = pl["point_b"]
                        rows.append(
                            finalize_line_circle_record(
                                scene_dir=scene_dir,
                                idx=pl["id"],
                                x1=x1,
                                y1=y1,
                                x2=x2,
                                y2=y2,
                                angle=pl.get("angle", 0.0),
                                source=f"simulation_dataset.full_scene_json.{line_key}",
                            )
                        )

            except Exception as e:
                print(f"Warning: failed to parse line metadata from {csv_path}: {e}")

    if not rows:
        for jp in raw_scene_json_paths_for_lines(scene_dir):
            try:
                with open(jp, "r") as f:
                    raw = json.load(f)

                line_args, line_key = get_line_args_from_json_any(raw)

                for i, line_arg in enumerate(line_args):
                    pl = parse_line_arg_any(line_arg, i)
                    x1, y1 = pl["point_a"]
                    x2, y2 = pl["point_b"]
                    rows.append(
                        finalize_line_circle_record(
                            scene_dir=scene_dir,
                            idx=pl["id"],
                            x1=x1,
                            y1=y1,
                            x2=x2,
                            y2=y2,
                            angle=pl.get("angle", 0.0),
                            source=f"raw_scene_json:{jp.name}:{line_key}",
                        )
                    )

                if rows:
                    break

            except Exception as e:
                print(f"Warning: failed to parse raw line JSON {jp}: {e}")

    if not rows:
        return pd.DataFrame(columns=[
            "scene",
            "scene_name",
            "scene_key",
            "line_id",
            "line_x1",
            "line_y1",
            "line_x2",
            "line_y2",
            "line_mid_x",
            "line_mid_y",
            "line_length_endpoint_distance",
            "line_angle",
            "line_circle_diameter",
            "line_circle_radius",
            "line_endpoint_source",
        ])

    out = pd.DataFrame(rows)

    out = out[
        np.isfinite(out["line_circle_radius"]) &
        (out["line_circle_radius"] > 0)
    ].copy()

    out = out.drop_duplicates(subset=["scene", "line_id"], keep="first")

    return out

def label_one_line_scene(group, scene_key=None):
    group = group.copy()

    if scene_key is None:
        if "scene_key" in group.columns and len(group):
            scene_key = group["scene_key"].iloc[0]
        else:
            raise KeyError("scene_key is required to label one line scene.")

    if "scene_key" not in group.columns:
        group["scene_key"] = scene_key

    sort_cols = present_columns(group, ORDER_COL_CANDIDATES)

    if sort_cols:
        group = group.sort_values(sort_cols).copy()
    else:
        group = group.sort_index().copy()

    circles = line_df[line_df["scene_key"] == scene_key].copy()

    if len(circles) == 0:
        group["inside_any_container_circle"] = False
        group["container_id_hit"] = np.nan
        group["container_distance_to_center"] = np.nan
        group["container_radius_hit"] = np.nan
        group["circle_episode"] = 1
        group["circle_phase"] = "between_or_before_circles"
        group["circle_label"] = "before_c1"
        group["transition_event"] = "no_line_circles_found"
        group["circle_source_type"] = "line"
        return group

    labels = []
    phases = []
    episodes = []
    transition_events = []
    hit_line_ids = []
    hit_distances = []
    hit_radii = []
    inside_flags = []

    current_episode = 0
    active_line_id = None
    was_inside = False

    circle_centers = circles[["line_mid_x", "line_mid_y"]].to_numpy(dtype=float)
    circle_radii = circles["line_circle_radius"].to_numpy(dtype=float)
    circle_ids = circles["line_id"].to_numpy(dtype=int)

    for _, row in group.iterrows():
        px = float(row["pred_x"])
        py = float(row["pred_y"])

        d = np.sqrt(
            (px - circle_centers[:, 0]) ** 2 +
            (py - circle_centers[:, 1]) ** 2
        )

        inside_idx = np.where(np.isfinite(d) & (d <= circle_radii))[0]

        if len(inside_idx) == 0:
            label = f"before_c{current_episode + 1}"
            phase = "between_or_before_circles"
            episode = current_episode + 1
            event = "leave_circle" if was_inside else "none"
            hit_id = np.nan
            hit_dist = np.nan
            hit_rad = np.nan
            inside = False

            active_line_id = None
            was_inside = False

        else:
            best_i = inside_idx[np.argmin(d[inside_idx])]
            hit_id = int(circle_ids[best_i])
            hit_dist = float(d[best_i])
            hit_rad = float(circle_radii[best_i])
            inside = True

            if active_line_id is None or (not was_inside) or hit_id != active_line_id:
                current_episode += 1
                active_line_id = hit_id
                event = "enter_circle"
            else:
                event = "none"

            label = f"c{current_episode}"
            phase = "inside_circle"
            episode = current_episode
            was_inside = True

        labels.append(label)
        phases.append(phase)
        episodes.append(episode)
        transition_events.append(event)
        hit_line_ids.append(hit_id)
        hit_distances.append(hit_dist)
        hit_radii.append(hit_rad)
        inside_flags.append(inside)

    # Reuse the same column names as container-labeled scenes where possible,
    # so these can append cleanly to the existing labeled CSV.
    group["inside_any_container_circle"] = inside_flags
    group["container_id_hit"] = hit_line_ids
    group["container_distance_to_center"] = hit_distances
    group["container_radius_hit"] = hit_radii
    group["circle_episode"] = episodes
    group["circle_phase"] = phases
    group["circle_label"] = labels
    group["transition_event"] = transition_events

    # Extra explicit line metadata.
    group["circle_source_type"] = "line"
    group["line_id_hit"] = hit_line_ids
    group["line_distance_to_center"] = hit_distances
    group["line_radius_hit"] = hit_radii

    return group

# Build endpoint-defined geometry for every prediction scene.
container_frames = []
line_frames = []

scene_path_lookup = (
    pred_df[["scene_key", "scene"]]
    .drop_duplicates("scene_key")
    .copy()
)

for row in scene_path_lookup.itertuples(index=False):
    scene_path = Path(row.scene)

    containers = load_scene_containers(scene_path)
    if len(containers):
        container_frames.append(containers)
        continue

    # Match the earlier workflow: use line circles only for scenes
    # without endpoint-defined containers.
    lines = load_scene_lines(scene_path)
    if len(lines):
        line_frames.append(lines)

container_df = (
    pd.concat(container_frames, ignore_index=True)
    if container_frames
    else pd.DataFrame(columns=["scene_key", "container_id"])
)

line_df = (
    pd.concat(line_frames, ignore_index=True)
    if line_frames
    else pd.DataFrame(columns=["scene_key", "line_id"])
)

container_df.to_csv(CONTAINER_CSV_PATH, index=False)
line_df.to_csv(LINE_CSV_PATH, index=False)

container_scene_keys = set(container_df.get("scene_key", pd.Series(dtype=str)).astype(str))
line_scene_keys = set(line_df.get("scene_key", pd.Series(dtype=str)).astype(str))

labeled_parts = []

for scene_key, group in pred_df.groupby("scene_key", sort=True):
    scene_key = str(scene_key)

    if scene_key in container_scene_keys:
        labeled = label_one_scene(group, scene_key=scene_key)
        labeled["circle_source_type"] = "container"

    elif scene_key in line_scene_keys:
        labeled = label_one_line_scene(group, scene_key=scene_key)

    else:
        sort_columns = present_columns(group, ORDER_COL_CANDIDATES)
        labeled = group.sort_values(sort_columns).copy() if sort_columns else group.copy()
        labeled["inside_any_container_circle"] = False
        labeled["container_id_hit"] = np.nan
        labeled["container_distance_to_center"] = np.nan
        labeled["container_radius_hit"] = np.nan
        labeled["circle_episode"] = 1
        labeled["circle_phase"] = "between_or_before_circles"
        labeled["circle_label"] = "before_c1"
        labeled["transition_event"] = "no_container_or_line_circles_found"
        labeled["circle_source_type"] = "none"

    labeled_parts.append(labeled)

labeled_df = pd.concat(labeled_parts, ignore_index=True)
labeled_df = labeled_df.sort_values(["scene_key", "future_offset"]).reset_index(drop=True)
labeled_df.to_csv(LABELED_CSV_PATH, index=False)

label_diagnostics = (
    labeled_df.groupby(["circle_source_type", "circle_label"])
    .size()
    .rename("n_rows")
    .reset_index()
)

print("Prediction scenes:", pred_df["scene_key"].nunique())
print("Container scenes:", len(container_scene_keys))
print("Line scenes:", len(line_scene_keys))
print(
    "Scenes without parsed container/line geometry:",
    labeled_df.loc[
        labeled_df["circle_source_type"] == "none",
        "scene_key",
    ].nunique(),
)
print("Labeled CSV:", LABELED_CSV_PATH)
display(label_diagnostics)


In [ ]:

# ============================================================
# 7. Final planned-contrast mixed model
#
# log1p_rnn_predicted_error
# ~ future_distance_z
# + c1_vs_before_c1
# + after_c1_vs_c1
# + scene_total_frames_z
# + scene_n_objects_z
# + (1 + future_distance_z | scene_key)
# ============================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display

MODEL_OUTDIR = ANALYSIS_ROOT / "mixed_model"
MODEL_OUTDIR.mkdir(parents=True, exist_ok=True)

MODEL_DATA_CSV = MODEL_OUTDIR / "mixed_model_row_level_data.csv"
MODEL_SUMMARY_TXT = MODEL_OUTDIR / "mixed_model_summary.txt"
SIGNIFICANCE_CSV = MODEL_OUTDIR / "mixed_model_significance_summary.csv"
R2_CSV = MODEL_OUTDIR / "mixed_model_marginal_conditional_r2.csv"
SCENE_PREDICTORS_CSV = MODEL_OUTDIR / "scene_level_predictors.csv"

model_df = labeled_df.copy()

model_df["rnn_predicted_error"] = pd.to_numeric(
    model_df["predicted_error_px"],
    errors="coerce",
)
model_df = model_df[
    np.isfinite(model_df["rnn_predicted_error"])
    & (model_df["rnn_predicted_error"] >= 0)
].copy()
model_df["log1p_rnn_predicted_error"] = np.log1p(
    model_df["rnn_predicted_error"]
)

# Future distance is the queried horizon from the last observed input frame.
model_df["future_distance"] = pd.to_numeric(
    model_df["future_offset"],
    errors="coerce",
)
future_mean = model_df["future_distance"].mean()
future_sd = model_df["future_distance"].std(ddof=0)

if not np.isfinite(future_sd) or future_sd == 0:
    raise ValueError("future_distance has invalid or zero population SD.")

model_df["future_distance_z"] = (
    model_df["future_distance"] - future_mean
) / future_sd

# Preserve the earlier three-stage collapsing and planned contrasts exactly.
model_df["circle_label_raw"] = model_df["circle_label"].astype(str)
model_df["circle_stage3"] = model_df["circle_label_raw"]

model_df.loc[
    model_df["circle_label_raw"].isin(
        ["c2", "before_c3", "c3", "before_c4"]
    ),
    "circle_stage3",
] = "after_c1"

model_df.loc[
    model_df["circle_label_raw"].str.match(
        r"^(c[2-9]\d*|before_c[3-9]\d*)$",
        na=False,
    ),
    "circle_stage3",
] = "after_c1"

model_df = model_df[
    model_df["circle_stage3"].isin(
        ["before_c1", "c1", "after_c1"]
    )
].copy()

model_df["c1_vs_before_c1"] = model_df["circle_stage3"].map({
    "before_c1": -0.5,
    "c1": 0.5,
    "after_c1": 0.0,
}).astype(float)

model_df["after_c1_vs_c1"] = model_df["circle_stage3"].map({
    "before_c1": 0.0,
    "c1": -0.5,
    "after_c1": 0.5,
}).astype(float)


def infer_scene_total_frames(scene_path, fallback_rows=None):
    """Use the same rendered-frame-first logic as the earlier notebook."""
    scene_path = Path(str(scene_path))
    frames_dir = scene_path / "frames"

    if frames_dir.exists():
        frame_files = sorted(frames_dir.glob("frame_*.png"))
        if frame_files:
            return int(len(frame_files))

    sim_csv = scene_path / "simulation_dataset.csv"
    if sim_csv.exists():
        try:
            sim_df = pd.read_csv(sim_csv)
            for column in ["frame", "frame_index", "timestep"]:
                if column in sim_df.columns:
                    return int(sim_df[column].nunique())
            return int(len(sim_df))
        except Exception:
            pass

    if fallback_rows is not None and len(fallback_rows):
        for column in [
            "scene_total_frames",
            "true_final_frame",
            "target_frame",
            "input_last_frame",
        ]:
            if column in fallback_rows.columns:
                values = pd.to_numeric(
                    fallback_rows[column],
                    errors="coerce",
                )
                values = values[np.isfinite(values)]
                if len(values):
                    if column == "scene_total_frames":
                        return int(values.max())
                    return int(values.max()) + 1

    return np.nan


scene_frame_rows = []
for scene_key, group in labeled_df.groupby("scene_key", sort=True):
    scene_path = group["scene"].dropna().astype(str).iloc[0]
    scene_frame_rows.append({
        "scene_key": str(scene_key),
        "scene_total_frames": infer_scene_total_frames(
            scene_path,
            fallback_rows=group,
        ),
    })
scene_frame_df = pd.DataFrame(scene_frame_rows)

if len(container_df):
    container_count_df = (
        container_df.assign(
            scene_key=container_df["scene_key"].astype(str)
        )
        .groupby("scene_key", as_index=False)
        .agg(scene_n_containers=("container_id", "nunique"))
    )
else:
    container_count_df = pd.DataFrame(
        columns=["scene_key", "scene_n_containers"]
    )

if len(line_df):
    line_count_df = (
        line_df.assign(scene_key=line_df["scene_key"].astype(str))
        .groupby("scene_key", as_index=False)
        .agg(scene_n_lines=("line_id", "nunique"))
    )
else:
    line_count_df = pd.DataFrame(
        columns=["scene_key", "scene_n_lines"]
    )

scene_object_df = (
    pd.DataFrame({
        "scene_key": sorted(
            labeled_df["scene_key"].dropna().astype(str).unique()
        )
    })
    .merge(container_count_df, on="scene_key", how="left")
    .merge(line_count_df, on="scene_key", how="left")
)

for column in ["scene_n_containers", "scene_n_lines"]:
    scene_object_df[column] = (
        pd.to_numeric(scene_object_df[column], errors="coerce")
        .fillna(0)
        .astype(int)
    )

scene_object_df["scene_n_objects"] = (
    scene_object_df["scene_n_containers"]
    + scene_object_df["scene_n_lines"]
)

scene_level_df = scene_frame_df.merge(
    scene_object_df,
    on="scene_key",
    how="outer",
)

scene_level_df["scene_total_frames"] = pd.to_numeric(
    scene_level_df["scene_total_frames"],
    errors="coerce",
)
scene_level_df["scene_total_frames"] = (
    scene_level_df["scene_total_frames"]
    .fillna(scene_level_df["scene_total_frames"].median())
)


def add_population_zscore(data, column):
    values = pd.to_numeric(data[column], errors="coerce")
    mean_value = values.mean()
    sd_value = values.std(ddof=0)

    if not np.isfinite(sd_value) or sd_value == 0:
        raise ValueError(f"{column} has invalid or zero population SD.")

    data[column + "_z"] = (values - mean_value) / sd_value
    return data


scene_level_df = add_population_zscore(
    scene_level_df,
    "scene_total_frames",
)
scene_level_df = add_population_zscore(
    scene_level_df,
    "scene_n_objects",
)
scene_level_df.to_csv(SCENE_PREDICTORS_CSV, index=False)

model_df = model_df.merge(
    scene_level_df[
        [
            "scene_key",
            "scene_total_frames",
            "scene_total_frames_z",
            "scene_n_containers",
            "scene_n_lines",
            "scene_n_objects",
            "scene_n_objects_z",
        ]
    ],
    on="scene_key",
    how="left",
    suffixes=("", "_scene"),
)

required_columns = [
    "scene_key",
    "log1p_rnn_predicted_error",
    "future_distance_z",
    "c1_vs_before_c1",
    "after_c1_vs_c1",
    "scene_total_frames_z",
    "scene_n_objects_z",
]

model_df = model_df.dropna(subset=required_columns).copy()
for column in required_columns:
    if column != "scene_key":
        model_df = model_df[np.isfinite(model_df[column])].copy()

model_df.to_csv(MODEL_DATA_CSV, index=False)

formula = (
    "log1p_rnn_predicted_error ~ "
    "future_distance_z "
    "+ c1_vs_before_c1 "
    "+ after_c1_vs_c1 "
    "+ scene_total_frames_z "
    "+ scene_n_objects_z"
)

mixed_model = smf.mixedlm(
    formula=formula,
    data=model_df,
    groups=model_df["scene_key"],
    re_formula="1 + future_distance_z",
)

# Try a small set of optimizers and retain the first converged fit.
fit_attempts = []
mixed_result = None

for method in ["lbfgs", "powell", "cg"]:
    try:
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter("always")
            candidate = mixed_model.fit(
                method=method,
                reml=False,
                maxiter=2000,
                disp=False,
            )

        fit_attempts.append({
            "method": method,
            "converged": bool(candidate.converged),
            "log_likelihood": float(candidate.llf),
            "warnings": " | ".join(str(w.message) for w in caught),
        })

        if candidate.converged and np.isfinite(candidate.llf):
            mixed_result = candidate
            break
    except Exception as exc:
        fit_attempts.append({
            "method": method,
            "converged": False,
            "log_likelihood": np.nan,
            "warnings": f"{type(exc).__name__}: {exc}",
        })

if mixed_result is None:
    raise RuntimeError(
        "No MixedLM optimizer produced a converged finite-likelihood fit.\n"
        + pd.DataFrame(fit_attempts).to_string(index=False)
    )

with open(MODEL_SUMMARY_TXT, "w") as file:
    file.write(str(mixed_result.summary()))

# Fixed-effect significance summary only.
fixed_terms = list(mixed_result.fe_params.index)
confidence_intervals = mixed_result.conf_int().loc[fixed_terms]

significance_df = pd.DataFrame({
    "term": fixed_terms,
    "estimate": mixed_result.fe_params.loc[fixed_terms].to_numpy(),
    "std_error": mixed_result.bse.loc[fixed_terms].to_numpy(),
    "z_value": mixed_result.tvalues.loc[fixed_terms].to_numpy(),
    "p_value": mixed_result.pvalues.loc[fixed_terms].to_numpy(),
    "ci_95_lower": confidence_intervals.loc[fixed_terms, 0].to_numpy(),
    "ci_95_upper": confidence_intervals.loc[fixed_terms, 1].to_numpy(),
})

def significance_code(p_value):
    if not np.isfinite(p_value):
        return ""
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"

significance_df["significance"] = significance_df["p_value"].map(
    significance_code
)
significance_df["exp_estimate"] = np.exp(significance_df["estimate"])
significance_df["approx_percent_change"] = (
    significance_df["exp_estimate"] - 1.0
) * 100.0
significance_df.to_csv(SIGNIFICANCE_CSV, index=False)

# Nakagawa-style marginal and conditional R² with observation-specific
# random-slope variance: mean(diag(Z G Z')).
fixed_linear_predictor = (
    np.asarray(mixed_result.model.exog, dtype=float)
    @ np.asarray(mixed_result.fe_params, dtype=float)
)
var_fixed = float(np.var(fixed_linear_predictor, ddof=0))

random_design = np.asarray(
    mixed_result.model.exog_re,
    dtype=float,
)
random_covariance = np.asarray(
    mixed_result.cov_re,
    dtype=float,
)
random_variance_by_row = np.einsum(
    "ij,jk,ik->i",
    random_design,
    random_covariance,
    random_design,
)
var_random = float(np.mean(random_variance_by_row))
var_residual = float(mixed_result.scale)

variance_total = var_fixed + var_random + var_residual
marginal_r2 = var_fixed / variance_total
conditional_r2 = (var_fixed + var_random) / variance_total

r2_df = pd.DataFrame([{
    "model": formula + " + (1 + future_distance_z | scene_key)",
    "n_rows": int(mixed_result.nobs),
    "n_scenes": int(model_df["scene_key"].nunique()),
    "var_fixed_effects": var_fixed,
    "var_random_effects_mean_zgz": var_random,
    "var_residual": var_residual,
    "marginal_r2": marginal_r2,
    "conditional_r2": conditional_r2,
    "converged": bool(mixed_result.converged),
}])
r2_df.to_csv(R2_CSV, index=False)

print("Model rows:", len(model_df))
print("Scenes:", model_df["scene_key"].nunique())
print("Optimizer attempts:")
display(pd.DataFrame(fit_attempts))
print("\nMixed-model summary:")
print(mixed_result.summary())

print("\nFixed-effect significance summary:")
display(significance_df)

print("\nMarginal and conditional R²:")
display(r2_df)

print("Saved model data:", MODEL_DATA_CSV)
print("Saved summary:", MODEL_SUMMARY_TXT)
print("Saved significance table:", SIGNIFICANCE_CSV)
print("Saved R² table:", R2_CSV)


# Part I — RNN analysis figures

In [ ]:
set_figure_prefix("01_rnn")

# Clean four-output RNN analysis notebook

This notebook consolidates the requested analyses from the supplied notebooks and produces only four top-level results:

1. Non-straight-path RNN vs. true-simulation path-efficiency raincloud with paired Wilcoxon signed-rank result.
2. Mixed-effects model predicting RNN-predicted error: raw model summary plus the literal contrast-calculated parameter table.
3. Scene-average RNN-predicted error vs. pupil size with straight, non-straight, and pooled fits, plus Pearson-\(r\) and OLS slope tests for each line.
4. `high_yescol_nosp_3` RNN prediction illustration paired with the prediction-error curve.

No figures or tables are saved by this notebook.

In [ ]:
# ============================================================
# Shared imports, paths, colors, sizes, and plotting style
# ============================================================
from pathlib import Path
import re
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from PIL import Image
from scipy.stats import gaussian_kde, wilcoxon, pearsonr, norm
from IPython.display import display
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"

SCENE_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "scene_summary.csv"
CLASSIFIED_TRIAL_PATH = BASE_DIR / "data" / "eye_gaze" / "classified_trial_data.csv"
RNN_PREDICTION_CSV = (
    BASE_DIR
    / "rnn_analysis"
    / "rnn_predictions_until_contact.csv"
)
LMER_MODEL_DATA_CSV = (
    BASE_DIR
    / "rnn_analysis"
    / "mixed_model"
    / "mixed_model_row_level_data.csv"
)
COLLISION_LABEL_CSV = (
    BASE_DIR
    / "rnn_analysis"
    / "collision_labels"
    / "rnn_labeled_container_or_line_circle_phases.csv"
)
TEST_DATA_ROOT = BASE_DIR / "rnn_testing_data"

TARGET_SCENE = "high_yescol_nosp_3"
N_HISTORY = 15
ORIGINAL_FRAME_WIDTH = 800.0
ORIGINAL_FRAME_HEIGHT = 1024.0
EXCLUDED_SCENES = {"low_yescol_yessp_4"}

# Standard project colors.
NON_STRAIGHT_COLOR = "#1C77C3"
STRAIGHT_COLOR = "#F39237"
POOLED_COLOR = "#4D4D4D"
TRUE_COLOR = "red"
BEFORE_COLOR = "#5B5F97"
DURING_COLOR = "#F39237"
AFTER_COLOR = "#1C77C3"
SCENE_BG_COLOR = "#E6E6E6"

PATH_COLORS = {
    0: NON_STRAIGHT_COLOR,
    1: STRAIGHT_COLOR,
}
PATH_LABELS = {
    0: "Non-straight path",
    1: "Straight path",
}

# Standard figure sizes; the scene/error illustration is a two-panel variant.
STANDARD_FIGSIZE = (8.5, 6.5)
WIDE_FIGSIZE = (13.5, 6.3)
CI_ALPHA = 0.16

# Shared text standard for all non-scene figures.
# The two single-scene visual illustrations keep their explicit original sizes.
TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_SIZE = 14
BASE_FONT_SIZE = 14

plt.rcParams.update({
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.titlepad": 12,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "legend.title_fontsize": LEGEND_SIZE,
    "axes.linewidth": 1.0,
    "figure.dpi": 120,
})


def require_columns(df, columns, label):
    missing = [c for c in columns if c not in df.columns]
    if missing:
        raise KeyError(f"{label} is missing: {missing}")


def clean_numeric(df, columns):
    out = df.copy()
    for column in columns:
        out[column] = pd.to_numeric(out[column], errors="coerce")
    return out


def style_numeric_axis(ax, grid_axis="both", hide_left=False):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    if hide_left:
        ax.spines["left"].set_visible(False)
    ax.grid(axis=grid_axis, alpha=0.18, zorder=0)


def format_p(p):
    if p < 0.001:
        return "< .001"
    return f"= {p:.3f}"

In [ ]:
# ============================================================
# Shared helpers used by the requested figures/statistics
# ============================================================
def scaled_density(values, grid, maximum_height=0.38):
    values = np.asarray(values, dtype=float)
    if (
        len(values) < 2
        or len(np.unique(values)) < 2
        or np.isclose(np.std(values, ddof=1), 0)
    ):
        return np.zeros_like(grid)
    density = gaussian_kde(values)(grid)
    return density / density.max() * maximum_height


def fit_simple_ols_with_ci(dataframe, x_column, y_column, grid_points=300):
    fit_data = (
        dataframe[[x_column, y_column]]
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .copy()
    )
    if (
        len(fit_data) < 3
        or fit_data[x_column].nunique() < 2
        or fit_data[y_column].nunique() < 2
    ):
        raise RuntimeError("Not enough variation to fit the requested OLS line.")

    x = fit_data[x_column].to_numpy(dtype=float)
    y = fit_data[y_column].to_numpy(dtype=float)

    X = sm.add_constant(x, has_constant="add")
    result = sm.OLS(y, X).fit()

    x_grid = np.linspace(x.min(), x.max(), grid_points)
    X_grid = sm.add_constant(x_grid, has_constant="add")
    prediction = result.get_prediction(X_grid).summary_frame(alpha=0.05)

    pearson_r, pearson_p = pearsonr(x, y)

    return {
        "result": result,
        "x_grid": x_grid,
        "predicted_mean": prediction["mean"].to_numpy(dtype=float),
        "ci_lower": prediction["mean_ci_lower"].to_numpy(dtype=float),
        "ci_upper": prediction["mean_ci_upper"].to_numpy(dtype=float),
        "n": len(fit_data),
        "r": float(pearson_r),
        "r_p": float(pearson_p),
        "slope": float(result.params[1]),
        "slope_p": float(result.pvalues[1]),
    }


def clean_scene_filter(df, scene_name):
    if "scene_name" in df.columns:
        mask = df["scene_name"].astype(str).str.strip().eq(scene_name)
        if mask.any():
            return df.loc[mask].copy()

    for column in ["scene", "scene_dir", "path"]:
        if column in df.columns:
            normalized = df[column].astype(str).str.replace("\\", "/", regex=False)
            mask = normalized.str.contains(
                rf"/{re.escape(scene_name)}(?:/|$)",
                regex=True,
            )
            if mask.any():
                return df.loc[mask].copy()

    raise KeyError(f"Could not find {scene_name!r} in the supplied table.")


def find_scene_dir(scene_name):
    if not TEST_DATA_ROOT.exists():
        raise FileNotFoundError(f"Missing testing-data root: {TEST_DATA_ROOT}")

    exact = [
        p
        for p in TEST_DATA_ROOT.rglob(scene_name)
        if p.is_dir() and p.name == scene_name
    ]
    if len(exact) == 1:
        return exact[0]
    if len(exact) > 1:
        exp1 = [p for p in exact if p.parent.name == "exp1"]
        if len(exp1) == 1:
            return exp1[0]
        raise RuntimeError(f"Multiple directories matched {scene_name}: {exact}")
    raise FileNotFoundError(
        f"Could not find scene directory for {scene_name!r} under {TEST_DATA_ROOT}"
    )


def load_background_image(scene_dir, frame_index=N_HISTORY - 1):
    scene_dir = Path(scene_dir)
    expected = scene_dir / "frames" / f"frame_{frame_index:04d}.png"
    if expected.exists():
        return Image.open(expected).convert("RGB")

    files = sorted((scene_dir / "frames").glob("frame_*.png"))
    if not files:
        files = sorted((scene_dir / "frames").glob("*.png"))
    if not files:
        raise FileNotFoundError(f"No frame PNGs found in {scene_dir / 'frames'}")

    return Image.open(files[min(frame_index, len(files) - 1)]).convert("RGB")


def load_true_positions(scene_dir):
    path = Path(scene_dir) / "simulation_dataset.csv"
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_csv(path)
    require_columns(df, ["ball_x", "ball_y"], str(path))

    if "frame_index" in df.columns:
        df = df.sort_values("frame_index")
    elif "frame" in df.columns:
        df = df.sort_values("frame")

    return (
        df.dropna(subset=["ball_x", "ball_y"])
        .reset_index(drop=True)
    )


def make_publication_scene_background(image, background_gray=230):
    arr = np.asarray(image.convert("RGB")).copy()
    rgb = arr.astype(float)

    luminance = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )
    chroma = rgb.max(axis=-1) - rgb.min(axis=-1)
    neutral = chroma < 22

    dark_background = neutral & (luminance < 85)
    bright_slides_or_borders = neutral & (luminance > 150)

    arr[dark_background] = np.array([background_gray] * 3, dtype=np.uint8)
    arr[bright_slides_or_borders] = np.array([0, 0, 0], dtype=np.uint8)

    # Remove the scene/frame label in the original rendering.
    arr[8:60, 8:245] = np.array([background_gray] * 3, dtype=np.uint8)

    return Image.fromarray(arr)


def collapse_collision_stage(label):
    label = str(label)
    if label == "before_c1":
        return "Before collision"
    if label == "c1":
        return "During collision"
    if label == "after_c1" or label in {"c2", "before_c3", "c3", "before_c4"}:
        return "After collision"
    if re.match(r"^(c[2-9]\d*|before_c[3-9]\d*)$", label):
        return "After collision"
    return np.nan

## 1. Non-straight-path RNN vs. true-simulation path efficiency

In [ ]:
# ============================================================
# 1. Non-straight-path RNN vs. true-simulation path efficiency
# ============================================================

scene_summary = pd.read_csv(SCENE_SUMMARY_PATH, low_memory=False)

require_columns(
    scene_summary,
    ["scene_name", "true_path_eff", "rnn_path_eff", "straight_path"],
    "scene_summary.csv",
)

# ------------------------------------------------------------
# Prepare non-straight-path scenes
# ------------------------------------------------------------

path_df = clean_numeric(
    scene_summary[
        ["scene_name", "true_path_eff", "rnn_path_eff", "straight_path"]
    ],
    ["true_path_eff", "rnn_path_eff", "straight_path"],
)

path_df = (
    path_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["true_path_eff", "rnn_path_eff", "straight_path"])
    .loc[lambda d: d["straight_path"].astype(int).eq(0)]
    .copy()
)

path_df["path_efficiency_difference"] = (
    path_df["rnn_path_eff"] - path_df["true_path_eff"]
)

difference_values = path_df[
    "path_efficiency_difference"
].to_numpy(dtype=float)


# ============================================================
# Statistical test
# Paired, two-sided Wilcoxon signed-rank test
# ============================================================

wilcoxon_result = wilcoxon(
    path_df["rnn_path_eff"].to_numpy(dtype=float),
    path_df["true_path_eff"].to_numpy(dtype=float),
    alternative="two-sided",
    zero_method="wilcox",
    method="auto",
)

path_efficiency_stats = pd.DataFrame(
    {
        "n_scenes": [len(path_df)],
        "median_true_path_efficiency": [
            path_df["true_path_eff"].median()
        ],
        "median_rnn_path_efficiency": [
            path_df["rnn_path_eff"].median()
        ],
        "median_rnn_minus_true": [
            np.median(difference_values)
        ],
        "wilcoxon_W": [
            wilcoxon_result.statistic
        ],
        "p_value": [
            wilcoxon_result.pvalue
        ],
    }
)

display(path_efficiency_stats)


# ============================================================
# Raincloud figure
# ============================================================

value_range = (
    difference_values.max()
    - difference_values.min()
)

padding = (
    0.12 * value_range
    if value_range > 0
    else 0.05
)

x_grid = np.linspace(
    difference_values.min() - padding,
    difference_values.max() + padding,
    500,
)

density = scaled_density(
    difference_values,
    x_grid,
    maximum_height=0.38,
)


fig, ax = plt.subplots(
    figsize=STANDARD_FIGSIZE
)

cloud_baseline = 0.92
box_y = 0.42


# ------------------------------------------------------------
# Density cloud
# ------------------------------------------------------------

ax.fill_between(
    x_grid,
    cloud_baseline,
    cloud_baseline + density,
    color=NON_STRAIGHT_COLOR,
    alpha=0.35,
    linewidth=0,
    zorder=1,
)

ax.plot(
    x_grid,
    cloud_baseline + density,
    color=NON_STRAIGHT_COLOR,
    linewidth=2.2,
    zorder=2,
)


# ------------------------------------------------------------
# Boxplot
# ------------------------------------------------------------

ax.boxplot(
    difference_values,
    vert=False,
    positions=[box_y],
    widths=0.20,
    showfliers=True,
    patch_artist=True,
    manage_ticks=False,

    boxprops={
        "facecolor": "white",
        "edgecolor": NON_STRAIGHT_COLOR,
        "linewidth": 1.8,
    },

    whiskerprops={
        "color": NON_STRAIGHT_COLOR,
        "linewidth": 1.5,
    },

    capprops={
        "color": NON_STRAIGHT_COLOR,
        "linewidth": 1.5,
    },

    medianprops={
        "color": "black",
        "linewidth": 2.0,
    },

    flierprops={
        "marker": "o",
        "markerfacecolor": NON_STRAIGHT_COLOR,
        "markeredgecolor": NON_STRAIGHT_COLOR,
        "markersize": 5,
        "alpha": 0.9,
    },

    zorder=4,
)


# ------------------------------------------------------------
# Equality reference
# ------------------------------------------------------------

ax.axvline(
    0,
    color=POOLED_COLOR,
    linestyle="--",
    linewidth=1.5,
    zorder=0,
)


# ------------------------------------------------------------
# Labels and styling
# ------------------------------------------------------------

ax.set_title(
    "Non-straight-path scenes: RNN vs true path efficiency",
    fontsize=19,
    pad=14,
)

ax.set_xlabel(
    "RNN path efficiency − true path efficiency",
    fontsize=17,
    labelpad=10,
)

ax.set_yticks([])

ax.set_ylim(
    0.15,
    1.42,
)

ax.tick_params(
    axis="x",
    labelsize=14,
)


# Light x-axis grid
ax.grid(
    axis="x",
    alpha=0.18,
    zorder=0,
)


# Full boxed frame, consistent with the other figures
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.1)
    spine.set_color("black")


fig.tight_layout()

plt.show()

## 2. Mixed-effects model predicting RNN-predicted error

In [ ]:
# Fit the exact saved row-level model used in the source notebook:
#
# log1p_rnn_predicted_error
# ~ future_distance_z
# + c1_vs_before_c1
# + after_c1_vs_c1
# + scene_total_frames_z
# + scene_n_objects_z
# + (1 + future_distance_z | scene_key)

if not LMER_MODEL_DATA_CSV.exists():
    raise FileNotFoundError(
        f"Could not find {LMER_MODEL_DATA_CSV}\n"
        "Run the mixed-model section of the original RNN analysis notebook once "
        "to create mixed_model_row_level_data.csv."
    )

model_df = pd.read_csv(LMER_MODEL_DATA_CSV, low_memory=False)

required_model_columns = [
    "scene_key",
    "log1p_rnn_predicted_error",
    "future_distance_z",
    "c1_vs_before_c1",
    "after_c1_vs_c1",
    "scene_total_frames_z",
    "scene_n_objects_z",
]
require_columns(model_df, required_model_columns, "mixed_model_row_level_data.csv")

model_df = clean_numeric(
    model_df,
    [c for c in required_model_columns if c != "scene_key"],
)
model_df = (
    model_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=required_model_columns)
    .copy()
)

formula = (
    "log1p_rnn_predicted_error ~ "
    "future_distance_z "
    "+ c1_vs_before_c1 "
    "+ after_c1_vs_c1 "
    "+ scene_total_frames_z "
    "+ scene_n_objects_z"
)

mixed_model = smf.mixedlm(
    formula=formula,
    data=model_df,
    groups=model_df["scene_key"],
    re_formula="1 + future_distance_z",
)

mixed_result = None
fit_attempts = []

for method in ["lbfgs", "powell", "cg"]:
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            candidate = mixed_model.fit(
                method=method,
                reml=False,
                maxiter=2000,
                disp=False,
            )
        fit_attempts.append((method, bool(candidate.converged), float(candidate.llf)))
        if candidate.converged and np.isfinite(candidate.llf):
            mixed_result = candidate
            break
    except Exception:
        fit_attempts.append((method, False, np.nan))

if mixed_result is None:
    raise RuntimeError(f"No optimizer produced a converged fit: {fit_attempts}")

# Nakagawa-style marginal/conditional R² exactly as in the source notebook.
fixed_linear_predictor = (
    np.asarray(mixed_result.model.exog, dtype=float)
    @ np.asarray(mixed_result.fe_params, dtype=float)
)
var_fixed = float(np.var(fixed_linear_predictor, ddof=0))

random_design = np.asarray(mixed_result.model.exog_re, dtype=float)
random_covariance = np.asarray(mixed_result.cov_re, dtype=float)
random_variance_by_row = np.einsum(
    "ij,jk,ik->i",
    random_design,
    random_covariance,
    random_design,
)
var_random = float(np.mean(random_variance_by_row))
var_residual = float(mixed_result.scale)

variance_total = var_fixed + var_random + var_residual
marginal_r2 = var_fixed / variance_total
conditional_r2 = (var_fixed + var_random) / variance_total

# Literal planned contrasts. These are required because the two collision
# coding coefficients in the raw summary are not themselves the literal
# "during - before" and "after - during" stage differences.
fe_names = list(mixed_result.fe_params.index)
beta = mixed_result.fe_params.loc[fe_names].to_numpy(dtype=float)
cov_fe = (
    mixed_result.cov_params()
    .loc[fe_names, fe_names]
    .to_numpy(dtype=float)
)
name_to_idx = {name: i for i, name in enumerate(fe_names)}


def make_contrast(weights, label):
    L = np.zeros(len(fe_names), dtype=float)
    for name, weight in weights.items():
        L[name_to_idx[name]] = weight

    estimate = float(L @ beta)
    variance = float(L @ cov_fe @ L)
    se = float(np.sqrt(variance))
    z_value = estimate / se
    p_value = float(2 * norm.sf(abs(z_value)))

    return {
        "Parameter": label,
        "beta": estimate,
        "SE": se,
        "z": z_value,
        "CI_lower": estimate - 1.96 * se,
        "CI_upper": estimate + 1.96 * se,
        "p_value": p_value,
    }


contrast_table = pd.DataFrame([
    make_contrast(
        {"future_distance_z": 1},
        "Prediction horizon",
    ),
    make_contrast(
        {
            "c1_vs_before_c1": 1,
            "after_c1_vs_c1": -0.5,
        },
        "During vs. before collision",
    ),
    make_contrast(
        {
            "c1_vs_before_c1": -0.5,
            "after_c1_vs_c1": 1,
        },
        "After vs. during collision",
    ),
    make_contrast(
        {"scene_total_frames_z": 1},
        "Scene length",
    ),
    make_contrast(
        {"scene_n_objects_z": 1},
        "Number of objects",
    ),
])

contrast_table["95% CI"] = contrast_table.apply(
    lambda row: f"[{row['CI_lower']:.3f}, {row['CI_upper']:.3f}]",
    axis=1,
)
contrast_table["p"] = contrast_table["p_value"].map(
    lambda p: "< .001" if p < 0.001 else f"{p:.3f}"
)

final_contrast_table = contrast_table[
    ["Parameter", "beta", "SE", "z", "95% CI", "p"]
].copy()

print("Raw MixedLM result")
print(mixed_result.summary())
print(
    f"\nMarginal R² = {marginal_r2:.3f}; "
    f"Conditional R² = {conditional_r2:.3f}"
)
print("\nLiteral contrast-calculated parameter table")
display(
    final_contrast_table.style.format({
        "beta": "{:.3f}",
        "SE": "{:.3f}",
        "z": "{:.3f}",
    })
)

## 3. Scene-average RNN-predicted error vs. pupil size

In [ ]:
# Build the exact scene-level pupil/error analysis table used by the
# source notebook. This cell intentionally produces no output.

human = pd.read_csv(
    CLASSIFIED_TRIAL_PATH,
    dtype={"subject_id": "string", "scene_name": "string"},
    low_memory=False,
)
if "subject_id" not in human.columns and "id" in human.columns:
    human["subject_id"] = human["id"].astype("string")

require_columns(
    human,
    ["subject_id", "scene_name", "pupil"],
    "classified_trial_data.csv",
)

human["subject_id"] = (
    human["subject_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"^(\d+)\.0$", r"\1", regex=True)
)
human["scene_name"] = human["scene_name"].astype("string").str.strip()
human["pupil"] = pd.to_numeric(human["pupil"], errors="coerce")

human = (
    human
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["subject_id", "scene_name", "pupil"])
    .loc[lambda d: d["pupil"] > 0]
    .copy()
)
human["log_pupil"] = np.log(human["pupil"])

participant_scene_pupil = (
    human
    .groupby(["scene_name", "subject_id"], as_index=False)
    .agg(participant_mean_log_pupil=("log_pupil", "mean"))
)

scene_pupil = (
    participant_scene_pupil
    .groupby("scene_name", as_index=False)
    .agg(
        scene_mean_log_pupil_across_participants=(
            "participant_mean_log_pupil",
            "mean",
        )
    )
)

pred_df = pd.read_csv(RNN_PREDICTION_CSV, low_memory=False)
require_columns(
    pred_df,
    ["scene_name", "predicted_error_px", "is_terminal_prediction"],
    "rnn_predictions_until_contact.csv",
)

pred_df["scene_name"] = pred_df["scene_name"].astype(str).str.strip()
pred_df["predicted_error_px"] = pd.to_numeric(
    pred_df["predicted_error_px"],
    errors="coerce",
)
pred_df["is_terminal_prediction"] = pd.to_numeric(
    pred_df["is_terminal_prediction"],
    errors="coerce",
)

clean_predictions = (
    pred_df
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=["scene_name", "predicted_error_px"])
    .copy()
)

scene_average_error = (
    clean_predictions
    .groupby("scene_name", as_index=False)
    .agg(
        scene_average_rnn_predicted_error=(
            "predicted_error_px",
            "mean",
        )
    )
)
scene_average_error["log10_scene_average_rnn_predicted_error"] = np.where(
    scene_average_error["scene_average_rnn_predicted_error"] > 0,
    np.log10(scene_average_error["scene_average_rnn_predicted_error"]),
    np.nan,
)

# Preserve the original sample definition: every retained prediction scene
# must also have exactly one terminal-prediction row.
terminal_rows = clean_predictions.loc[
    clean_predictions["is_terminal_prediction"].eq(1)
].copy()
terminal_counts = terminal_rows.groupby("scene_name").size()
if (terminal_counts != 1).any():
    raise RuntimeError("Expected exactly one terminal prediction per scene.")

all_prediction_scenes = set(clean_predictions["scene_name"].unique())
terminal_scenes = set(terminal_rows["scene_name"].unique())
missing_terminal = sorted(all_prediction_scenes - terminal_scenes)
if missing_terminal:
    raise RuntimeError(
        "Missing terminal prediction rows for: "
        + ", ".join(missing_terminal[:20])
    )

terminal_scene_key = terminal_rows[["scene_name"]].drop_duplicates()

path_lookup = scene_summary[["scene_name", "straight_path"]].copy()
path_lookup["scene_name"] = path_lookup["scene_name"].astype(str).str.strip()
path_lookup["straight_path"] = pd.to_numeric(
    path_lookup["straight_path"],
    errors="coerce",
)
path_lookup = (
    path_lookup
    .dropna(subset=["scene_name", "straight_path"])
    .drop_duplicates(subset=["scene_name"], keep="first")
)

X_COLUMN = "log10_scene_average_rnn_predicted_error"
Y_COLUMN = "scene_mean_log_pupil_across_participants"

scene_analysis_df = (
    scene_pupil
    .merge(
        scene_average_error,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        terminal_scene_key,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        path_lookup,
        on="scene_name",
        how="left",
        validate="one_to_one",
    )
    .replace([np.inf, -np.inf], np.nan)
    .loc[lambda d: ~d["scene_name"].isin(EXCLUDED_SCENES)]
    .dropna(subset=[X_COLUMN, Y_COLUMN, "straight_path"])
    .sort_values("scene_name")
    .reset_index(drop=True)
)

scene_analysis_df["straight_path"] = (
    scene_analysis_df["straight_path"].astype(int)
)

In [ ]:
# Pearson r and OLS slope test for each of the three plotted lines.
group_definitions = [
    ("All scenes", None, POOLED_COLOR),
    ("Non-straight path", 0, NON_STRAIGHT_COLOR),
    ("Straight path", 1, STRAIGHT_COLOR),
]

fit_results = {}
stats_rows = []

for label, path_value, color in group_definitions:
    group = (
        scene_analysis_df
        if path_value is None
        else scene_analysis_df.loc[
            scene_analysis_df["straight_path"].eq(path_value)
        ]
    )

    fit = fit_simple_ols_with_ci(
        group,
        X_COLUMN,
        Y_COLUMN,
    )
    fit_results[label] = fit

    stats_rows.append({
        "line": label,
        "n_scenes": fit["n"],
        "pearson_r": fit["r"],
        "pearson_p": fit["r_p"],
        "ols_slope": fit["slope"],
        "ols_slope_p": fit["slope_p"],
    })

line_stats = pd.DataFrame(stats_rows)

print("Line-specific correlation and slope tests")
display(
    line_stats.style.format({
        "pearson_r": "{:.5f}",
        "pearson_p": "{:.6g}",
        "ols_slope": "{:.5f}",
        "ols_slope_p": "{:.6g}",
    })
)

fig, ax = plt.subplots(figsize=STANDARD_FIGSIZE)

# Scene points are color-coded by path type.
for path_value in [0, 1]:
    group = scene_analysis_df.loc[
        scene_analysis_df["straight_path"].eq(path_value)
    ]
    color = PATH_COLORS[path_value]

    ax.scatter(
        group[X_COLUMN],
        group[Y_COLUMN],
        s=62,
        alpha=0.80,
        color=color,
        edgecolors="black",
        linewidths=0.4,
        zorder=3,
    )

# Draw pooled CI first, then path-specific CIs/lines.
pooled_fit = fit_results["All scenes"]

ax.fill_between(
    pooled_fit["x_grid"],
    pooled_fit["ci_lower"],
    pooled_fit["ci_upper"],
    color=POOLED_COLOR,
    alpha=CI_ALPHA,
    linewidth=0,
    zorder=1,
)

ax.plot(
    pooled_fit["x_grid"],
    pooled_fit["predicted_mean"],
    color=POOLED_COLOR,
    linewidth=2.8,
    zorder=4,
)

for label, path_value, color in group_definitions[1:]:
    fit = fit_results[label]

    ax.fill_between(
        fit["x_grid"],
        fit["ci_lower"],
        fit["ci_upper"],
        color=color,
        alpha=CI_ALPHA,
        linewidth=0,
        zorder=1,
    )

    ax.plot(
        fit["x_grid"],
        fit["predicted_mean"],
        color=color,
        linewidth=2.8,
        zorder=5,
    )

legend_handles = [
    Line2D(
        [0], [0],
        color=NON_STRAIGHT_COLOR,
        lw=2.8,
        marker="o",
        markersize=7,
        markeredgecolor="black",
        markeredgewidth=0.4,
        label="Non-straight path",
    ),
    Line2D(
        [0], [0],
        color=STRAIGHT_COLOR,
        lw=2.8,
        marker="o",
        markersize=7,
        markeredgecolor="black",
        markeredgewidth=0.4,
        label="Straight path",
    ),
    Line2D(
        [0], [0],
        color=POOLED_COLOR,
        lw=2.8,
        label="All scenes",
    ),
]

ax.legend(
    handles=legend_handles,
    frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="black",
    fontsize=14,
)

# Apply standard axis styling first
style_numeric_axis(ax)

# Full boxed frame
for spine in ax.spines.values():
    spine.set_visible(True)
    spine.set_linewidth(1.1)
    spine.set_color("black")
    
# Match text sizing to the first figure
ax.set_title(
    "Scene-average pupil size vs RNN-predicted error",
    fontsize=19,
    pad=14,
)


ax.set_xlabel(
    "Scene-average log10 RNN-predicted error",
    fontsize=17,
    labelpad=10,
)

ax.set_ylabel(
    "Scene-average ln pupil size",
    fontsize=17,
    labelpad=10,
)

ax.tick_params(
    axis="both",
    labelsize=14,
)

fig.tight_layout()
plt.show()

## 4. `high_yescol_nosp_3` RNN prediction illustration and error curve

In [ ]:
# ============================================================
# Explicit data preparation for the two RNN illustration figures
# ============================================================

scene_dir = find_scene_dir(
    TARGET_SCENE
)

background = make_publication_scene_background(
    load_background_image(
        scene_dir,
        frame_index=N_HISTORY - 1,
    )
)

true_position_df = load_true_positions(
    scene_dir
)

true_xy = (
    true_position_df[
        [
            "ball_x",
            "ball_y",
        ]
    ]
    .to_numpy(
        dtype=float
    )
)


# ------------------------------------------------------------
# RNN future-prediction table
# ------------------------------------------------------------
rnn_all = pd.read_csv(
    RNN_PREDICTION_CSV,
    low_memory=False,
)

rnn_df = clean_scene_filter(
    rnn_all,
    TARGET_SCENE,
)


def _first_existing_column(
    dataframe,
    candidates,
):
    return next(
        (
            column
            for column in candidates
            if column in dataframe.columns
        ),
        None,
    )


rename_map = {}

column_candidates = {
    "future_offset": [
        "future_offset",
        "prediction_horizon",
        "future_distance",
        "offset",
        "rnn_pred_step",
    ],
    "predicted_x": [
        "predicted_x",
        "pred_x",
        "x_pred",
        "rnn_pred_x",
    ],
    "predicted_y": [
        "predicted_y",
        "pred_y",
        "y_pred",
        "rnn_pred_y",
    ],
    "predicted_error_px": [
        "predicted_error_px",
        "predicted_error",
        "error_px",
        "rnn_predicted_error",
    ],
}

for canonical, candidates in column_candidates.items():

    source_column = _first_existing_column(
        rnn_df,
        candidates,
    )

    if source_column is None:
        raise KeyError(
            f"Could not identify {canonical}. "
            f"Available RNN prediction columns: "
            f"{list(rnn_df.columns)}"
        )

    if source_column != canonical:
        rename_map[
            source_column
        ] = canonical


rnn_df = rnn_df.rename(
    columns=rename_map
)

for column in [
    "future_offset",
    "predicted_x",
    "predicted_y",
    "predicted_error_px",
]:
    rnn_df[column] = pd.to_numeric(
        rnn_df[column],
        errors="coerce",
    )


rnn_df = (
    rnn_df
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna(
        subset=[
            "future_offset",
            "predicted_x",
            "predicted_y",
            "predicted_error_px",
        ]
    )
    .sort_values(
        "future_offset"
    )
    .reset_index(
        drop=True
    )
)


# ------------------------------------------------------------
# Collision-stage labels for the predicted-error curve
# ------------------------------------------------------------
collision_all = pd.read_csv(
    COLLISION_LABEL_CSV,
    low_memory=False,
)

collision_scene = clean_scene_filter(
    collision_all,
    TARGET_SCENE,
)


def _detect_stage_column(
    dataframe,
):

    preferred = [
        "collision_stage",
        "collision_phase",
        "phase",
        "stage",
        "phase_label",
        "collision_label",
        "container_or_line_circle_phase",
    ]

    for column in preferred:
        if column in dataframe.columns:
            return column

    # Fall back to a string column whose values resemble the
    # original before_c1 / c1 / after_c1 coding.
    pattern = re.compile(
        r"^(?:before_c\d+|after_c\d+|c\d+)$"
    )

    for column in dataframe.columns:

        values = (
            dataframe[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        if len(values) == 0:
            continue

        match_rate = np.mean(
            [
                bool(
                    pattern.match(
                        value
                    )
                )
                for value in values.head(
                    200
                )
            ]
        )

        if match_rate >= 0.20:
            return column

    raise KeyError(
        "Could not identify a collision-stage/phase column. "
        f"Available columns: {list(dataframe.columns)}"
    )


stage_column = _detect_stage_column(
    collision_scene
)


raw_stage = (
    collision_scene[
        stage_column
    ]
    .astype(str)
    .str.strip()
)


# Accept either the original coded labels or already-collapsed labels.
collapsed_stage = raw_stage.map(
    collapse_collision_stage
)

already_collapsed = raw_stage.where(
    raw_stage.isin(
        [
            "Before collision",
            "During collision",
            "After collision",
        ]
    )
)

collision_scene = collision_scene.copy()

collision_scene[
    "collision_stage"
] = collapsed_stage.fillna(
    already_collapsed
)


# ------------------------------------------------------------
# Align collision stages to RNN future offsets
# ------------------------------------------------------------
offset_column = _first_existing_column(
    collision_scene,
    [
        "future_offset",
        "prediction_horizon",
        "future_distance",
        "offset",
        "rnn_pred_step",
    ],
)

if offset_column is not None:

    collision_scene[
        "_future_offset"
    ] = pd.to_numeric(
        collision_scene[
            offset_column
        ],
        errors="coerce",
    )

else:

    frame_column = _first_existing_column(
        collision_scene,
        [
            "frame_index",
            "frame",
            "frame_id",
        ],
    )

    if frame_column is not None:

        # History contains frames 0..14, so absolute frame 15
        # corresponds to future offset 1.
        collision_scene[
            "_future_offset"
        ] = (
            pd.to_numeric(
                collision_scene[
                    frame_column
                ],
                errors="coerce",
            )
            - (
                N_HISTORY
                - 1
            )
        )

    elif len(
        collision_scene
    ) == len(
        rnn_df
    ):

        collision_scene = (
            collision_scene
            .reset_index(
                drop=True
            )
        )

        collision_scene[
            "_future_offset"
        ] = rnn_df[
            "future_offset"
        ].to_numpy(
            dtype=float
        )

    else:

        raise RuntimeError(
            "Could not align collision labels to RNN prediction horizons. "
            "No offset/frame column was found and row counts differ."
        )


stage_lookup = (
    collision_scene[
        [
            "_future_offset",
            "collision_stage",
        ]
    ]
    .replace(
        [np.inf, -np.inf],
        np.nan,
    )
    .dropna(
        subset=[
            "_future_offset",
            "collision_stage",
        ]
    )
    .drop_duplicates(
        subset=[
            "_future_offset"
        ],
        keep="first",
    )
)


lmer_scene_df = (
    rnn_df[
        [
            "future_offset",
            "predicted_error_px",
        ]
    ]
    .merge(
        stage_lookup,
        left_on="future_offset",
        right_on="_future_offset",
        how="left",
        validate="one_to_one",
    )
    .drop(
        columns=[
            "_future_offset"
        ]
    )
    .sort_values(
        "future_offset"
    )
    .reset_index(
        drop=True
    )
)


# The collision-label table can end a few frames before the RNN
# prediction table. Collision stage is a persistent phase label, so
# carry the last available stage forward across any unlabeled tail.
# bfill() also makes the code robust to an unlabeled initial offset.
lmer_scene_df[
    "collision_stage"
] = (
    lmer_scene_df[
        "collision_stage"
    ]
    .ffill()
    .bfill()
)


if lmer_scene_df[
    "collision_stage"
].isna().any():
    raise RuntimeError(
        "No usable collision-stage labels were found for the "
        "RNN prediction-error illustration."
    )


In [ ]:
# ============================================================
# Figure text-size settings
# ============================================================

FIGURE_TITLE_SIZE = 19
LEGEND_TEXT_SIZE = 13
COLORBAR_LABEL_SIZE = 17


# ============================================================
# Shared horizon-to-color mapping
# ============================================================

horizon_min = float(rnn_df["future_offset"].min())
horizon_max = float(rnn_df["future_offset"].max())

horizon_norm = Normalize(
    vmin=horizon_min,
    vmax=horizon_max,
)

horizon_cmap = plt.cm.viridis


# ============================================================
# Figure 1: RNN future predictions
# ============================================================

fig_scene, ax_scene = plt.subplots(
    figsize=(6.2, 7.8)
)

ax_scene.set_facecolor(SCENE_BG_COLOR)

ax_scene.imshow(
    background,
    zorder=0,
)


# ------------------------------------------------------------
# Pure simulation trajectory
# ------------------------------------------------------------

true_future = true_xy[N_HISTORY - 1:]

ax_scene.plot(
    true_future[:, 0],
    true_future[:, 1],
    color=TRUE_COLOR,
    linewidth=2.3,
    alpha=0.88,
    zorder=1,
)


# ------------------------------------------------------------
# Show one RNN prediction every 10 frames,
# plus the final prediction
# ------------------------------------------------------------

offsets = rnn_df[
    "future_offset"
].to_numpy(dtype=int)

keep = (
    offsets % 10 == 0
)

keep[-1] = True

shown = rnn_df.loc[
    keep
].copy()

xy = shown[
    [
        "predicted_x",
        "predicted_y",
    ]
].to_numpy(dtype=float)

errors = shown[
    "predicted_error_px"
].to_numpy(dtype=float)

horizons = shown[
    "future_offset"
].to_numpy(dtype=float)

point_colors = horizon_cmap(
    horizon_norm(horizons)
)


# ------------------------------------------------------------
# Predicted-error circles
# ------------------------------------------------------------

for (x_pos, y_pos), radius, color in zip(
    xy,
    errors,
    point_colors,
):
    if np.isfinite(radius) and radius > 0:

        ax_scene.add_patch(
            plt.Circle(
                (x_pos, y_pos),
                radius,
                fill=False,
                linestyle=(0, (3, 2)),
                linewidth=1.15,
                alpha=0.72,
                color=color,
                zorder=2,
            )
        )


# ------------------------------------------------------------
# RNN-predicted positions
# ------------------------------------------------------------

ax_scene.scatter(
    xy[:, 0],
    xy[:, 1],
    s=40,
    c=point_colors,
    edgecolors="black",
    linewidths=0.45,
    zorder=4,
)


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

scene_handles = [
    Line2D(
        [0], [0],
        color=TRUE_COLOR,
        lw=2.3,
        label="Pure simulation trajectory",
    ),
    Line2D(
        [0], [0],
        color=POOLED_COLOR,
        lw=1.15,
        ls="--",
        label="Predicted error radius",
    ),
]

ax_scene.legend(
    handles=scene_handles,
    loc="upper right",
    frameon=True,
    framealpha=0.90,
    fontsize=LEGEND_TEXT_SIZE,
)


# ------------------------------------------------------------
# Scene limits and title
# ------------------------------------------------------------

ax_scene.set_xlim(
    0,
    ORIGINAL_FRAME_WIDTH,
)

ax_scene.set_ylim(
    ORIGINAL_FRAME_HEIGHT,
    0,
)

ax_scene.set_title(
    "RNN future predictions",
    fontsize=FIGURE_TITLE_SIZE,
    pad=14,
)

ax_scene.axis("off")


# ------------------------------------------------------------
# Individual colorbar for scene figure
# ------------------------------------------------------------

# ------------------------------------------------------------
# Scene limits and title
# ------------------------------------------------------------

ax_scene.set_xlim(
    0,
    ORIGINAL_FRAME_WIDTH,
)

ax_scene.set_ylim(
    ORIGINAL_FRAME_HEIGHT,
    0,
)

ax_scene.set_title(
    "RNN future predictions",
    fontsize=FIGURE_TITLE_SIZE,
    pad=14,
)

ax_scene.axis("off")

fig_scene.tight_layout()

plt.show()


# ============================================================
# Figure 2: Predicted error vs prediction horizon
# ============================================================

curve_df = lmer_scene_df.copy()

x = curve_df[
    "future_offset"
].to_numpy(dtype=float)

predicted_error = curve_df[
    "predicted_error_px"
].to_numpy(dtype=float)


# ------------------------------------------------------------
# Define collision-stage regions
# ------------------------------------------------------------

if len(x) > 1:

    mids = (
        x[:-1] + x[1:]
    ) / 2.0

    left_edge = (
        x[0]
        - (x[1] - x[0]) / 2.0
    )

    right_edge = (
        x[-1]
        + (x[-1] - x[-2]) / 2.0
    )

else:

    mids = np.array([])

    left_edge = (
        x[0] - 0.5
    )

    right_edge = (
        x[0] + 0.5
    )


stage_colors = {
    "Before collision": BEFORE_COLOR,
    "During collision": DURING_COLOR,
    "After collision": AFTER_COLOR,
}

stages = (
    curve_df[
        "collision_stage"
    ]
    .astype(str)
    .to_numpy()
)

run_starts = np.r_[
    0,
    np.where(
        stages[1:] != stages[:-1]
    )[0] + 1,
]

run_ends = np.r_[
    run_starts[1:] - 1,
    len(curve_df) - 1,
]


# ============================================================
# Plot
# ============================================================

fig_error, ax_error = plt.subplots(
    figsize=STANDARD_FIGSIZE
)


# ------------------------------------------------------------
# Collision-stage background shading
# ------------------------------------------------------------

for run_start, run_end in zip(
    run_starts,
    run_ends,
):

    stage = stages[
        run_start
    ]

    left = (
        left_edge
        if run_start == 0
        else mids[run_start - 1]
    )

    right = (
        right_edge
        if run_end == len(curve_df) - 1
        else mids[run_end]
    )

    ax_error.axvspan(
        left,
        right,
        color=stage_colors.get(
            stage,
            POOLED_COLOR,
        ),
        alpha=0.08,
        zorder=0,
    )

    # ------------------------------------------------------------
# Collision-stage labels
# ------------------------------------------------------------

# ------------------------------------------------------------
# Collision-stage background shading + labels
# ------------------------------------------------------------

stage_labels = {
    "Before collision": "Before\ncollision",
    "During collision": "During\ncollision",
    "After collision": "After\ncollision",
}

for run_start, run_end in zip(
    run_starts,
    run_ends,
):

    stage = stages[
        run_start
    ]

    left = (
        left_edge
        if run_start == 0
        else mids[run_start - 1]
    )

    right = (
        right_edge
        if run_end == len(curve_df) - 1
        else mids[run_end]
    )

    # Background shading
    ax_error.axvspan(
        left,
        right,
        color=stage_colors.get(
            stage,
            POOLED_COLOR,
        ),
        alpha=0.08,
        zorder=0,
    )

    # Stage label
    ax_error.text(
        (left + right) / 2.0,
        20,
        stage_labels.get(stage, stage),
        ha="center",
        va="center",
        fontsize=14,
        color=POOLED_COLOR,
        fontweight="semibold",
        zorder=5,
    )

# ------------------------------------------------------------
# Predicted-error curve colored by prediction horizon
# ------------------------------------------------------------

if len(x) > 1:

    points = np.column_stack(
        [
            x,
            predicted_error,
        ]
    )

    segments = np.stack(
        [
            points[:-1],
            points[1:],
        ],
        axis=1,
    )

    segment_horizons = (
        x[:-1] + x[1:]
    ) / 2.0

    lc = LineCollection(
        segments,
        cmap=horizon_cmap,
        norm=horizon_norm,
        linewidths=2.8,
        zorder=3,
    )

    lc.set_array(
        segment_horizons
    )

    ax_error.add_collection(
        lc
    )

else:

    ax_error.scatter(
        x,
        predicted_error,
        c=x,
        cmap=horizon_cmap,
        norm=horizon_norm,
        s=40,
        zorder=3,
    )


# ------------------------------------------------------------
# Axis limits
# ------------------------------------------------------------

error_range = (
    np.nanmax(predicted_error)
    - np.nanmin(predicted_error)
)

y_pad = max(
    2.0,
    0.06 * error_range,
)

ax_error.set_xlim(
    left_edge,
    right_edge,
)

ax_error.set_ylim(
    np.nanmin(predicted_error) - y_pad,
    np.nanmax(predicted_error) + y_pad,
)


# ------------------------------------------------------------
# Labels and styling
# ------------------------------------------------------------

style_numeric_axis(
    ax_error
)

# Full boxed frame
for spine in ax_error.spines.values():

    spine.set_visible(
        True
    )

    spine.set_linewidth(
        1.1
    )

    spine.set_color(
        "black"
    )


ax_error.set_title(
    "Predicted error vs prediction horizon",
    fontsize=FIGURE_TITLE_SIZE,
    pad=14,
)

ax_error.set_xlabel(
    "Prediction horizon (frames)",
    fontsize=17,
    labelpad=10,
)

ax_error.set_ylabel(
    "Predicted error (pixels)",
    fontsize=17,
    labelpad=10,
)

ax_error.tick_params(
    axis="both",
    labelsize=14,
)


# ------------------------------------------------------------
# Individual colorbar for error figure
# ------------------------------------------------------------

smappable_error = ScalarMappable(
    norm=horizon_norm,
    cmap=horizon_cmap,
)

smappable_error.set_array([])

cbar_error = fig_error.colorbar(
    smappable_error,
    ax=ax_error,
    fraction=0.04,
    pad=0.03,
)

cbar_error.set_label(
    "Prediction horizon (frames)",
    fontsize=COLORBAR_LABEL_SIZE,
    labelpad=10,
)

cbar_error.ax.tick_params(
    labelsize=TICK_SIZE,
)

cbar_error.ax.invert_yaxis()

fig_error.tight_layout()

plt.show()

# Part II — Meta-control model figures

In [ ]:
set_figure_prefix("02_meta_control")

# Meta-control model: final analysis figures

Clean notebook containing only the requested Meta-control model analyses.

**Figure text standard**
- Title: 19 pt
- Axis labels: 17 pt
- Tick labels: 14 pt
- Legend: 14 pt

The analysis logic, scene filtering, trajectory construction, threshold sweep, and statistical models follow the supplied source notebooks. No figures are saved in this notebook.


In [ ]:
# ============================================================
# 0. Imports, paths, settings, and shared visual standard
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
from PIL import Image
from IPython.display import display

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import pearsonr, wilcoxon, rankdata
from statsmodels.stats.multitest import multipletests




HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"

# -----------------------------
# Shared inputs
# -----------------------------
SCENE_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "scene_summary.csv"
HUMAN_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "empirical_scene_level_rt_accuracy_summary.csv"
CLASSIFIED_TRIAL_PATH = BASE_DIR / "data" / "eye_gaze" / "classified_trial_data.csv"

META_CONTROL_PARAMETER_LABEL = "A150_S20_e22"

META_CONTROL_SUMMARY_CANDIDATES = [
    BASE_DIR / "meta_control" / META_CONTROL_PARAMETER_LABEL / "hybrid_scene_summary.csv",
    BASE_DIR / "meta_control" / "combined_hybrid_scene_summary.csv",
]

META_CONTROL_PREDICTION_DIR = (
    BASE_DIR
    / "meta_control"
    / META_CONTROL_PARAMETER_LABEL
    / "hybrid_predictions"
)

# Example scene
TARGET_SCENE = "high_yescol_nosp_3"
TEST_DATA_ROOT = BASE_DIR / "rnn_testing_data"

N_HISTORY = 15
ORIGINAL_FRAME_WIDTH = 800.0
ORIGINAL_FRAME_HEIGHT = 1024.0

# -----------------------------
# Analysis settings preserved from source notebook
# -----------------------------
FRAMES_PER_SECOND = 60.0
MS_PER_SECOND = 1000.0

THRESHOLDS = np.arange(50, 121, 10, dtype=float)
SCATTER_THRESHOLD = 120.0
TRACE_START_FRAME = 15

HUMAN_X_SHIFT = 0.0
HUMAN_Y_SHIFT = -40.0

META_CONTROL_X_SHIFT = 560.0
META_CONTROL_Y_SHIFT = 0.0

DISPLAY_X_MIN = 560.0
DISPLAY_X_MAX = 1360.0
DISPLAY_Y_MIN = 0.0
DISPLAY_Y_MAX = 1080.0

USE_STRICT_SEGMENT_Y_RANGE = True
INCLUDE_OVERLAP_IN_BOTH_POOLS = True

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260729

POLYNOMIAL_ORDER = 8
EXCLUDED_EYE_SCENE = "low_yescol_yessp_4"

# -----------------------------
# Standard project colors
# -----------------------------
NON_STRAIGHT_COLOR = "#1C77C3"
STRAIGHT_COLOR = "#F39237"
POOLED_COLOR = "#4D4D4D"

ACCURACY_COLOR = "#1C77C3"
FIT_COLOR = "#4D4D4D"

TRUE_COLOR = "red"
ABSTRACTION_COLOR = "coral"
SIMULATION_COLOR = "green"

SCENE_BG_COLOR = "#E6E6E6"

CI_ALPHA = 0.16
THRESHOLD_CI_ALPHA = 0.18

PATH_GROUP_ORDER = ["no_straight_path", "straight_path"]
PATH_COLORS = {
    "no_straight_path": NON_STRAIGHT_COLOR,
    "straight_path": STRAIGHT_COLOR,
}

PATH_LABELS = {
    "no_straight_path": "Non-straight path",
    "straight_path": "Straight path",
}

# -----------------------------
# Figure text standard
# -----------------------------
TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_SIZE = 14
BASE_FONT_SIZE = 14

plt.rcParams.update({
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "legend.title_fontsize": LEGEND_SIZE,
    "figure.dpi": 120,
})

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


# -----------------------------
# Input checks
# -----------------------------
for path, label in [
    (SCENE_SUMMARY_PATH, "scene_summary.csv"),
    (HUMAN_SUMMARY_PATH, "empirical_scene_level_rt_accuracy_summary.csv"),
    (CLASSIFIED_TRIAL_PATH, "classified_trial_data.csv"),
    (META_CONTROL_PREDICTION_DIR, "Meta-control model prediction directory"),
    (TEST_DATA_ROOT, "RNN testing-data root"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

META_CONTROL_SUMMARY_PATH = next(
    (path for path in META_CONTROL_SUMMARY_CANDIDATES if path.exists()),
    None,
)

if META_CONTROL_SUMMARY_PATH is None:
    raise FileNotFoundError(
        "Could not find the Meta-control model scene summary. Tried:\n"
        + "\n".join(str(path) for path in META_CONTROL_SUMMARY_CANDIDATES)
    )

print("Meta-control model summary:", META_CONTROL_SUMMARY_PATH)
print("Meta-control model predictions:", META_CONTROL_PREDICTION_DIR)


In [ ]:
# ============================================================
# 1. Shared helper functions and behavioral data
# ============================================================

def safe_numeric(values):
    return pd.to_numeric(values, errors="coerce")


def normalize_scene_name(value):
    text = Path(str(value).strip().replace("\\", "/")).name
    for suffix in (".json", ".mp4", ".csv"):
        if text.lower().endswith(suffix):
            text = text[:-len(suffix)]
    return text.strip()


def find_first_column(dataframe, candidates, required=False, label="column"):
    column = next(
        (candidate for candidate in candidates if candidate in dataframe.columns),
        None,
    )
    if required and column is None:
        raise KeyError(
            f"Could not identify {label}. Tried {candidates}.\n"
            f"Available columns:\n{list(dataframe.columns)}"
        )
    return column


def in_display_bounds(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return (
        np.isfinite(x)
        & np.isfinite(y)
        & (x >= DISPLAY_X_MIN)
        & (x <= DISPLAY_X_MAX)
        & (y >= DISPLAY_Y_MIN)
        & (y <= DISPLAY_Y_MAX)
    )


def normalize_trace_value(value):
    text = str(value).strip().lower()
    if "abstract" in text:
        return "abstraction"
    if "sim" in text:
        return "simulation"
    return np.nan


def normalize_trace_series(values):
    return values.map(normalize_trace_value)


def fit_line_with_95_ci(x, y, n_line=300):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    design = sm.add_constant(x, has_constant="add")
    result = sm.OLS(y, design).fit()

    x_line = np.linspace(np.min(x), np.max(x), int(n_line))
    prediction_design = sm.add_constant(x_line, has_constant="add")
    prediction = result.get_prediction(
        prediction_design
    ).summary_frame(alpha=0.05)

    return {
        "result": result,
        "x_line": x_line,
        "mean": prediction["mean"].to_numpy(dtype=float),
        "ci_lower": prediction["mean_ci_lower"].to_numpy(dtype=float),
        "ci_upper": prediction["mean_ci_upper"].to_numpy(dtype=float),
    }


def polynomial_curve(x, y, order=8, n_line=400):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]

    if len(x) <= order:
        raise ValueError(
            f"Need more than {order} valid rows for an order-{order} curve."
        )

    fitted = np.polynomial.Polynomial.fit(
        x,
        y,
        deg=int(order),
    )

    x_line = np.linspace(
        np.min(x),
        np.max(x),
        int(n_line),
    )

    y_line = fitted(x_line)

    return x_line, y_line


def add_standard_text_sizes(ax):
    ax.title.set_fontsize(TITLE_SIZE)
    ax.xaxis.label.set_fontsize(AXIS_LABEL_SIZE)
    ax.yaxis.label.set_fontsize(AXIS_LABEL_SIZE)
    ax.tick_params(
        axis="both",
        labelsize=TICK_SIZE,
    )


# ------------------------------------------------------------
# Common scene metadata
# ------------------------------------------------------------
scene_summary_raw = pd.read_csv(
    SCENE_SUMMARY_PATH,
    low_memory=False,
)

scene_name_column = find_first_column(
    scene_summary_raw,
    [
        "scene_name",
        "scene",
        "scene_id",
        "json_name",
        "trial_name",
        "trial",
        "name",
    ],
    required=True,
    label="scene-name column in scene_summary.csv",
)

if "simulation_time" not in scene_summary_raw.columns:
    raise KeyError("scene_summary.csv must contain `simulation_time`.")

if "straight_path" not in scene_summary_raw.columns:
    raise KeyError("scene_summary.csv must contain `straight_path`.")

scene_meta = (
    scene_summary_raw
    .rename(columns={scene_name_column: "scene_name"})
    .copy()
)

scene_meta["scene_name"] = (
    scene_meta["scene_name"]
    .map(normalize_scene_name)
)

scene_meta["simulation_time"] = safe_numeric(
    scene_meta["simulation_time"]
)

scene_meta["straight_path"] = safe_numeric(
    scene_meta["straight_path"]
).astype("Int64")

scene_meta = (
    scene_meta[
        [
            "scene_name",
            "simulation_time",
            "straight_path",
        ]
    ]
    .drop_duplicates("scene_name")
)

scene_meta["path_group"] = (
    scene_meta["straight_path"]
    .map({
        0: "no_straight_path",
        1: "straight_path",
    })
    .astype("string")
)

scene_meta["true_simulation_time_ms"] = (
    scene_meta["simulation_time"]
    / FRAMES_PER_SECOND
    * MS_PER_SECOND
)


# ------------------------------------------------------------
# Human scene-level outcomes
# ------------------------------------------------------------
human_summary_raw = pd.read_csv(
    HUMAN_SUMMARY_PATH,
    low_memory=False,
)

human_scene_column = find_first_column(
    human_summary_raw,
    ["scene_name", "scenename", "scene", "name"],
    required=True,
    label="scene-name column in human summary",
)

for required_column in [
    "mean_judgment_time_sec",
    "mean_accuracy",
]:
    if required_column not in human_summary_raw.columns:
        raise KeyError(
            f"{HUMAN_SUMMARY_PATH} is missing `{required_column}`."
        )

human_summary = (
    human_summary_raw
    .rename(columns={human_scene_column: "scene_name"})
    .copy()
)

human_summary["scene_name"] = (
    human_summary["scene_name"]
    .map(normalize_scene_name)
)

human_summary["mean_judgment_time_sec"] = safe_numeric(
    human_summary["mean_judgment_time_sec"]
)

human_summary["mean_accuracy"] = safe_numeric(
    human_summary["mean_accuracy"]
)

human_summary["human_response_time_ms"] = (
    human_summary["mean_judgment_time_sec"]
    * MS_PER_SECOND
)

human_summary = (
    human_summary[
        [
            "scene_name",
            "mean_judgment_time_sec",
            "human_response_time_ms",
            "mean_accuracy",
        ]
    ]
    .drop_duplicates("scene_name")
)

common_scene_data = (
    scene_meta
    .merge(
        human_summary,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )
)


# ------------------------------------------------------------
# Meta-control model scene-level summary
# ------------------------------------------------------------
meta_control_raw = pd.read_csv(
    META_CONTROL_SUMMARY_PATH,
    low_memory=False,
)

if "parameter_label" in meta_control_raw.columns:
    meta_control_raw = meta_control_raw.loc[
        meta_control_raw["parameter_label"]
        .astype(str)
        .eq(META_CONTROL_PARAMETER_LABEL)
    ].copy()

required_meta_control_columns = [
    "scene_name",
    "hybrid_total_step_count",
    "hybrid_correct",
]

missing = [
    column
    for column in required_meta_control_columns
    if column not in meta_control_raw.columns
]

if missing:
    raise KeyError(
        f"Meta-control model summary is missing {missing}."
    )

meta_control_scene_summary = meta_control_raw.copy()

meta_control_scene_summary["scene_name"] = (
    meta_control_scene_summary["scene_name"]
    .map(normalize_scene_name)
)

meta_control_scene_summary["model_total_step_count"] = safe_numeric(
    meta_control_scene_summary["hybrid_total_step_count"]
)

meta_control_scene_summary["model_run_time_ms"] = (
    meta_control_scene_summary["model_total_step_count"]
    / FRAMES_PER_SECOND
    * MS_PER_SECOND
)

meta_control_scene_summary["model_correct"] = safe_numeric(
    meta_control_scene_summary["hybrid_correct"]
)

meta_control_scene_summary["model_accuracy"] = (
    meta_control_scene_summary["model_correct"]
)

meta_control_scene_summary = (
    meta_control_scene_summary[
        [
            "scene_name",
            "model_total_step_count",
            "model_run_time_ms",
            "model_correct",
            "model_accuracy",
        ]
    ]
    .drop_duplicates("scene_name")
    .merge(
        common_scene_data,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )
)

print("Behavioral scenes:", len(meta_control_scene_summary))


## 1. `high_yescol_nosp_3`: Meta-control model trajectory


In [ ]:
# ============================================================
# 2. high_yescol_nosp_3 scene illustration:
#    Meta-control model trajectory
# ============================================================
# ============================================================
# Figure text-size settings
# ============================================================

FIGURE_TITLE_SIZE = 19
LEGEND_TEXT_SIZE = 13
COLORBAR_LABEL_SIZE = 17

def find_scene_dir(scene_name):
    exact = [
        path
        for path in TEST_DATA_ROOT.rglob(scene_name)
        if path.is_dir()
        and path.name == scene_name
    ]

    if len(exact) == 1:
        return exact[0]

    if len(exact) > 1:
        exp1 = [
            path
            for path in exact
            if path.parent.name == "exp1"
        ]

        if len(exp1) == 1:
            return exp1[0]

        raise RuntimeError(
            f"Multiple directories matched {scene_name}: {exact}"
        )

    raise FileNotFoundError(
        f"Could not find scene directory for {scene_name!r} "
        f"under {TEST_DATA_ROOT}"
    )


def get_frame_files(scene_dir):
    frame_dir = Path(scene_dir) / "frames"

    files = sorted(
        frame_dir.glob("frame_*.png")
    )

    if not files:
        files = sorted(
            frame_dir.glob("*.png")
        )

    return files


def load_background_image(
    scene_dir,
    frame_index=N_HISTORY - 1,
):
    scene_dir = Path(scene_dir)

    expected = (
        scene_dir
        / "frames"
        / f"frame_{frame_index:04d}.png"
    )

    if expected.exists():
        return (
            Image.open(expected).convert("RGB"),
            expected,
        )

    files = get_frame_files(scene_dir)

    if not files:
        raise FileNotFoundError(
            f"No frame PNGs found in {scene_dir / 'frames'}"
        )

    chosen = files[
        min(frame_index, len(files) - 1)
    ]

    return (
        Image.open(chosen).convert("RGB"),
        chosen,
    )


def load_true_positions(scene_dir):
    path = (
        Path(scene_dir)
        / "simulation_dataset.csv"
    )

    if not path.exists():
        raise FileNotFoundError(path)

    data = pd.read_csv(path)

    if not {"ball_x", "ball_y"}.issubset(
        data.columns
    ):
        raise KeyError(
            f"{path} must contain ball_x and ball_y"
        )

    if "frame_index" in data.columns:
        data = data.sort_values("frame_index")

    elif "frame" in data.columns:
        data = data.sort_values("frame")

    return (
        data
        .dropna(
            subset=[
                "ball_x",
                "ball_y",
            ]
        )
        .reset_index(drop=True)
    )


def make_publication_scene_background(
    image,
    background_gray=230,
):
    arr = np.asarray(
        image.convert("RGB")
    ).copy()

    rgb = arr.astype(float)

    luminance = (
        0.2126 * rgb[..., 0]
        + 0.7152 * rgb[..., 1]
        + 0.0722 * rgb[..., 2]
    )

    chroma = (
        rgb.max(axis=-1)
        - rgb.min(axis=-1)
    )

    neutral = chroma < 22

    dark_background = (
        neutral
        & (luminance < 85)
    )

    bright_slides_or_borders = (
        neutral
        & (luminance > 150)
    )

    arr[dark_background] = np.array(
        [background_gray] * 3,
        dtype=np.uint8,
    )

    arr[bright_slides_or_borders] = np.array(
        [0, 0, 0],
        dtype=np.uint8,
    )

    # Remove scene/frame text in the upper-left corner.
    arr[8:60, 8:245] = np.array(
        [background_gray] * 3,
        dtype=np.uint8,
    )

    return Image.fromarray(arr)


def find_meta_control_prediction_csv(
    scene_name,
):
    candidates = sorted(
        META_CONTROL_PREDICTION_DIR.glob(
            f"*{scene_name}*hybrid_predictions.csv"
        )
    )

    if not candidates:
        candidates = sorted(
            META_CONTROL_PREDICTION_DIR.glob(
                f"*{scene_name}*.csv"
            )
        )

    if len(candidates) != 1:
        raise RuntimeError(
            f"Expected exactly one Meta-control model "
            f"prediction CSV for {scene_name}; "
            f"found {candidates}"
        )

    return candidates[0]


def standardize_meta_control_plot_df(
    prediction_df,
):
    data = prediction_df.copy()

    if "frame" not in data.columns:
        for column in [
            "target_frame",
            "time_step",
            "t",
        ]:
            if column in data.columns:
                data["frame"] = data[column]
                break

    if "x" not in data.columns:
        for column in [
            "hybrid_x",
            "pred_x",
        ]:
            if column in data.columns:
                data["x"] = data[column]
                break

    if "y" not in data.columns:
        for column in [
            "hybrid_y",
            "pred_y",
        ]:
            if column in data.columns:
                data["y"] = data[column]
                break

    if "source" not in data.columns:
        for column in [
            "segment_type",
            "mode",
            "prediction_type",
        ]:
            if column in data.columns:
                data["source"] = data[column]
                break

    required = [
        "frame",
        "x",
        "y",
        "source",
    ]

    missing = [
        column
        for column in required
        if column not in data.columns
    ]

    if missing:
        raise KeyError(
            f"Meta-control model prediction CSV "
            f"is missing: {missing}"
        )

    data["source"] = (
        data["source"]
        .astype(str)
        .str.lower()
    )

    data.loc[
        data["source"].str.contains(
            "abstract"
        ),
        "source",
    ] = "abstraction"

    data.loc[
        data["source"].str.contains(
            "sim"
        ),
        "source",
    ] = "simulation"

    for column in [
        "frame",
        "x",
        "y",
    ]:
        data[column] = safe_numeric(
            data[column]
        )

    data = (
        data
        .dropna(
            subset=[
                "frame",
                "x",
                "y",
                "source",
            ]
        )
        .copy()
    )

    data["frame"] = (
        data["frame"]
        .astype(int)
    )

    tie_column = (
        "segment_id"
        if "segment_id" in data.columns
        else "source"
    )

    return (
        data
        .sort_values(
            [
                "frame",
                tie_column,
            ]
        )
        .drop_duplicates(
            subset=["frame"],
            keep="last",
        )
        .sort_values("frame")
        .reset_index(drop=True)
    )


scene_dir = find_scene_dir(
    TARGET_SCENE
)

background_raw, background_path = (
    load_background_image(
        scene_dir
    )
)

background = (
    make_publication_scene_background(
        background_raw
    )
)

true_df = load_true_positions(
    scene_dir
)

true_xy = true_df[
    ["ball_x", "ball_y"]
].to_numpy(dtype=float)

prediction_path = (
    find_meta_control_prediction_csv(
        TARGET_SCENE
    )
)

meta_control_plot_df = (
    standardize_meta_control_plot_df(
        pd.read_csv(
            prediction_path,
            low_memory=False,
        )
    )
)


fig, ax = plt.subplots(
    figsize=(6.2, 7.8)
)

ax.set_facecolor(
    SCENE_BG_COLOR
)

ax.imshow(
    background,
    zorder=0,
)

# Pure-simulation trajectory underneath model trajectory.
ax.plot(
    true_xy[:, 0],
    true_xy[:, 1],
    color=TRUE_COLOR,
    linewidth=2.2,
    alpha=0.82,
    zorder=1,
)

color_map = {
    "abstraction": ABSTRACTION_COLOR,
    "simulation": SIMULATION_COLOR,
}

# Connect only within the same reasoning mode.
for index in range(
    len(meta_control_plot_df) - 1
):
    point_a = meta_control_plot_df.iloc[index]
    point_b = meta_control_plot_df.iloc[index + 1]

    source_a = str(
        point_a["source"]
    )

    source_b = str(
        point_b["source"]
    )

    if (
        source_a
        == source_b
        == "abstraction"
    ):
        ax.plot(
            [
                point_a["x"],
                point_b["x"],
            ],
            [
                point_a["y"],
                point_b["y"],
            ],
            color=ABSTRACTION_COLOR,
            linewidth=2.7,
            alpha=0.93,
            zorder=4,
        )

    elif (
        source_a
        == source_b
        == "simulation"
        and int(point_b["frame"])
        == int(point_a["frame"]) + 1
    ):
        ax.plot(
            [
                point_a["x"],
                point_b["x"],
            ],
            [
                point_a["y"],
                point_b["y"],
            ],
            color=SIMULATION_COLOR,
            linewidth=2.25,
            alpha=0.90,
            zorder=4,
        )


for source, group in (
    meta_control_plot_df
    .groupby(
        "source",
        sort=False,
    )
):
    ax.scatter(
        group["x"],
        group["y"],
        s=27,
        color=color_map.get(
            source,
            "black",
        ),
        alpha=0.96,
        edgecolors="black",
        linewidths=0.35,
        zorder=5,
    )


# ------------------------------------------------------------
# Legend
# ------------------------------------------------------------

scene_handles = [
    Line2D(
        [0],
        [0],
        color=TRUE_COLOR,
        lw=2.2,
        label="Pure simulation trajectory",
    ),
    Line2D(
        [0],
        [0],
        color=ABSTRACTION_COLOR,
        lw=2.7,
        marker="o",
        markersize=7,
        markeredgecolor="black",
        markeredgewidth=0.35,
        label="Abstraction",
    ),
    Line2D(
        [0],
        [0],
        color=SIMULATION_COLOR,
        lw=2.25,
        marker="o",
        markersize=7,
        markeredgecolor="black",
        markeredgewidth=0.35,
        label="Simulation",
    ),
]

ax.legend(
    handles=scene_handles,
    loc="upper right",
    frameon=True,
    framealpha=0.90,
    fontsize=LEGEND_TEXT_SIZE,
)

# ------------------------------------------------------------
# Scene limits and title
# ------------------------------------------------------------

ax.set_xlim(
    0,
    ORIGINAL_FRAME_WIDTH,
)

ax.set_ylim(
    ORIGINAL_FRAME_HEIGHT,
    0,
)

ax.set_title(
    "Meta-control model trajectory",
    fontsize=FIGURE_TITLE_SIZE,
    pad=14,
)

ax.axis("off")

fig.tight_layout()

plt.show()

## 2. Run time / response time vs true simulation time

The fitted lines and confidence intervals use the original interaction OLS. The statistics table reports the Pearson \(r\), fitted slope, and the slope-test \(p\)-value for each plotted path group.


In [ ]:
# ============================================================
# 3. Meta-control model run time vs true simulation time
#    Human response time vs true simulation time
#
# ORIGINAL interaction OLS:
# outcome ~ true_simulation_time * path_group
#
# Figures use predictions directly from this interaction model.
#
# One-sided interaction test:
# H1: straight-path slope < non-straight-path slope
# i.e., interaction beta (straight - non-straight) < 0
# ============================================================

from scipy.stats import pearsonr, t as t_dist
from matplotlib.ticker import FuncFormatter


# ============================================================
# Shared figure formatting
# ============================================================

TIME_FIGSIZE = (8.5, 6.5)

# Fixed axes rectangle:
# [left, bottom, width, height]
#
# Because both figures use exactly this rectangle,
# their plotting boxes are guaranteed to have the
# same position and dimensions.
TIME_AXES_POSITION = [
    0.16,   # left
    0.15,   # bottom
    0.80,   # width
    0.74,   # height
]

FIGURE_TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_TEXT_SIZE = 12

TITLE_PAD = 12

# Explicit label coordinates relative to the axes.
# These prevent tick-label width from shifting the axis labels.
X_LABEL_Y = -0.105
Y_LABEL_X = -0.115


# ============================================================
# Tick formatting
# ============================================================

def integer_tick_formatter(value, position):
    """
    Display millisecond tick values without unnecessary
    decimal points, e.g. 1000 rather than 1000.0.
    """
    return f"{value:.0f}"


TIME_TICK_FORMATTER = FuncFormatter(
    integer_tick_formatter
)


# ============================================================
# Fit and plot
# ============================================================

def fit_and_plot_group_aware_time_model(
    data,
    outcome_column,
    outcome_label,
    title,
):

    # --------------------------------------------------------
    # Experiment 1 scenes
    # --------------------------------------------------------

    exp1 = data.loc[
        ~data["scene_name"]
        .astype(str)
        .str.startswith(
            "scene",
            na=False,
        )
    ].copy()

    exp1 = (
        exp1
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=[
                "true_simulation_time_ms",
                outcome_column,
                "path_group",
            ]
        )
    )

    exp1["path_group"] = pd.Categorical(
        exp1["path_group"],
        categories=PATH_GROUP_ORDER,
        ordered=True,
    )


    # --------------------------------------------------------
    # Center predictor exactly as in original model
    # --------------------------------------------------------

    true_time_mean = float(
        exp1[
            "true_simulation_time_ms"
        ].mean()
    )

    exp1[
        "true_simulation_time_ms_centered"
    ] = (
        exp1[
            "true_simulation_time_ms"
        ]
        - true_time_mean
    )


    # --------------------------------------------------------
    # Original interaction OLS
    # --------------------------------------------------------

    formula = (
        f"{outcome_column} "
        "~ true_simulation_time_ms_centered "
        "* C(path_group)"
    )

    result = smf.ols(
        formula,
        data=exp1,
    ).fit()


    time_term = (
        "true_simulation_time_ms_centered"
    )

    interaction_term = (
        "true_simulation_time_ms_centered:"
        "C(path_group)[T.straight_path]"
    )


    # --------------------------------------------------------
    # Group-specific slopes from interaction OLS
    # --------------------------------------------------------

    non_straight_slope = float(
        result.params[
            time_term
        ]
    )

    interaction_slope = float(
        result.params[
            interaction_term
        ]
    )

    straight_slope = (
        non_straight_slope
        + interaction_slope
    )


    # --------------------------------------------------------
    # Group-specific slope tests
    # --------------------------------------------------------

    non_straight_test = result.t_test(
        f"{time_term} = 0"
    )

    straight_test = result.t_test(
        f"{time_term} + {interaction_term} = 0"
    )

    non_straight_p = float(
        np.asarray(
            non_straight_test.pvalue
        ).squeeze()
    )

    straight_p = float(
        np.asarray(
            straight_test.pvalue
        ).squeeze()
    )


    # --------------------------------------------------------
    # One-sided interaction test
    #
    # H1:
    # straight slope < non-straight slope
    #
    # Since interaction =
    # straight slope - non-straight slope,
    # H1 corresponds to interaction beta < 0.
    # --------------------------------------------------------

    interaction_t = float(
        result.tvalues[
            interaction_term
        ]
    )

    interaction_p_one_sided = float(
        t_dist.cdf(
            interaction_t,
            df=result.df_resid,
        )
    )


    slope_values = {
        "no_straight_path": non_straight_slope,
        "straight_path": straight_slope,
    }

    slope_p_values = {
        "no_straight_path": non_straight_p,
        "straight_path": straight_p,
    }


    # ========================================================
    # Figure — original interaction-OLS predictions
    # ========================================================

    fig = plt.figure(
        figsize=TIME_FIGSIZE
    )

    ax = fig.add_axes(
        TIME_AXES_POSITION
    )


    statistics_rows = []


    # --------------------------------------------------------
    # Plot each path group
    # --------------------------------------------------------

    for group_name in PATH_GROUP_ORDER:

        group_data = exp1.loc[
            exp1["path_group"].eq(
                group_name
            )
        ].copy()

        color = PATH_COLORS[
            group_name
        ]


        # ----------------------------------------------------
        # Scatter
        # ----------------------------------------------------

        ax.scatter(
            group_data[
                "true_simulation_time_ms"
            ],
            group_data[
                outcome_column
            ],
            s=58,
            alpha=0.82,
            color=color,
            edgecolors="black",
            linewidths=0.35,
            zorder=3,
        )


        # ----------------------------------------------------
        # Prediction grid from ORIGINAL interaction model
        # ----------------------------------------------------

        x_grid = np.linspace(
            group_data[
                "true_simulation_time_ms"
            ].min(),
            group_data[
                "true_simulation_time_ms"
            ].max(),
            300,
        )

        prediction_grid = pd.DataFrame({
            "true_simulation_time_ms_centered": (
                x_grid
                - true_time_mean
            ),
            "path_group": pd.Categorical(
                [group_name] * len(x_grid),
                categories=PATH_GROUP_ORDER,
                ordered=True,
            ),
        })


        prediction = (
            result
            .get_prediction(
                prediction_grid
            )
            .summary_frame(
                alpha=0.05
            )
        )


        # ----------------------------------------------------
        # 95% CI
        # ----------------------------------------------------

        ax.fill_between(
            x_grid,
            prediction[
                "mean_ci_lower"
            ].to_numpy(dtype=float),
            prediction[
                "mean_ci_upper"
            ].to_numpy(dtype=float),
            color=color,
            alpha=CI_ALPHA,
            linewidth=0,
            zorder=1,
        )


        # ----------------------------------------------------
        # Interaction-OLS fitted line
        # ----------------------------------------------------

        ax.plot(
            x_grid,
            prediction[
                "mean"
            ].to_numpy(dtype=float),
            color=color,
            linewidth=2.8,
            zorder=4,
        )


        # ----------------------------------------------------
        # Pearson r for this path group
        # ----------------------------------------------------

        correlation = pearsonr(
            group_data[
                "true_simulation_time_ms"
            ],
            group_data[
                outcome_column
            ],
        )


        statistics_rows.append({
            "analysis": title,
            "line": PATH_LABELS[
                group_name
            ],
            "n_scenes": len(
                group_data
            ),
            "r": float(
                correlation.statistic
            ),
            "beta": slope_values[
                group_name
            ],
            "p": slope_p_values[
                group_name
            ],
        })


    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    legend_handles = [
        Line2D(
            [0], [0],
            color=NON_STRAIGHT_COLOR,
            lw=2.8,
            marker="o",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=0.4,
            label="Non-straight path",
        ),
        Line2D(
            [0], [0],
            color=STRAIGHT_COLOR,
            lw=2.8,
            marker="o",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=0.4,
            label="Straight path",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        loc="upper left",
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="black",
        fontsize=LEGEND_TEXT_SIZE,
    )


    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    ax.set_title(
        title,
        fontsize=FIGURE_TITLE_SIZE,
        pad=TITLE_PAD,
    )


    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "True simulation time (ms)",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax.set_ylabel(
        outcome_label,
        fontsize=AXIS_LABEL_SIZE,
    )


    # Explicit coordinates ensure identical label placement
    # regardless of tick-label width.

    ax.xaxis.set_label_coords(
        0.5,
        X_LABEL_Y,
    )

    ax.yaxis.set_label_coords(
        Y_LABEL_X,
        0.5,
    )


    # --------------------------------------------------------
    # Tick formatting
    # --------------------------------------------------------

    ax.tick_params(
        axis="both",
        labelsize=TICK_SIZE,
    )

    # Remove unnecessary ".0" from millisecond ticks.
    ax.xaxis.set_major_formatter(
        TIME_TICK_FORMATTER
    )

    ax.yaxis.set_major_formatter(
        TIME_TICK_FORMATTER
    )


    # --------------------------------------------------------
    # Full boxed frame
    # --------------------------------------------------------

    for spine in ax.spines.values():

        spine.set_visible(
            True
        )

        spine.set_linewidth(
            1.1
        )

        spine.set_color(
            "black"
        )


    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------

    ax.grid(
        alpha=0.22,
        zorder=0,
    )


    # IMPORTANT:
    # Do NOT use fig.tight_layout() here.
    #
    # The axes rectangle has already been fixed explicitly,
    # so tight_layout() would reintroduce differences based
    # on tick-label width.

    plt.show()


    # --------------------------------------------------------
    # Add interaction statistics to summary
    # --------------------------------------------------------

    interaction_summary = {
        "analysis": title,
        "interaction_beta_straight_minus_non_straight": (
            interaction_slope
        ),
        "interaction_t": interaction_t,
        "interaction_p": (
            interaction_p_one_sided
        ),
    }


    return (
        pd.DataFrame(
            statistics_rows
        ),
        interaction_summary,
        result,
        exp1,
    )


# ============================================================
# Meta-control model run time
# ============================================================

(
    model_time_stats,
    model_interaction_stats,
    model_time_fit,
    model_time_data,
) = fit_and_plot_group_aware_time_model(
    data=meta_control_scene_summary,
    outcome_column="model_run_time_ms",
    outcome_label="Model run time (ms)",
    title="Model run time vs true simulation time",
)


# ============================================================
# Human response time
# ============================================================

(
    human_time_stats,
    human_interaction_stats,
    human_time_fit,
    human_time_data,
) = fit_and_plot_group_aware_time_model(
    data=meta_control_scene_summary,
    outcome_column="human_response_time_ms",
    outcome_label="Human response time (ms)",
    title="Human response time vs true simulation time",
)


# ============================================================
# Group-specific line statistics
# ============================================================

time_line_statistics = pd.concat(
    [
        model_time_stats,
        human_time_stats,
    ],
    ignore_index=True,
)

print(
    "Group-specific statistics "
    "(r = Pearson correlation; "
    "beta and p = slopes/tests from interaction OLS)"
)

display(
    time_line_statistics.style.format({
        "r": "{:.5f}",
        "beta": "{:.6f}",
        "p": "{:.6g}",
    })
)


# ============================================================
# Straight vs non-straight interaction statistics
# ============================================================

interaction_statistics = pd.DataFrame([
    model_interaction_stats,
    human_interaction_stats,
])

print(
    "Interaction test: "
    "H1 = straight-path slope < non-straight-path slope"
)

display(
    interaction_statistics.style.format({
        "interaction_beta_straight_minus_non_straight": "{:.6f}",
        "interaction_t": "{:.4f}",
        "interaction_p": "{:.6g}",
    })
)

## 3. Accuracy vs true simulation time


In [ ]:
# ============================================================
# 4. Meta-control model accuracy vs true simulation time
#    Human average accuracy vs true simulation time
# ============================================================

from matplotlib.ticker import FuncFormatter


# ============================================================
# Shared figure formatting
# ============================================================

ACCURACY_FIGSIZE = (8.5, 6.5)

# Fixed axes rectangle:
# [left, bottom, width, height]
ACCURACY_AXES_POSITION = [
    0.16,   # left
    0.15,   # bottom
    0.80,   # width
    0.74,   # height
]

FIGURE_TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14

TITLE_PAD = 12

# Explicit axis-label coordinates
X_LABEL_Y = -0.105
Y_LABEL_X = -0.115


# ============================================================
# Tick formatting
# ============================================================

def integer_tick_formatter(value, position):
    """
    Display millisecond x-axis ticks without unnecessary
    decimal points, e.g. 1000 instead of 1000.0.
    """
    return f"{value:.0f}"


TIME_TICK_FORMATTER = FuncFormatter(
    integer_tick_formatter
)


# ============================================================
# Experiment 2 scenes
# ============================================================

exp2 = meta_control_scene_summary.loc[
    meta_control_scene_summary[
        "scene_name"
    ]
    .astype(str)
    .str.startswith(
        "scene",
        na=False,
    )
].copy()

exp2 = exp2.replace(
    [np.inf, -np.inf],
    np.nan,
)


# ============================================================
# Model accuracy
# ============================================================

model_accuracy_data = (
    exp2
    .dropna(
        subset=[
            "true_simulation_time_ms",
            "model_accuracy",
        ]
    )
    .sort_values(
        "true_simulation_time_ms"
    )
)


if (
    (
        model_accuracy_data[
            "model_accuracy"
        ] < 0
    )
    |
    (
        model_accuracy_data[
            "model_accuracy"
        ] > 1
    )
).any():

    raise ValueError(
        "model_accuracy must lie between 0 and 1."
    )


# ------------------------------------------------------------
# Preserve source convention:
# exclude largest-time scene from polynomial fit,
# while still plotting that scene
# ------------------------------------------------------------

excluded_index = (
    model_accuracy_data[
        "true_simulation_time_ms"
    ].idxmax()
)

model_curve_data = (
    model_accuracy_data
    .drop(
        index=excluded_index
    )
    .copy()
)


model_x_line, model_y_line = polynomial_curve(
    model_curve_data[
        "true_simulation_time_ms"
    ],
    model_curve_data[
        "model_accuracy"
    ],
    order=POLYNOMIAL_ORDER,
)


# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig_model_accuracy = plt.figure(
    figsize=ACCURACY_FIGSIZE
)

ax_model_accuracy = fig_model_accuracy.add_axes(
    ACCURACY_AXES_POSITION
)


# ------------------------------------------------------------
# Scatter
# ------------------------------------------------------------

ax_model_accuracy.scatter(
    model_accuracy_data[
        "true_simulation_time_ms"
    ],
    model_accuracy_data[
        "model_accuracy"
    ],
    s=58,
    alpha=0.72,
    color=ACCURACY_COLOR,
    edgecolors="black",
    linewidths=0.35,
    zorder=2,
)


# ------------------------------------------------------------
# Polynomial curve
# ------------------------------------------------------------

ax_model_accuracy.plot(
    model_x_line,
    model_y_line,
    color=ACCURACY_COLOR,
    linewidth=3.0,
    zorder=3,
)


# ------------------------------------------------------------
# Title and labels
# ------------------------------------------------------------

ax_model_accuracy.set_title(
    "Model accuracy vs true simulation time",
    fontsize=FIGURE_TITLE_SIZE,
    pad=TITLE_PAD,
)

ax_model_accuracy.set_xlabel(
    "True simulation time (ms)",
    fontsize=AXIS_LABEL_SIZE,
)

ax_model_accuracy.set_ylabel(
    "Model accuracy",
    fontsize=AXIS_LABEL_SIZE,
)


# Explicit coordinates ensure identical placement
ax_model_accuracy.xaxis.set_label_coords(
    0.5,
    X_LABEL_Y,
)

ax_model_accuracy.yaxis.set_label_coords(
    Y_LABEL_X,
    0.5,
)


# ------------------------------------------------------------
# Axis limits and ticks
# ------------------------------------------------------------

ax_model_accuracy.set_yticks(
    [0, 1]
)

ax_model_accuracy.set_ylim(
    -0.08,
    1.08,
)

ax_model_accuracy.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
)

ax_model_accuracy.xaxis.set_major_formatter(
    TIME_TICK_FORMATTER
)


# ------------------------------------------------------------
# Full boxed frame
# ------------------------------------------------------------

for spine in ax_model_accuracy.spines.values():

    spine.set_visible(
        True
    )

    spine.set_linewidth(
        1.1
    )

    spine.set_color(
        "black"
    )


# ------------------------------------------------------------
# Grid
# ------------------------------------------------------------

ax_model_accuracy.grid(
    alpha=0.22,
    zorder=0,
)

# Do NOT use tight_layout()
plt.show()


# ============================================================
# Human average accuracy
# ============================================================

human_accuracy_data = (
    exp2
    .dropna(
        subset=[
            "true_simulation_time_ms",
            "mean_accuracy",
        ]
    )
    .sort_values(
        "true_simulation_time_ms"
    )
)


human_x_line, human_y_line = polynomial_curve(
    human_accuracy_data[
        "true_simulation_time_ms"
    ],
    human_accuracy_data[
        "mean_accuracy"
    ],
    order=POLYNOMIAL_ORDER,
)


# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig_human_accuracy = plt.figure(
    figsize=ACCURACY_FIGSIZE
)

ax_human_accuracy = fig_human_accuracy.add_axes(
    ACCURACY_AXES_POSITION
)


# ------------------------------------------------------------
# Scatter
# ------------------------------------------------------------

ax_human_accuracy.scatter(
    human_accuracy_data[
        "true_simulation_time_ms"
    ],
    human_accuracy_data[
        "mean_accuracy"
    ],
    s=58,
    alpha=0.72,
    color=ACCURACY_COLOR,
    edgecolors="black",
    linewidths=0.35,
    zorder=2,
)


# ------------------------------------------------------------
# Polynomial curve
# ------------------------------------------------------------

ax_human_accuracy.plot(
    human_x_line,
    human_y_line,
    color=ACCURACY_COLOR,
    linewidth=3.0,
    zorder=3,
)


# ------------------------------------------------------------
# Title and labels
# ------------------------------------------------------------

ax_human_accuracy.set_title(
    "Human average accuracy vs true simulation time",
    fontsize=FIGURE_TITLE_SIZE,
    pad=TITLE_PAD,
)

ax_human_accuracy.set_xlabel(
    "True simulation time (ms)",
    fontsize=AXIS_LABEL_SIZE,
)

ax_human_accuracy.set_ylabel(
    "Human average accuracy",
    fontsize=AXIS_LABEL_SIZE,
)


# Explicit coordinates ensure identical placement
ax_human_accuracy.xaxis.set_label_coords(
    0.5,
    X_LABEL_Y,
)

ax_human_accuracy.yaxis.set_label_coords(
    Y_LABEL_X,
    0.5,
)


# ------------------------------------------------------------
# Axis limits and ticks
# ------------------------------------------------------------

ax_human_accuracy.set_ylim(
    -0.05,
    1.05,
)

ax_human_accuracy.tick_params(
    axis="both",
    labelsize=TICK_SIZE,
)

ax_human_accuracy.xaxis.set_major_formatter(
    TIME_TICK_FORMATTER
)


# ------------------------------------------------------------
# Full boxed frame
# ------------------------------------------------------------

for spine in ax_human_accuracy.spines.values():

    spine.set_visible(
        True
    )

    spine.set_linewidth(
        1.1
    )

    spine.set_color(
        "black"
    )


# ------------------------------------------------------------
# Grid
# ------------------------------------------------------------

ax_human_accuracy.grid(
    alpha=0.22,
    zorder=0,
)

# Do NOT use tight_layout()
plt.show()

## 4. Eye movements: abstraction vs simulation

Meta-control model only. The threshold sweep preserves the source notebook's gaze-to-trace matching, overlap handling, participant-level differences, bootstrap median confidence intervals, and Wilcoxon tests.


In [ ]:
# ============================================================
# 5. Build Meta-control model trace segments and prepare gaze
# ============================================================

def build_meta_control_segments(
    prediction_dir,
):
    prediction_files = sorted(
        Path(prediction_dir).glob(
            "*__hybrid_predictions.csv"
        )
    )

    if not prediction_files:
        raise FileNotFoundError(
            "No Meta-control model prediction files "
            f"found under {prediction_dir}"
        )

    parts = []

    for prediction_path in prediction_files:

        prediction = pd.read_csv(
            prediction_path,
            low_memory=False,
        )

        prediction[
            "prediction_file"
        ] = str(
            prediction_path
        )

        if "scene" in prediction.columns:
            prediction["scene_name"] = (
                prediction["scene"]
                .map(
                    normalize_scene_name
                )
            )

        else:
            prediction["scene_name"] = (
                normalize_scene_name(
                    prediction_path.name
                    .replace(
                        "__hybrid_predictions.csv",
                        "",
                    )
                    .split("__")[-1]
                )
            )

        parts.append(
            prediction
        )

    raw = pd.concat(
        parts,
        ignore_index=True,
    )

    required = {
        "scene_name",
        "source",
        "frame",
        "segment_id",
        "x",
        "y",
    }

    missing = (
        required
        - set(raw.columns)
    )

    if missing:
        raise KeyError(
            "Meta-control model prediction files "
            f"are missing: {sorted(missing)}"
        )

    raw["scene_name"] = (
        raw["scene_name"]
        .map(normalize_scene_name)
    )

    raw["reasoning_trace"] = (
        normalize_trace_series(
            raw["source"]
        )
    )

    raw["frame"] = safe_numeric(
        raw["frame"]
    )

    raw["source_segment_id"] = safe_numeric(
        raw["segment_id"]
    )

    raw["source_segment_key"] = (
        raw[
            "source_segment_id"
        ]
        .astype("string")
        .fillna("__NA__")
    )

    raw["x_model"] = (
        safe_numeric(
            raw["x"]
        )
        + META_CONTROL_X_SHIFT
    )

    raw["y_model"] = (
        safe_numeric(
            raw["y"]
        )
        + META_CONTROL_Y_SHIFT
    )

    raw["original_row_order"] = np.arange(
        len(raw),
        dtype=int,
    )

    raw = raw.loc[
        raw["reasoning_trace"]
        .isin(
            [
                "simulation",
                "abstraction",
            ]
        )
        & np.isfinite(
            raw["frame"]
        )
        & np.isfinite(
            raw["x_model"]
        )
        & np.isfinite(
            raw["y_model"]
        )
    ].copy()


    # --------------------------------------------------------
    # Abstraction segments
    # --------------------------------------------------------
    abstraction_rows = raw.loc[
        raw["reasoning_trace"]
        .eq("abstraction")
    ].copy()

    if (
        "abstraction_line_role"
        in abstraction_rows.columns
    ):
        abstraction_rows[
            "line_role"
        ] = (
            abstraction_rows[
                "abstraction_line_role"
            ]
            .astype("string")
            .fillna("")
            .str.strip()
            .str.lower()
        )

    else:
        abstraction_rows[
            "line_role"
        ] = ""

    abstraction_segment_rows = []

    for (
        scene_name,
        segment_key,
    ), group in (
        abstraction_rows
        .groupby(
            [
                "scene_name",
                "source_segment_key",
            ],
            sort=False,
        )
    ):
        group = group.sort_values(
            [
                "frame",
                "original_row_order",
            ],
            kind="mergesort",
        )

        start_candidates = group.loc[
            group[
                "line_role"
            ].eq("start")
        ]

        end_candidates = group.loc[
            group[
                "line_role"
            ].eq("endpoint")
        ]

        start_row = (
            start_candidates.iloc[0]
            if len(start_candidates)
            else group.iloc[0]
        )

        end_row = (
            end_candidates.iloc[-1]
            if len(end_candidates)
            else group.iloc[-1]
        )

        original_start_frame = float(
            start_row["frame"]
        )

        end_frame = float(
            end_row["frame"]
        )

        if (
            not np.isfinite(
                original_start_frame
            )
            or not np.isfinite(
                end_frame
            )
            or end_frame
            <= original_start_frame
            or end_frame
            < TRACE_START_FRAME
        ):
            continue

        original_start_x = float(
            start_row["x_model"]
        )

        original_start_y = float(
            start_row["y_model"]
        )

        end_x = float(
            end_row["x_model"]
        )

        end_y = float(
            end_row["y_model"]
        )

        retained_start_frame = float(
            max(
                original_start_frame,
                TRACE_START_FRAME,
            )
        )

        fraction = (
            (
                retained_start_frame
                - original_start_frame
            )
            /
            (
                end_frame
                - original_start_frame
            )
        )

        fraction = float(
            np.clip(
                fraction,
                0.0,
                1.0,
            )
        )

        retained_start_x = (
            original_start_x
            + fraction
            * (
                end_x
                - original_start_x
            )
        )

        retained_start_y = (
            original_start_y
            + fraction
            * (
                end_y
                - original_start_y
            )
        )

        if (
            end_frame
            <= retained_start_frame
        ):
            continue

        if not (
            in_display_bounds(
                [retained_start_x],
                [retained_start_y],
            )[0]
            and in_display_bounds(
                [end_x],
                [end_y],
            )[0]
        ):
            continue

        abstraction_segment_rows.append({
            "scene_name": scene_name,
            "sample_idx": "__single__",
            "reasoning_trace": "abstraction",
            "source_segment_key": str(
                segment_key
            ),
            "start_frame": retained_start_frame,
            "end_frame": end_frame,
            "start_x": retained_start_x,
            "start_y": retained_start_y,
            "end_x": end_x,
            "end_y": end_y,
        })

    abstraction_segments = pd.DataFrame(
        abstraction_segment_rows
    )


    # --------------------------------------------------------
    # Consecutive simulation segments
    # --------------------------------------------------------
    simulation_points = (
        raw.loc[
            raw["reasoning_trace"]
            .eq("simulation")
            & raw["frame"]
            .ge(TRACE_START_FRAME)
        ]
        .sort_values(
            [
                "scene_name",
                "source_segment_key",
                "frame",
                "original_row_order",
            ],
            kind="mergesort",
        )
        .copy()
    )

    group = simulation_points.groupby(
        [
            "scene_name",
            "source_segment_key",
        ],
        sort=False,
    )

    simulation_points[
        "start_frame"
    ] = group["frame"].shift(1)

    simulation_points[
        "start_x"
    ] = group["x_model"].shift(1)

    simulation_points[
        "start_y"
    ] = group["y_model"].shift(1)

    simulation_points[
        "end_frame"
    ] = simulation_points["frame"]

    simulation_points[
        "end_x"
    ] = simulation_points["x_model"]

    simulation_points[
        "end_y"
    ] = simulation_points["y_model"]

    valid = (
        np.isfinite(
            simulation_points[
                "start_frame"
            ]
        )
        & (
            simulation_points[
                "end_frame"
            ]
            - simulation_points[
                "start_frame"
            ]
        ).eq(1)
        & in_display_bounds(
            simulation_points[
                "start_x"
            ],
            simulation_points[
                "start_y"
            ],
        )
        & in_display_bounds(
            simulation_points[
                "end_x"
            ],
            simulation_points[
                "end_y"
            ],
        )
    )

    simulation_segments = (
        simulation_points.loc[
            valid,
            [
                "scene_name",
                "source_segment_key",
                "start_frame",
                "end_frame",
                "start_x",
                "start_y",
                "end_x",
                "end_y",
            ],
        ]
        .copy()
    )

    simulation_segments[
        "sample_idx"
    ] = "__single__"

    simulation_segments[
        "reasoning_trace"
    ] = "simulation"

    segment_columns = [
        "scene_name",
        "sample_idx",
        "reasoning_trace",
        "source_segment_key",
        "start_frame",
        "end_frame",
        "start_x",
        "start_y",
        "end_x",
        "end_y",
    ]

    segments = pd.concat(
        [
            abstraction_segments.reindex(
                columns=segment_columns
            ),
            simulation_segments.reindex(
                columns=segment_columns
            ),
        ],
        ignore_index=True,
    )

    segments = (
        segments
        .sort_values(
            [
                "scene_name",
                "start_frame",
                "end_frame",
                "reasoning_trace",
                "source_segment_key",
            ],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    segments.insert(
        2,
        "segment_id",
        (
            segments
            .groupby(
                "scene_name"
            )
            .cumcount()
            + 1
        ),
    )

    if len(segments) == 0:
        raise RuntimeError(
            "No valid Meta-control model "
            "segments were created."
        )

    return segments


meta_control_segments = (
    build_meta_control_segments(
        META_CONTROL_PREDICTION_DIR
    )
)


# ------------------------------------------------------------
# Prepare human gaze exactly as in the source analysis
# ------------------------------------------------------------
human = pd.read_csv(
    CLASSIFIED_TRIAL_PATH,
    dtype={
        "subject_id": "string",
        "scene_name": "string",
        "segment_class": "string",
    },
    low_memory=False,
)

if (
    "subject_id" not in human.columns
    and "id" in human.columns
):
    human["subject_id"] = (
        human["id"]
        .astype("string")
    )

required_human_columns = [
    "subject_id",
    "scene_name",
    "x",
    "y",
    "pupil",
    "segment_class",
]

missing_human = [
    column
    for column in required_human_columns
    if column not in human.columns
]

if missing_human:
    raise KeyError(
        f"{CLASSIFIED_TRIAL_PATH} "
        f"is missing {missing_human}."
    )

TIME_COLUMN = find_first_column(
    human,
    [
        "raw_time",
        "normalized_time",
        "time",
        "timestamp",
    ],
    required=True,
    label="human gaze time column",
)

human["subject_id"] = (
    human["subject_id"]
    .astype("string")
    .str.strip()
    .str.replace(
        r"^(\d+)\.0$",
        r"\1",
        regex=True,
    )
)

human["scene_name"] = (
    human["scene_name"]
    .map(normalize_scene_name)
)

human["x_raw"] = safe_numeric(
    human["x"]
)

human["y_raw"] = safe_numeric(
    human["y"]
)

human["x_aligned"] = (
    human["x_raw"]
    + HUMAN_X_SHIFT
)

human["y_aligned"] = (
    human["y_raw"]
    + HUMAN_Y_SHIFT
)

human["pupil"] = safe_numeric(
    human["pupil"]
)

human[TIME_COLUMN] = safe_numeric(
    human[TIME_COLUMN]
)

human["log_pupil"] = np.where(
    human["pupil"] > 0,
    np.log(
        human["pupil"]
    ),
    np.nan,
)

human["segment_class_clean"] = (
    human["segment_class"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace(
        "_",
        " ",
        regex=False,
    )
    .str.replace(
        "-",
        " ",
        regex=False,
    )
    .str.replace(
        r"\s+",
        " ",
        regex=True,
    )
)

human["is_saccade"] = (
    human[
        "segment_class_clean"
    ]
    .str.contains(
        "saccade",
        na=False,
    )
)

human["is_smooth_pursuit"] = (
    human[
        "segment_class_clean"
    ]
    .str.contains(
        "pursuit",
        na=False,
    )
    |
    human[
        "segment_class_clean"
    ]
    .str.contains(
        "smooth",
        na=False,
    )
)

human = human.loc[
    human["subject_id"].notna()
    & np.isfinite(
        human["x_aligned"]
    )
    & np.isfinite(
        human["y_aligned"]
    )
    & np.isfinite(
        human[TIME_COLUMN]
    )
    & in_display_bounds(
        human["x_aligned"],
        human["y_aligned"],
    )
].copy()

human = (
    human
    .sort_values(
        [
            "subject_id",
            "scene_name",
            TIME_COLUMN,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)

human["gaze_id"] = (
    human
    .groupby(
        [
            "subject_id",
            "scene_name",
        ],
        sort=False,
    )
    .cumcount()
    + 1
)

print(
    "Meta-control model trace scenes:",
    meta_control_segments[
        "scene_name"
    ].nunique(),
)

print(
    "Prepared human gaze scenes:",
    human[
        "scene_name"
    ].nunique(),
)


In [ ]:
# ============================================================
# Shared threshold-sweep figure formatting
# ============================================================

THRESHOLD_FIGSIZE = (8.5, 6.5)

# Fixed axes rectangle:
# [left, bottom, width, height]
THRESHOLD_AXES_POSITION = [
    0.16,   # left
    0.15,   # bottom
    0.80,   # width
    0.74,   # height
]

FIGURE_TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14

TITLE_PAD = 12

# Explicit label coordinates so different-width
# tick labels cannot move the axis labels.
X_LABEL_Y = -0.105
Y_LABEL_X = -0.115


# ============================================================
# Figures
# ============================================================

for specification in analysis_specs:

    plot_data = (
        threshold_summary.loc[
            threshold_summary[
                "analysis"
            ].eq(
                specification[
                    "analysis"
                ]
            )
        ]
        .sort_values(
            "threshold"
        )
        .reset_index(
            drop=True
        )
    )


    # --------------------------------------------------------
    # Plot values
    # --------------------------------------------------------

    x_values = (
        plot_data[
            "threshold"
        ].to_numpy(
            dtype=float
        )
    )

    median_values = (
        plot_data[
            "median_difference"
        ].to_numpy(
            dtype=float
        )
    )

    lower_values = (
        plot_data[
            "bootstrap_95_ci_lower"
        ].to_numpy(
            dtype=float
        )
    )

    upper_values = (
        plot_data[
            "bootstrap_95_ci_upper"
        ].to_numpy(
            dtype=float
        )
    )


    # ========================================================
    # Figure
    # ========================================================

    fig = plt.figure(
        figsize=THRESHOLD_FIGSIZE
    )

    ax = fig.add_axes(
        THRESHOLD_AXES_POSITION
    )


    # --------------------------------------------------------
    # Bootstrap 95% CI
    # --------------------------------------------------------

    ax.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=ACCURACY_COLOR,
        alpha=THRESHOLD_CI_ALPHA,
        linewidth=0,
        zorder=1,
    )


    # --------------------------------------------------------
    # Median difference
    # --------------------------------------------------------

    ax.plot(
        x_values,
        median_values,
        marker="o",
        markersize=7,
        color=ACCURACY_COLOR,
        linewidth=2.6,
        zorder=3,
    )


    # --------------------------------------------------------
    # Zero reference line
    # --------------------------------------------------------

    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.1,
        alpha=0.65,
        zorder=0,
    )


    # --------------------------------------------------------
    # Significance-label offset
    # --------------------------------------------------------

    finite_values = np.concatenate([
        median_values[
            np.isfinite(
                median_values
            )
        ],
        lower_values[
            np.isfinite(
                lower_values
            )
        ],
        upper_values[
            np.isfinite(
                upper_values
            )
        ],
    ])


    if len(
        finite_values
    ):

        y_range = (
            finite_values.max()
            - finite_values.min()
        )

        offset = (
            0.05 * y_range
            if y_range > 0
            else 0.01
        )

    else:

        offset = 0.01


    # --------------------------------------------------------
    # Significance labels
    # --------------------------------------------------------

    for row in plot_data.itertuples(
        index=False
    ):

        label = (
            row.bootstrap_significance
        )

        if (
            label != ""
            and np.isfinite(
                row.median_difference
            )
        ):

            ax.text(
                row.threshold,
                row.median_difference
                + offset,
                label,
                ha="center",
                va="bottom",
                fontsize=TICK_SIZE,
                fontweight="bold",
            )


    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    ax.set_title(
        specification[
            "title"
        ],
        fontsize=FIGURE_TITLE_SIZE,
        pad=TITLE_PAD,
    )


    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Trace-area radius threshold (px)",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax.set_ylabel(
        "Within-subject difference",
        fontsize=AXIS_LABEL_SIZE,
    )


    # Explicit coordinates guarantee identical placement
    # across the two figures.
    ax.xaxis.set_label_coords(
        0.5,
        X_LABEL_Y,
    )

    ax.yaxis.set_label_coords(
        Y_LABEL_X,
        0.5,
    )


    # --------------------------------------------------------
    # Ticks
    # --------------------------------------------------------

    ax.set_xticks(
        THRESHOLDS
    )

    # Display threshold values without unnecessary ".0"
    ax.set_xticklabels(
        [
            f"{float(threshold):g}"
            for threshold in THRESHOLDS
        ],
        fontsize=TICK_SIZE,
    )

    ax.tick_params(
        axis="y",
        labelsize=TICK_SIZE,
    )


    # --------------------------------------------------------
    # Full boxed frame
    # --------------------------------------------------------

    for spine in ax.spines.values():

        spine.set_visible(
            True
        )

        spine.set_linewidth(
            1.1
        )

        spine.set_color(
            "black"
        )


    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------

    ax.grid(
        alpha=0.22,
        zorder=0,
    )


    # IMPORTANT:
    # Do NOT use fig.tight_layout().
    #
    # The axes rectangle is fixed explicitly, so differences
    # in y-axis tick-label width cannot move or resize the box.

    plt.show()


# ============================================================
# Statistics output
# ============================================================

print(
    "Meta-control model eye-movement "
    "threshold-sweep statistics"
)

display(
    threshold_summary.style.format({
        "median_difference": "{:.5f}",
        "bootstrap_95_ci_lower": "{:.5f}",
        "bootstrap_95_ci_upper": "{:.5f}",
        "bootstrap_p_value": "{:.6g}",
        "wilcoxon_W": "{:.3f}",
        "wilcoxon_p_value_raw": "{:.6g}",
        "wilcoxon_p_value_holm_across_thresholds": "{:.6g}",
    })
)

## 5. Scene-level abstraction proportion vs eye measures

Meta-control model only. Statistics are displayed separately and are **not** printed on the figures.


In [ ]:
# ============================================================
# 7. Meta-control model only, scene level:
#    abstraction proportion vs saccade proportion
#    abstraction proportion vs pupil size
# ============================================================


# ============================================================
# Shared figure formatting
# ============================================================

SCENE_SCATTER_FIGSIZE = (8.5, 6.5)

# Fixed axes rectangle:
# [left, bottom, width, height]
SCENE_SCATTER_AXES_POSITION = [
    0.16,   # left
    0.15,   # bottom
    0.80,   # width
    0.74,   # height
]

FIGURE_TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_TEXT_SIZE = 12

TITLE_PAD = 12

# Explicit axis-label coordinates.
# These guarantee identical placement across both figures.
X_LABEL_Y = -0.105
Y_LABEL_X = -0.115


# ============================================================
# Helper functions
# ============================================================

def add_segment_length(
    segments,
):
    output = segments.copy()

    output[
        "segment_length_px"
    ] = np.sqrt(
        (
            output["end_x"]
            - output["start_x"]
        ) ** 2
        +
        (
            output["end_y"]
            - output["start_y"]
        ) ** 2
    )

    return output


def meta_control_scene_abstraction_proportion(
    segments,
):
    lengths = (
        add_segment_length(
            segments
        )
        .groupby(
            [
                "scene_name",
                "reasoning_trace",
            ],
            as_index=False,
        )
        .agg(
            total_length_px=(
                "segment_length_px",
                "sum",
            )
        )
        .pivot_table(
            index="scene_name",
            columns="reasoning_trace",
            values="total_length_px",
            fill_value=0,
            aggfunc="sum",
        )
        .reset_index()
    )

    for column in [
        "abstraction",
        "simulation",
    ]:
        if column not in lengths.columns:
            lengths[column] = 0.0

    lengths[
        "abstraction_proportion"
    ] = np.where(
        (
            lengths["abstraction"]
            + lengths["simulation"]
        ) > 0,
        (
            lengths["abstraction"]
            /
            (
                lengths["abstraction"]
                + lengths["simulation"]
            )
        ),
        np.nan,
    )

    return lengths[
        [
            "scene_name",
            "abstraction",
            "simulation",
            "abstraction_proportion",
        ]
    ]


# ============================================================
# Human scene metrics:
# participant × scene first,
# then participant average within scene
# ============================================================

human_scene_data = human.copy()

human_scene_data = (
    human_scene_data
    .sort_values(
        [
            "subject_id",
            "scene_name",
            TIME_COLUMN,
        ],
        kind="mergesort",
    )
    .reset_index(drop=True)
)


participant_scene_metrics = (
    human_scene_data
    .groupby(
        [
            "scene_name",
            "subject_id",
        ],
        as_index=False,
    )
    .agg(
        participant_mean_log_pupil=(
            "log_pupil",
            "mean",
        ),
        participant_saccade_proportion=(
            "is_saccade",
            "mean",
        ),
    )
)


scene_human_metrics = (
    participant_scene_metrics
    .groupby(
        "scene_name",
        as_index=False,
    )
    .agg(
        scene_mean_saccade_proportion=(
            "participant_saccade_proportion",
            "mean",
        ),
        scene_mean_log_pupil=(
            "participant_mean_log_pupil",
            "mean",
        ),
        n_participants=(
            "subject_id",
            "nunique",
        ),
    )
)


# ============================================================
# Meta-control abstraction proportion
# ============================================================

abstraction_data = (
    meta_control_scene_abstraction_proportion(
        meta_control_segments
    )
)


# ============================================================
# Merge scene-level data
# ============================================================

scene_analysis_data = (
    abstraction_data
    .merge(
        scene_human_metrics,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        scene_meta[
            [
                "scene_name",
                "straight_path",
                "path_group",
            ]
        ],
        on="scene_name",
        how="left",
        validate="one_to_one",
    )
)


scene_analysis_data = (
    scene_analysis_data.loc[
        scene_analysis_data[
            "scene_name"
        ].ne(
            EXCLUDED_EYE_SCENE
        )
    ]
    .copy()
)


scene_analysis_data[
    "straight_path"
] = safe_numeric(
    scene_analysis_data[
        "straight_path"
    ]
)


# ============================================================
# Scatter-plot specifications
# ============================================================

scatter_specs = [
    {
        "y_column": (
            "scene_mean_saccade_proportion"
        ),
        "y_label": (
            "Human saccade proportion"
        ),
        "title": (
            "Saccade proportion vs abstraction proportion"
        ),
    },
    {
        "y_column": (
            "scene_mean_log_pupil"
        ),
        "y_label": (
            "Human mean log pupil size"
        ),
        "title": (
            "Pupil size vs abstraction proportion"
        ),
    },
]


# ============================================================
# Fit, plot, and summarize
# ============================================================

scene_scatter_statistics_rows = []


for specification in scatter_specs:

    y_column = specification[
        "y_column"
    ]


    # --------------------------------------------------------
    # Clean data
    # --------------------------------------------------------

    data = (
        scene_analysis_data
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=[
                "abstraction_proportion",
                y_column,
                "straight_path",
            ]
        )
    )


    # --------------------------------------------------------
    # Pearson correlation
    # --------------------------------------------------------

    correlation = pearsonr(
        data[
            "abstraction_proportion"
        ],
        data[
            y_column
        ],
    )


    # --------------------------------------------------------
    # OLS line + 95% CI
    # --------------------------------------------------------

    line = fit_line_with_95_ci(
        data[
            "abstraction_proportion"
        ],
        data[
            y_column
        ],
    )

    slope = float(
        line[
            "result"
        ].params[1]
    )

    slope_p = float(
        line[
            "result"
        ].pvalues[1]
    )


    # ========================================================
    # Figure
    # ========================================================

    fig = plt.figure(
        figsize=SCENE_SCATTER_FIGSIZE
    )

    ax = fig.add_axes(
        SCENE_SCATTER_AXES_POSITION
    )


    # --------------------------------------------------------
    # Scatter points by path type
    # --------------------------------------------------------

    for path_value, color in [
        (
            0,
            NON_STRAIGHT_COLOR,
        ),
        (
            1,
            STRAIGHT_COLOR,
        ),
    ]:

        group = data.loc[
            data[
                "straight_path"
            ].eq(
                path_value
            )
        ]


        ax.scatter(
            group[
                "abstraction_proportion"
            ],
            group[
                y_column
            ],
            s=58,
            alpha=0.80,
            color=color,
            edgecolors="black",
            linewidths=0.35,
            zorder=3,
        )


    # --------------------------------------------------------
    # 95% CI
    # --------------------------------------------------------

    ax.fill_between(
        line[
            "x_line"
        ],
        line[
            "ci_lower"
        ],
        line[
            "ci_upper"
        ],
        color=FIT_COLOR,
        alpha=CI_ALPHA,
        linewidth=0,
        zorder=1,
    )


    # --------------------------------------------------------
    # Overall fitted line
    # --------------------------------------------------------

    ax.plot(
        line[
            "x_line"
        ],
        line[
            "mean"
        ],
        color=FIT_COLOR,
        linewidth=2.8,
        zorder=4,
    )


    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------

    legend_handles = [
        Line2D(
            [0], [0],
            color=NON_STRAIGHT_COLOR,
            marker="o",
            linestyle="None",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=0.4,
            label="Non-straight path",
        ),
        Line2D(
            [0], [0],
            color=STRAIGHT_COLOR,
            marker="o",
            linestyle="None",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=0.4,
            label="Straight path",
        ),
        Line2D(
            [0], [0],
            color="black",
            linewidth=2.8,
            label="All scenes",
        ),
    ]


    ax.legend(
        handles=legend_handles,
        loc="upper left",
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="black",
        fontsize=LEGEND_TEXT_SIZE,
    )


    # --------------------------------------------------------
    # Title
    # --------------------------------------------------------

    ax.set_title(
        specification[
            "title"
        ],
        fontsize=FIGURE_TITLE_SIZE,
        pad=TITLE_PAD,
    )


    # --------------------------------------------------------
    # Axis labels
    # --------------------------------------------------------

    ax.set_xlabel(
        "Model abstraction proportion",
        fontsize=AXIS_LABEL_SIZE,
    )

    ax.set_ylabel(
        specification[
            "y_label"
        ],
        fontsize=AXIS_LABEL_SIZE,
    )


    # Explicit coordinates guarantee the labels are at
    # exactly the same locations in both figures.
    ax.xaxis.set_label_coords(
        0.5,
        X_LABEL_Y,
    )

    ax.yaxis.set_label_coords(
        Y_LABEL_X,
        0.5,
    )


    # --------------------------------------------------------
    # Tick styling
    # --------------------------------------------------------

    ax.tick_params(
        axis="both",
        labelsize=TICK_SIZE,
    )


    # --------------------------------------------------------
    # Full boxed frame
    # --------------------------------------------------------

    for spine in ax.spines.values():

        spine.set_visible(
            True
        )

        spine.set_linewidth(
            1.1
        )

        spine.set_color(
            "black"
        )


    # --------------------------------------------------------
    # Grid
    # --------------------------------------------------------

    ax.grid(
        alpha=0.22,
        zorder=0,
    )


    # IMPORTANT:
    # Do NOT use fig.tight_layout().
    # The axes rectangle is explicitly fixed so differences
    # in tick-label width cannot move or resize the plot box.

    plt.show()


    # --------------------------------------------------------
    # Statistics
    # --------------------------------------------------------

    scene_scatter_statistics_rows.append({
        "analysis": specification[
            "title"
        ],
        "n_scenes": len(data),
        "pearson_r": float(
            correlation.statistic
        ),
        "ols_slope": slope,
        "p_value": slope_p,
    })


# ============================================================
# Statistics table
# ============================================================

scene_scatter_statistics = pd.DataFrame(
    scene_scatter_statistics_rows
)


print(
    "Scene-level correlation and slope statistics "
    "(not printed on figures)"
)


display(
    scene_scatter_statistics.style.format({
        "pearson_r": "{:.5f}",
        "ols_slope": "{:.5f}",
        "p_value": "{:.6g}",
    })
)

# Part IIIA — Self-contained comparison-data preparation (no reference figures)

These cells rebuild the three source tables needed by Part IIIB directly under `~/Downloads/Meta-control/final_figure_analysis/model_comparison/`.

No plotting section from the reference notebook is run here.


In [ ]:

# ============================================================
# 1. Imports, paths, settings, and output folders
# ============================================================
from pathlib import Path
import json
import math
import os
import re
import warnings
import sys
import importlib
from typing import Any, Callable, Optional, Union

import pydantic

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf

from scipy.stats import pearsonr, mannwhitneyu, wilcoxon, rankdata, t
from statsmodels.stats.multitest import multipletests
from IPython.display import display

pd.set_option("display.max_columns", 250)
pd.set_option("display.width", 240)

HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"

# -----------------------------
# Inputs
# -----------------------------
SCENE_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "scene_summary.csv"
HUMAN_SUMMARY_PATH = BASE_DIR / "data" / "scene_summary" / "empirical_scene_level_rt_accuracy_summary.csv"
CLASSIFIED_TRIAL_PATH = BASE_DIR / "data" / "eye_gaze" / "classified_trial_data.csv"

HMC_PARAMETER_LABEL = "A150_S20_e22"
HMC_SUMMARY_CANDIDATES = [
    BASE_DIR / "meta_control" / HMC_PARAMETER_LABEL / "hybrid_scene_summary.csv",
    BASE_DIR / "meta_control" / "combined_hybrid_scene_summary.csv",
]
HMC_PREDICTION_DIR = (
    BASE_DIR / "meta_control" / HMC_PARAMETER_LABEL / "hybrid_predictions"
)

BLENDED_MODEL_PATH = BASE_DIR / "blended_model" / "model_predictions.csv"

# Optional manual overrides if the blended CSV uses unusual column names.
BLENDED_RUNTIME_COLUMN = None
BLENDED_RUNTIME_UNIT = "auto"   # one of: auto, frames, ms, sec
BLENDED_CORRECT_COLUMN = None
BLENDED_PREDICTED_HIT_COLUMN = None
BLENDED_TRUE_HIT_COLUMN = None


PROJECT_ROOT = BASE_DIR / "physics_abstraction_master"

# The point-level trajectory file does not contain blended accuracy.
# The JSON configuration classes and BlendedModel implementation are embedded
# in this notebook. The original physics engine modules (`scene.py` and
# `utility.py`) and Experiment 2 JSON scene files are read from physics_abstraction_master.
# These optional overrides are needed only if automatic discovery fails.
BLENDED_PHYSICS_ENGINE_DIR = PROJECT_ROOT / "python"
BLENDED_EXP2_SCENE_DIR = PROJECT_ROOT / "data" / "json" / "experiment2"
BLENDED_ACCURACY_SAMPLES = 50
BLENDED_MODEL_PARAMETERS = {
    "view": False,
    "noise": 0.001,
    "N": 5,
    "D": 75,
    "E": 0.9,
}
REBUILD_BLENDED_COLLISION_CACHE = False

# -----------------------------
# Outputs
# -----------------------------
OUTPUT_ROOT = BASE_DIR / "final_figure_analysis" / "model_comparison"
MODEL_OUTPUTS = {
    "HMC": OUTPUT_ROOT / "HMC",
    "blended": OUTPUT_ROOT / "blended",
}
for model_dir in MODEL_OUTPUTS.values():
    model_dir.mkdir(parents=True, exist_ok=True)
    (model_dir / "tables").mkdir(parents=True, exist_ok=True)

COMPARISON_OUTPUT = OUTPUT_ROOT / "HMC_vs_blended"
COMPARISON_OUTPUT.mkdir(parents=True, exist_ok=True)
(COMPARISON_OUTPUT / "tables").mkdir(parents=True, exist_ok=True)

BLENDED_COLLISION_CACHE_PATH = (
    MODEL_OUTPUTS["blended"] / "tables" / "blended_collision_samples_source_logic_v1.csv"
)

# -----------------------------
# Shared analysis settings
# -----------------------------
FRAMES_PER_SECOND = 60.0
MS_PER_SECOND = 1000.0

THRESHOLDS = np.arange(50, 121, 10, dtype=float)
SCATTER_THRESHOLD = 120.0
HMC_TRACE_START_FRAME = 15

HUMAN_X_SHIFT = 0.0
HUMAN_Y_SHIFT = -40.0

HMC_X_SHIFT = 560.0
HMC_Y_SHIFT = 0.0
BLENDED_X_SHIFT = 560.0
BLENDED_Y_SHIFT = 0.0

DISPLAY_X_MIN = 560.0
DISPLAY_X_MAX = 1360.0
DISPLAY_Y_MIN = 0.0
DISPLAY_Y_MAX = 1080.0

USE_STRICT_SEGMENT_Y_RANGE = True
INCLUDE_OVERLAP_IN_BOTH_POOLS = True

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260729
REBUILD_DISTANCE_CACHE = False

POLYNOMIAL_ORDER = 8
EXCLUDED_EYE_SCENE = "low_yescol_yessp_4"

NO_STRAIGHT_COLOR = "#1C77C3"
STRAIGHT_COLOR = "#F39237"
ACCURACY_COLOR = "#1C77C3"
FIT_COLOR = "#4D4D4D"
CI_ALPHA = 0.16
THRESHOLD_CI_ALPHA = 0.18

PATH_GROUP_ORDER = ["no_straight_path", "straight_path"]
PATH_COLORS = {
    "no_straight_path": NO_STRAIGHT_COLOR,
    "straight_path": STRAIGHT_COLOR,
}

plt.rcParams.update({
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "legend.title_fontsize": LEGEND_SIZE,
})

for path, label in [
    (SCENE_SUMMARY_PATH, "scene_summary.csv"),
    (HUMAN_SUMMARY_PATH, "empirical_scene_level_rt_accuracy_summary.csv"),
    (CLASSIFIED_TRIAL_PATH, "classified_trial_data.csv"),
    (BLENDED_MODEL_PATH, "model_predictions.csv"),
    (HMC_PREDICTION_DIR, "HMC prediction directory"),
]:
    if not path.exists():
        raise FileNotFoundError(f"Missing {label}: {path}")

HMC_SUMMARY_PATH = next(
    (path for path in HMC_SUMMARY_CANDIDATES if path.exists()),
    None,
)
if HMC_SUMMARY_PATH is None:
    raise FileNotFoundError(
        "Could not find the HMC scene summary. Tried:\n"
        + "\n".join(str(path) for path in HMC_SUMMARY_CANDIDATES)
    )

print("HMC summary:", HMC_SUMMARY_PATH)
print("HMC predictions:", HMC_PREDICTION_DIR)
print("Blended predictions:", BLENDED_MODEL_PATH)
print("Human gaze shift:", {"x": HUMAN_X_SHIFT, "y": HUMAN_Y_SHIFT})
print("Output root:", OUTPUT_ROOT)


In [ ]:
# ============================================================
# 1B. Embedded JSON utilities and blended-model implementation
# ============================================================
# This cell embeds the contents needed from the supplied json_utilities.py and
# model_utilities.py. Separate copies of those two files are not required.
#
# BlendedModel.sample still uses the project's underlying physics engine:
# scene.Scene and utility.get_objects_from_scene_config. The loader below finds
# scene.py and utility.py automatically, usually in the project's python folder.


class LineConfig(pydantic.BaseModel):
    """A datatype representing a line's properties."""
    point_a: list[Union[int, float]]
    point_b: list[Union[int, float]]


class GoalConfig(pydantic.BaseModel):
    """A datatype representing a goal's properties."""
    position: list[Union[int, float]]


class BallConfig(pydantic.BaseModel):
    """A datatype representing a ball's properties."""
    position: list[Union[int, float]]


class ContainerConfig(pydantic.BaseModel):
    """A datatype representing a container's properties."""
    position: list[Union[int, float]]
    width: Union[int, float]
    length: Union[int, float]
    angle: Union[int, float]
    id: Optional[int] = 99


class SceneConfig(pydantic.BaseModel):
    """A datatype representing scene configurations."""
    name: str
    screen_size: list[Union[int, float]]
    objects: list[str]
    ball_args: list[BallConfig]
    goal_args: list[GoalConfig]
    container_args: Optional[list[ContainerConfig]] = None
    line_args: Optional[list[LineConfig]] = None
    bottom_border_args: list[list[Union[int, float]]]
    plinko_border_args: list[Union[int, float]]


def _model_dump_compat(model):
    """Return a dictionary for either Pydantic v1 or v2 models."""
    if hasattr(model, "model_dump"):
        return model.model_dump()
    if hasattr(model, "dict"):
        return model.dict()
    if isinstance(model, dict):
        return model
    raise TypeError(f"Cannot convert {type(model)} to a dictionary.")


def dict_to_model(scene_json: dict) -> SceneConfig:
    """Convert a JSON-compatible dictionary into a SceneConfig."""
    return SceneConfig(**scene_json)


def json_file_to_model(path_to_json) -> SceneConfig:
    """Read a scene JSON file and convert it into a SceneConfig."""
    with open(path_to_json, "r") as file:
        return SceneConfig(**json.load(file))


def model_to_json_file(scene_model: SceneConfig, path_to_json):
    """Write a SceneConfig to disk as JSON."""
    with open(path_to_json, "w") as file:
        json.dump(_model_dump_compat(scene_model), file)


_PHYSICS_SCENE_MODULE = None
_PHYSICS_UTILITY_MODULE = None
_PHYSICS_ENGINE_DIR_RESOLVED = None


def _unique_paths(values):
    output = []
    seen = set()
    for value in values:
        if value is None:
            continue
        path = Path(value).expanduser()
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)
        if key not in seen:
            output.append(path)
            seen.add(key)
    return output


def _home_glob_limited(patterns):
    matches = []
    for pattern in patterns:
        try:
            matches.extend(HOME.glob(pattern))
        except Exception:
            pass
    return matches


def _valid_physics_modules(scene_module, utility_module):
    return (
        hasattr(scene_module, "Scene")
        and hasattr(utility_module, "get_objects_from_scene_config")
    )


def _load_physics_dependencies(exp2_scene_dir=None):
    """Locate and import scene.py and utility.py used by the original model."""
    global _PHYSICS_SCENE_MODULE
    global _PHYSICS_UTILITY_MODULE
    global _PHYSICS_ENGINE_DIR_RESOLVED

    if _valid_physics_modules(
        _PHYSICS_SCENE_MODULE,
        _PHYSICS_UTILITY_MODULE,
    ):
        return (
            _PHYSICS_SCENE_MODULE,
            _PHYSICS_UTILITY_MODULE,
            _PHYSICS_ENGINE_DIR_RESOLVED,
        )

    # First accept modules that are already importable in the active environment.
    try:
        scene_module = importlib.import_module("scene")
        utility_module = importlib.import_module("utility")
        if _valid_physics_modules(scene_module, utility_module):
            _PHYSICS_SCENE_MODULE = scene_module
            _PHYSICS_UTILITY_MODULE = utility_module
            scene_file = getattr(scene_module, "__file__", None)
            _PHYSICS_ENGINE_DIR_RESOLVED = (
                Path(scene_file).resolve().parent if scene_file else None
            )
            return (
                _PHYSICS_SCENE_MODULE,
                _PHYSICS_UTILITY_MODULE,
                _PHYSICS_ENGINE_DIR_RESOLVED,
            )
    except Exception:
        pass

    explicit_dir = (
        None
        if BLENDED_PHYSICS_ENGINE_DIR is None
        else Path(BLENDED_PHYSICS_ENGINE_DIR).expanduser()
    )
    paired_project_python = None
    if exp2_scene_dir is not None:
        scene_dir = Path(exp2_scene_dir).expanduser()
        # .../data/json/experiment2 -> project root -> project/python
        if len(scene_dir.parents) >= 3:
            paired_project_python = scene_dir.parents[2] / "python"

    candidates = _unique_paths(
        [
            explicit_dir,
            paired_project_python,
            Path.cwd(),
            Path.cwd() / "python",
            Path.cwd().parent / "python",
            PROJECT_ROOT / "python",
        ]
        + _home_glob_limited([
            "*/python",
            "*/*/python",
            "*/*/*/python",
        ])
    )
    candidates = [
        path
        for path in candidates
        if (
            path.is_dir()
            and (path / "scene.py").exists()
            and (path / "utility.py").exists()
        )
    ]

    import_errors = []
    for candidate in candidates:
        candidate_text = str(candidate.resolve())
        if candidate_text not in sys.path:
            sys.path.insert(0, candidate_text)
        importlib.invalidate_caches()

        # Remove any unrelated modules with these generic names before retrying.
        for module_name in ("scene", "utility"):
            existing = sys.modules.get(module_name)
            existing_file = getattr(existing, "__file__", "") if existing else ""
            if existing is not None and candidate_text not in str(existing_file):
                sys.modules.pop(module_name, None)

        try:
            scene_module = importlib.import_module("scene")
            utility_module = importlib.import_module("utility")
            if not _valid_physics_modules(scene_module, utility_module):
                raise ImportError(
                    "Imported modules do not expose Scene and "
                    "get_objects_from_scene_config."
                )
            _PHYSICS_SCENE_MODULE = scene_module
            _PHYSICS_UTILITY_MODULE = utility_module
            _PHYSICS_ENGINE_DIR_RESOLVED = candidate.resolve()
            return (
                _PHYSICS_SCENE_MODULE,
                _PHYSICS_UTILITY_MODULE,
                _PHYSICS_ENGINE_DIR_RESOLVED,
            )
        except Exception as error:
            import_errors.append(f"{candidate}: {error}")

    details = "\n".join(import_errors[-5:]) if import_errors else "No candidate folder was found."
    raise ImportError(
        "The notebook embeds json_utilities.py and model_utilities.py, but "
        "BlendedModel.sample also requires the original physics-engine files "
        "scene.py and utility.py. They could not be imported automatically.\n\n"
        "Set BLENDED_PHYSICS_ENGINE_DIR in the settings cell to the project "
        "folder that contains scene.py and utility.py.\n\n"
        f"Discovery details:\n{details}"
    )


def model_call_wrapper(model: Callable, model_parameters: dict[str, Any]):
    """A model-call wrapper that uses currying."""
    def model_call(input_):
        return model(input_, *model_parameters)
    return model_call


def get_scene_objects(
    object_constructors: Callable,
    object_arguments: pydantic.BaseModel,
    noise=None,
):
    """Return object instances for one scene configuration."""
    objects = []
    for obj, obj_args in zip(object_constructors, object_arguments):
        if obj.__name__ == "Ball" and noise:
            noisy_position = np.asarray(
                _model_dump_compat(obj_args)["position"],
                dtype=float,
            )
            noisy_position = noisy_position * np.random.normal(1, noise, 2)
            objects.append(obj(BallConfig(position=noisy_position.tolist())))
        else:
            try:
                objects.append(obj(obj_args))
            except TypeError:
                objects.append(obj())
    return objects


def _get_scene_from_config(scene_config: SceneConfig, noise: float):
    scene_module, utility_module, _ = _load_physics_dependencies()
    object_argument_pairs = utility_module.get_objects_from_scene_config(
        scene_args_config=scene_config
    )
    objects = get_scene_objects(
        object_constructors=object_argument_pairs["object_constructors"],
        object_arguments=object_argument_pairs["arguments"],
        noise=noise,
    )
    return scene_module.Scene(objects)


def _parse_model_output_with_pos(scene_instance):
    position = None
    for scene_object in scene_instance.objects:
        if scene_object.name == "Ball":
            position = list(scene_object.body.position)
            break
    if position is None:
        raise RuntimeError("The instantiated scene contains no Ball object.")
    return ModelOutputWithPos(ball_position=position)


def _parse_model_output(scene_instance):
    return ModelOutput(
        collision=scene_instance.physics.handlers["ball_goal"].data["colliding"],
        simulation_ticks=scene_instance.physics.tick,
        trajectory_length=scene_instance.ball_distance_traveled,
    )


class ModelOutput(pydantic.BaseModel):
    """Model outputs used by the behavioral analyses."""
    collision: bool
    simulation_ticks: int
    trajectory_length: float


class ModelOutputWithPos(pydantic.BaseModel):
    """Model output containing the terminal ball position."""
    ball_position: list[float]


class PhysModel:
    """Base class for models used in the physics-abstraction project."""
    def __init__(self):
        pass


class SimulationModel(PhysModel):
    """Pure simulation model."""
    def __init__(self, model_parameters: dict[str, Any]):
        super().__init__()
        self.model_parameters = model_parameters
        self.output = None

    def sample(self, scene_config: SceneConfig):
        scene_instance = _get_scene_from_config(
            scene_config=scene_config,
            noise=self.model_parameters["noise"],
        )
        scene_instance.instantiate_scene()
        scene_instance.run(view=self.model_parameters["view"])
        self.output = _parse_model_output(scene_instance)
        return self.output


class BlendedModel(PhysModel):
    """Blended simulation-and-abstraction model."""
    def __init__(self, model_parameters: dict[str, Any]):
        super().__init__()
        self.model_parameters = model_parameters
        self.output = None

    def sample(self, scene_config: SceneConfig):
        """Run one stochastic blended-model sample on a SceneConfig."""
        scene_instance = _get_scene_from_config(
            scene_config=scene_config,
            noise=self.model_parameters["noise"],
        )
        scene_instance.instantiate_scene()
        scene_instance.run_path(
            view=self.model_parameters["view"],
            N=self.model_parameters["N"],
            D=self.model_parameters["D"],
            E=self.model_parameters["E"],
        )
        self.output = _parse_model_output(scene_instance)
        return self.output

    def sample_with_pos(self, scene_config: SceneConfig):
        """Run one blended-model sample and return its terminal position."""
        scene_instance = _get_scene_from_config(
            scene_config=scene_config,
            noise=self.model_parameters["noise"],
        )
        scene_instance.instantiate_scene()
        scene_instance.run_path(
            view=self.model_parameters["view"],
            N=self.model_parameters["N"],
            D=self.model_parameters["D"],
            E=self.model_parameters["E"],
        )
        self.output = _parse_model_output_with_pos(scene_instance)
        return self.output


print("Embedded SceneConfig and BlendedModel definitions are ready.")


In [ ]:

# ============================================================
# 2. Shared helper functions
# ============================================================
def safe_numeric(values):
    return pd.to_numeric(values, errors="coerce")


def normalize_scene_name(value):
    text = Path(str(value).strip().replace("\\", "/")).name
    for suffix in (".json", ".mp4", ".csv"):
        if text.lower().endswith(suffix):
            text = text[:-len(suffix)]
    return text.strip()


def find_first_column(dataframe, candidates, required=False, label="column"):
    column = next((candidate for candidate in candidates if candidate in dataframe.columns), None)
    if required and column is None:
        raise KeyError(
            f"Could not identify {label}. Tried {candidates}.\n"
            f"Available columns:\n{list(dataframe.columns)}"
        )
    return column


def in_display_bounds(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    return (
        np.isfinite(x)
        & np.isfinite(y)
        & (x >= DISPLAY_X_MIN)
        & (x <= DISPLAY_X_MAX)
        & (y >= DISPLAY_Y_MIN)
        & (y <= DISPLAY_Y_MAX)
    )


def normalize_trace_value(value):
    text = str(value).strip().lower()
    if "abstract" in text:
        return "abstraction"
    if "sim" in text:
        return "simulation"
    return np.nan


def normalize_trace_series(values):
    return values.map(normalize_trace_value)


def parse_binary_value(value):
    if pd.isna(value):
        return np.nan
    if isinstance(value, (bool, np.bool_)):
        return int(bool(value))
    if isinstance(value, (int, np.integer, float, np.floating)):
        if not np.isfinite(float(value)):
            return np.nan
        return int(float(value) >= 0.5)

    text = str(value).strip().lower().replace("-", "_").replace(" ", "_")
    positive = {
        "1", "true", "yes", "y", "correct", "hit", "goal", "goal_hit",
        "model_correct", "success", "positive",
    }
    negative = {
        "0", "false", "no", "n", "incorrect", "miss", "ground",
        "ground_hit", "model_incorrect", "failure", "negative",
    }
    if text in positive:
        return 1
    if text in negative:
        return 0
    if "goal" in text and "ground" not in text:
        return 1
    if "ground" in text:
        return 0
    return np.nan


def first_nonmissing(values):
    values = pd.Series(values).dropna()
    return values.iloc[0] if len(values) else np.nan


def last_nonmissing(values):
    values = pd.Series(values).dropna()
    return values.iloc[-1] if len(values) else np.nan


def save_figure(fig, output_dir, file_stub):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    pdf_path = output_dir / f"{file_stub}.pdf"
    png_path = output_dir / f"{file_stub}.png"
    fig.savefig(pdf_path, format="pdf", bbox_inches="tight")
    fig.savefig(png_path, dpi=300, bbox_inches="tight")
    print("Saved PDF:", pdf_path)
    print("Saved PNG:", png_path)
    return pdf_path, png_path


def format_p_value(p_value):
    if pd.isna(p_value):
        return "NA"
    if p_value < 0.001:
        return f"{p_value:.2e}"
    return f"{p_value:.3f}"


def add_r_p_annotation(ax, r_value, p_value, n_value):
    ax.text(
        0.04,
        0.96,
        f"r = {r_value:.3f}\np = {format_p_value(p_value)}\nN = {int(n_value)}",
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=12,
    )


def significance_label(p_value):
    if pd.isna(p_value):
        return "not testable"
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


def fit_line_with_95_ci(x, y, n_line=300):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    design = sm.add_constant(x, has_constant="add")
    result = sm.OLS(y, design).fit()
    x_line = np.linspace(np.min(x), np.max(x), int(n_line))
    prediction_design = sm.add_constant(x_line, has_constant="add")
    prediction = result.get_prediction(prediction_design).summary_frame(alpha=0.05)
    return {
        "result": result,
        "x_line": x_line,
        "mean": prediction["mean"].to_numpy(dtype=float),
        "ci_lower": prediction["mean_ci_lower"].to_numpy(dtype=float),
        "ci_upper": prediction["mean_ci_upper"].to_numpy(dtype=float),
    }


def polynomial_curve(x, y, order=8, n_line=400):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    valid = np.isfinite(x) & np.isfinite(y)
    x = x[valid]
    y = y[valid]
    if len(x) <= order:
        raise ValueError(f"Need more than {order} valid rows for an order-{order} curve.")
    # Scaling inside Polynomial.fit avoids the numerical instability of raw high-order powers.
    fitted = np.polynomial.Polynomial.fit(x, y, deg=int(order))
    x_line = np.linspace(np.min(x), np.max(x), int(n_line))
    y_line = fitted(x_line)
    return x_line, y_line


def bootstrap_median_summary(values, seed, n_bootstrap=N_BOOTSTRAP):
    """Bootstrap the participant-level median and return CI and two-sided significance.

    The two-sided bootstrap p-value is the doubled smaller tail probability of
    the resampled medians relative to zero, with a +1 correction so it is never
    exactly zero. The same bootstrap draws therefore determine both the plotted
    confidence band and the plotted significance markers.
    """
    values = safe_numeric(pd.Series(values))
    values = values.loc[np.isfinite(values)].to_numpy(dtype=float)
    n_values = int(len(values))

    if n_values == 0:
        return {
            "bootstrap_n_participants": 0,
            "bootstrap_median_estimate": np.nan,
            "bootstrap_95_ci_lower_median_difference": np.nan,
            "bootstrap_95_ci_upper_median_difference": np.nan,
            "bootstrap_p_value_two_sided": np.nan,
            "bootstrap_significance": "not testable",
            "bootstrap_95_ci_excludes_zero": False,
        }

    sample_median = float(np.median(values))
    rng = np.random.default_rng(int(seed))
    sampled_indices = rng.integers(
        low=0,
        high=n_values,
        size=(int(n_bootstrap), n_values),
    )
    bootstrap_medians = np.median(values[sampled_indices], axis=1)
    lower, upper = np.quantile(bootstrap_medians, [0.025, 0.975])

    # Two-sided empirical tail probability relative to zero.
    # The +1 correction provides a finite minimum p-value of 2/(B + 1).
    lower_tail = (
        np.count_nonzero(bootstrap_medians <= 0.0) + 1
    ) / (len(bootstrap_medians) + 1)
    upper_tail = (
        np.count_nonzero(bootstrap_medians >= 0.0) + 1
    ) / (len(bootstrap_medians) + 1)
    p_value = float(min(1.0, 2.0 * min(lower_tail, upper_tail)))
    ci_excludes_zero = bool((lower > 0.0) or (upper < 0.0))

    return {
        "bootstrap_n_participants": n_values,
        "bootstrap_median_estimate": sample_median,
        "bootstrap_95_ci_lower_median_difference": float(lower),
        "bootstrap_95_ci_upper_median_difference": float(upper),
        "bootstrap_p_value_two_sided": p_value,
        "bootstrap_significance": significance_label(p_value),
        "bootstrap_95_ci_excludes_zero": ci_excludes_zero,
    }


def bootstrap_median_ci(values, seed, n_bootstrap=N_BOOTSTRAP):
    """Backward-compatible wrapper returning only the percentile CI."""
    summary = bootstrap_median_summary(
        values=values,
        seed=seed,
        n_bootstrap=n_bootstrap,
    )
    return (
        summary["bootstrap_95_ci_lower_median_difference"],
        summary["bootstrap_95_ci_upper_median_difference"],
    )

def paired_wilcoxon_summary(differences):
    difference = safe_numeric(pd.Series(differences))
    difference = difference.loc[np.isfinite(difference)].to_numpy(dtype=float)
    n_paired = int(len(difference))
    if n_paired == 0:
        return {
            "n_paired_participants": 0,
            "n_nonzero_differences": 0,
            "median_difference": np.nan,
            "mean_difference": np.nan,
            "wilcoxon_W": np.nan,
            "p_value_raw": np.nan,
            "rank_biserial_r": np.nan,
        }

    median_difference = float(np.median(difference))
    mean_difference = float(np.mean(difference))
    nonzero = difference[~np.isclose(difference, 0.0)]
    n_nonzero = int(len(nonzero))
    if n_nonzero == 0:
        return {
            "n_paired_participants": n_paired,
            "n_nonzero_differences": 0,
            "median_difference": median_difference,
            "mean_difference": mean_difference,
            "wilcoxon_W": 0.0,
            "p_value_raw": 1.0,
            "rank_biserial_r": 0.0,
        }

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        test = wilcoxon(
            difference,
            alternative="two-sided",
            zero_method="wilcox",
            correction=False,
            method="auto",
        )

    ranks = rankdata(np.abs(nonzero))
    positive_rank_sum = float(ranks[nonzero > 0].sum())
    negative_rank_sum = float(ranks[nonzero < 0].sum())
    total_rank_sum = positive_rank_sum + negative_rank_sum
    rank_biserial = (
        (positive_rank_sum - negative_rank_sum) / total_rank_sum
        if total_rank_sum > 0 else 0.0
    )

    return {
        "n_paired_participants": n_paired,
        "n_nonzero_differences": n_nonzero,
        "median_difference": median_difference,
        "mean_difference": mean_difference,
        "wilcoxon_W": float(test.statistic),
        "p_value_raw": float(test.pvalue),
        "rank_biserial_r": float(rank_biserial),
    }


In [ ]:

# ============================================================
# 3. Load common scene metadata and human scene-level outcomes
# ============================================================
scene_summary_raw = pd.read_csv(SCENE_SUMMARY_PATH, low_memory=False)
scene_name_column = find_first_column(
    scene_summary_raw,
    ["scene_name", "scene", "scene_id", "json_name", "trial_name", "trial", "name"],
    required=True,
    label="scene-name column in scene_summary.csv",
)
if "simulation_time" not in scene_summary_raw.columns:
    raise KeyError(
        f"{SCENE_SUMMARY_PATH} must contain `simulation_time`.\n"
        f"Available columns: {list(scene_summary_raw.columns)}"
    )
if "straight_path" not in scene_summary_raw.columns:
    raise KeyError(
        f"{SCENE_SUMMARY_PATH} must contain `straight_path`.\n"
        f"Available columns: {list(scene_summary_raw.columns)}"
    )

scene_meta = scene_summary_raw.rename(columns={scene_name_column: "scene_name"}).copy()
scene_meta["scene_name"] = scene_meta["scene_name"].map(normalize_scene_name)
scene_meta["simulation_time"] = safe_numeric(scene_meta["simulation_time"])
scene_meta["straight_path"] = safe_numeric(scene_meta["straight_path"]).astype("Int64")
scene_meta = scene_meta[["scene_name", "simulation_time", "straight_path"]].drop_duplicates("scene_name")

invalid_path = scene_meta.loc[
    scene_meta["straight_path"].notna() & ~scene_meta["straight_path"].isin([0, 1])
]
if len(invalid_path):
    raise ValueError("straight_path contains values other than 0 and 1.")

scene_meta["path_group"] = (
    scene_meta["straight_path"]
    .map({0: "no_straight_path", 1: "straight_path"})
    .astype("string")
)
scene_meta["true_simulation_time_ms"] = (
    scene_meta["simulation_time"] / FRAMES_PER_SECOND * MS_PER_SECOND
)

human_summary_raw = pd.read_csv(HUMAN_SUMMARY_PATH, low_memory=False)
human_scene_column = find_first_column(
    human_summary_raw,
    ["scene_name", "scenename", "scene", "name"],
    required=True,
    label="scene-name column in the human summary",
)
for required_column in ["mean_judgment_time_sec", "mean_accuracy"]:
    if required_column not in human_summary_raw.columns:
        raise KeyError(
            f"{HUMAN_SUMMARY_PATH} is missing `{required_column}`.\n"
            f"Available columns: {list(human_summary_raw.columns)}"
        )

human_summary = human_summary_raw.rename(columns={human_scene_column: "scene_name"}).copy()
human_summary["scene_name"] = human_summary["scene_name"].map(normalize_scene_name)
human_summary["mean_judgment_time_sec"] = safe_numeric(human_summary["mean_judgment_time_sec"])
human_summary["mean_accuracy"] = safe_numeric(human_summary["mean_accuracy"])
human_summary["human_response_time_ms"] = (
    human_summary["mean_judgment_time_sec"] * MS_PER_SECOND
)
human_summary = human_summary[
    ["scene_name", "mean_judgment_time_sec", "human_response_time_ms", "mean_accuracy"]
].drop_duplicates("scene_name")

common_scene_data = (
    scene_meta
    .merge(human_summary, on="scene_name", how="inner", validate="one_to_one")
)

print("Common scene rows:", len(common_scene_data))
print("Exp1 rows:", (~common_scene_data["scene_name"].str.startswith("scene", na=False)).sum())
print("Exp2 rows:", common_scene_data["scene_name"].str.startswith("scene", na=False).sum())
display(common_scene_data.head())


In [ ]:

# ============================================================
# 4. Load the HMC scene-level summary
# ============================================================
hmc_raw = pd.read_csv(HMC_SUMMARY_PATH, low_memory=False)

if "parameter_label" in hmc_raw.columns:
    hmc_raw = hmc_raw.loc[
        hmc_raw["parameter_label"].astype(str).eq(HMC_PARAMETER_LABEL)
    ].copy()

required_hmc_columns = [
    "scene_name",
    "hybrid_total_step_count",
    "hybrid_correct",
]
missing_hmc = [column for column in required_hmc_columns if column not in hmc_raw.columns]
if missing_hmc:
    raise KeyError(
        f"HMC summary is missing {missing_hmc}.\n"
        f"Available columns: {list(hmc_raw.columns)}"
    )

hmc_scene_summary = hmc_raw.copy()
hmc_scene_summary["scene_name"] = hmc_scene_summary["scene_name"].map(normalize_scene_name)
hmc_scene_summary["model_total_step_count"] = safe_numeric(
    hmc_scene_summary["hybrid_total_step_count"]
)
hmc_scene_summary["model_run_time_ms"] = (
    hmc_scene_summary["model_total_step_count"]
    / FRAMES_PER_SECOND
    * MS_PER_SECOND
)
hmc_scene_summary["model_correct"] = safe_numeric(
    hmc_scene_summary["hybrid_correct"]
)
hmc_scene_summary["model_accuracy"] = hmc_scene_summary["model_correct"]
hmc_scene_summary["model"] = "HMC"

if "true_hit" in hmc_scene_summary.columns:
    hmc_scene_summary["true_hit"] = hmc_scene_summary["true_hit"].map(parse_binary_value)
else:
    hmc_scene_summary["true_hit"] = np.nan

hmc_scene_summary = (
    hmc_scene_summary[
        ["scene_name", "model_total_step_count", "model_run_time_ms", "model_correct", "model_accuracy", "true_hit", "model"]
    ]
    .drop_duplicates("scene_name")
    .merge(common_scene_data, on="scene_name", how="inner", validate="one_to_one")
)

print("HMC scenes:", len(hmc_scene_summary))
display(hmc_scene_summary.head())


In [ ]:
# ============================================================
# 5. Load the blended model and construct scene-level metrics
# ============================================================
blended_raw = pd.read_csv(BLENDED_MODEL_PATH, low_memory=False)
print("Blended columns:")
print(list(blended_raw.columns))

required_blended_trace_columns = ["scene_name", "sample_idx", "x", "y", "segment_class"]
missing_blended_trace = [
    column for column in required_blended_trace_columns
    if column not in blended_raw.columns
]
if missing_blended_trace:
    raise KeyError(
        f"{BLENDED_MODEL_PATH} is missing {missing_blended_trace}.\n"
        f"Available columns: {list(blended_raw.columns)}"
    )

blended = blended_raw.copy()
blended["_file_order"] = np.arange(len(blended), dtype=int)
blended["scene_name"] = blended["scene_name"].map(normalize_scene_name)
blended["sample_idx"] = (
    blended["sample_idx"]
    .astype("string")
    .fillna("__single__")
    .str.strip()
)
blended["reasoning_trace"] = normalize_trace_series(blended["segment_class"])

# ------------------------------------------------------------
# Runtime column resolution
# ------------------------------------------------------------
runtime_ms_candidates = [
    "model_run_time_ms", "run_time_ms", "runtime_ms", "model_time_ms",
]
runtime_sec_candidates = [
    "model_run_time_sec", "run_time_sec", "runtime_sec", "model_time_sec",
]
runtime_frame_candidates = [
    "model_total_step_count", "total_step_count", "model_run_steps",
    "run_steps", "runtime_steps", "n_model_steps", "step_count",
    "total_steps", "n_steps",
]
runtime_ambiguous_candidates = [
    "model_run_time", "run_time", "runtime", "model_time",
]

runtime_column = BLENDED_RUNTIME_COLUMN
runtime_unit = BLENDED_RUNTIME_UNIT

if runtime_column is None:
    if find_first_column(blended, runtime_ms_candidates) is not None:
        runtime_column = find_first_column(blended, runtime_ms_candidates)
        runtime_unit = "ms"
    elif find_first_column(blended, runtime_sec_candidates) is not None:
        runtime_column = find_first_column(blended, runtime_sec_candidates)
        runtime_unit = "sec"
    elif find_first_column(blended, runtime_frame_candidates) is not None:
        runtime_column = find_first_column(blended, runtime_frame_candidates)
        runtime_unit = "frames"
    elif find_first_column(blended, runtime_ambiguous_candidates) is not None:
        runtime_column = find_first_column(blended, runtime_ambiguous_candidates)
        runtime_unit = runtime_unit if runtime_unit != "auto" else "frames"

group_columns = ["scene_name", "sample_idx"]

if runtime_column is not None:
    blended[runtime_column] = safe_numeric(blended[runtime_column])
    runtime_per_sample = (
        blended
        .groupby(group_columns, as_index=False)
        .agg(runtime_value=(runtime_column, last_nonmissing))
    )
    if runtime_unit == "ms":
        runtime_per_sample["model_run_time_ms"] = safe_numeric(
            runtime_per_sample["runtime_value"]
        )
        runtime_per_sample["model_total_step_count"] = (
            runtime_per_sample["model_run_time_ms"]
            / MS_PER_SECOND
            * FRAMES_PER_SECOND
        )
    elif runtime_unit == "sec":
        runtime_per_sample["model_run_time_ms"] = (
            safe_numeric(runtime_per_sample["runtime_value"]) * MS_PER_SECOND
        )
        runtime_per_sample["model_total_step_count"] = (
            safe_numeric(runtime_per_sample["runtime_value"]) * FRAMES_PER_SECOND
        )
    elif runtime_unit == "frames":
        runtime_per_sample["model_total_step_count"] = safe_numeric(
            runtime_per_sample["runtime_value"]
        )
        runtime_per_sample["model_run_time_ms"] = (
            runtime_per_sample["model_total_step_count"]
            / FRAMES_PER_SECOND
            * MS_PER_SECOND
        )
    else:
        raise ValueError(
            f"Unsupported BLENDED_RUNTIME_UNIT={runtime_unit!r}. "
            "Use auto, frames, ms, or sec."
        )
    runtime_method = (
        f"explicit column `{runtime_column}` interpreted as {runtime_unit}"
    )
else:
    # Match the trajectory representation used by the blended threshold notebook:
    # each simulation row is one simulation step, while each contiguous
    # abstraction run is one abstraction decision.
    blended_runtime = blended.sort_values(
        ["scene_name", "sample_idx", "_file_order"],
        kind="mergesort",
    ).copy()
    blended_runtime["previous_trace"] = (
        blended_runtime
        .groupby(group_columns, sort=False)["reasoning_trace"]
        .shift(1)
    )
    blended_runtime["new_abstraction_decision"] = (
        blended_runtime["reasoning_trace"].eq("abstraction")
        & ~blended_runtime["previous_trace"].eq("abstraction")
    )
    runtime_per_sample = (
        blended_runtime
        .groupby(group_columns, as_index=False)
        .agg(
            simulation_step_count=(
                "reasoning_trace",
                lambda values: int(
                    pd.Series(values).eq("simulation").sum()
                ),
            ),
            abstraction_step_count=(
                "new_abstraction_decision",
                "sum",
            ),
        )
    )
    runtime_per_sample["model_total_step_count"] = (
        runtime_per_sample["simulation_step_count"]
        + runtime_per_sample["abstraction_step_count"]
    )
    runtime_per_sample["model_run_time_ms"] = (
        runtime_per_sample["model_total_step_count"]
        / FRAMES_PER_SECOND
        * MS_PER_SECOND
    )
    runtime_method = (
        "trajectory fallback: simulation rows + contiguous abstraction decisions"
    )

print("Blended runtime method:", runtime_method)

runtime_scene_summary = (
    runtime_per_sample
    .groupby("scene_name", as_index=False)
    .agg(
        model_total_step_count=("model_total_step_count", "mean"),
        model_run_time_ms=("model_run_time_ms", "mean"),
        n_runtime_samples=("sample_idx", "nunique"),
    )
)

# ------------------------------------------------------------
# Correctness / predicted-outcome resolution
# ------------------------------------------------------------
# The supplied source notebook does NOT obtain blended accuracy from
# model_predictions.csv. It constructs BlendedModel, calls sample(scene_model)
# 50 times for each Experiment 2 scene, and defines model accuracy as the
# mean returned `collision` value. It does not compare collision with true_hit.
correct_candidates = [
    "blended_correct", "model_correct", "prediction_correct", "pred_correct",
    "is_correct", "correct",
]
predicted_hit_candidates = [
    "blended_pred_hit", "model_pred_hit", "pred_hit", "predicted_hit",
    "hit_prediction", "predicted_outcome", "model_outcome", "prediction",
]
true_hit_candidates = [
    "true_hit", "actual_hit", "ground_truth_hit", "target_hit",
    "true_outcome", "actual_outcome", "ground_truth",
]
terminal_event_candidates = [
    "rollout_terminal_event", "terminal_event", "predicted_terminal_event",
    "stop_event_type", "simulation_stop_reason",
]

correct_column = (
    BLENDED_CORRECT_COLUMN
    or find_first_column(blended, correct_candidates)
)
predicted_hit_column = (
    BLENDED_PREDICTED_HIT_COLUMN
    or find_first_column(blended, predicted_hit_candidates)
)
true_hit_column = (
    BLENDED_TRUE_HIT_COLUMN
    or find_first_column(blended, true_hit_candidates)
)
terminal_event_column = find_first_column(
    blended,
    terminal_event_candidates,
)

truth_lookup = (
    hmc_scene_summary[["scene_name", "true_hit"]]
    .assign(true_hit=lambda data: safe_numeric(data["true_hit"]))
    .dropna(subset=["true_hit"])
    .drop_duplicates("scene_name")
)

def _deduplicate_paths(paths):
    output = []
    seen = set()
    for value in paths:
        if value is None:
            continue
        path = Path(value).expanduser()
        try:
            key = str(path.resolve())
        except Exception:
            key = str(path)
        if key not in seen:
            output.append(path)
            seen.add(key)
    return output


def _limited_home_glob(patterns):
    matches = []
    for pattern in patterns:
        try:
            matches.extend(HOME.glob(pattern))
        except Exception:
            pass
    return matches


def resolve_blended_scene_directory():
    """Find the Experiment 2 scene JSON directory used by BlendedModel."""
    explicit_scenes = (
        None
        if BLENDED_EXP2_SCENE_DIR is None
        else Path(BLENDED_EXP2_SCENE_DIR).expanduser()
    )
    candidates = _deduplicate_paths(
        [
            explicit_scenes,
            Path.cwd() / "data" / "json" / "experiment2",
            Path.cwd().parent / "data" / "json" / "experiment2",
            PROJECT_ROOT / "data" / "json" / "experiment2",
        ]
        + _limited_home_glob([
            "*/data/json/experiment2",
            "*/*/data/json/experiment2",
            "*/*/*/data/json/experiment2",
        ])
    )
    candidates = [
        path
        for path in candidates
        if path.is_dir() and any(path.glob("*.json"))
    ]
    if not candidates:
        raise FileNotFoundError(
            "Could not automatically locate the Experiment 2 JSON scene "
            "directory. Set BLENDED_EXP2_SCENE_DIR in the settings cell to "
            "the folder containing the scene_*.json files."
        )
    return candidates[0]


def sample_blended_collision_outcomes():
    """Sample collision outcomes exactly as in the supplied source notebook."""
    cache_path = Path(BLENDED_COLLISION_CACHE_PATH)
    required_cache_columns = {
        "scene_name",
        "accuracy_sample_idx",
        "predicted_hit",
        "model_correct",
        "accuracy_definition",
    }

    if cache_path.exists() and not REBUILD_BLENDED_COLLISION_CACHE:
        cached = pd.read_csv(cache_path, low_memory=False)
        valid_definition = (
            "accuracy_definition" in cached.columns
            and cached["accuracy_definition"]
            .astype(str)
            .eq("collision_probability")
            .all()
        )
        if required_cache_columns.issubset(cached.columns) and valid_definition:
            cached["scene_name"] = cached["scene_name"].map(
                normalize_scene_name
            )
            print("Reading blended collision cache:", cache_path)
            return cached
        warnings.warn(
            f"Ignoring incompatible blended collision cache: {cache_path}"
        )

    exp2_scene_dir = resolve_blended_scene_directory()
    _, _, physics_engine_dir = _load_physics_dependencies(exp2_scene_dir)
    print("Embedded blended-model implementation: active")
    print("Physics engine directory:", physics_engine_dir)
    print("Blended Experiment 2 JSON directory:", exp2_scene_dir)
    print("Blended parameters:", BLENDED_MODEL_PARAMETERS)
    print("Accuracy samples per scene:", BLENDED_ACCURACY_SAMPLES)
    print(
        "Accuracy definition: mean collision probability, matching the "
        "supplied source notebook"
    )

    exp2_scenes = (
        common_scene_data.loc[
            common_scene_data["scene_name"]
            .astype(str)
            .str.startswith("scene", na=False),
            ["scene_name"],
        ]
        .drop_duplicates()
        .merge(
            truth_lookup,
            on="scene_name",
            how="left",
            validate="one_to_one",
        )
        .sort_values("scene_name")
        .reset_index(drop=True)
    )

    records = []
    for scene_number, row in exp2_scenes.iterrows():
        scene_name = str(row["scene_name"])
        true_hit = row.get("true_hit", np.nan)
        scene_path = exp2_scene_dir / f"{scene_name}.json"

        if not scene_path.exists():
            raise FileNotFoundError(
                f"Missing blended Experiment 2 scene JSON: {scene_path}"
            )

        scene_model = json_file_to_model(scene_path)
        blended_model = BlendedModel(dict(BLENDED_MODEL_PARAMETERS))

        for accuracy_sample_idx in range(int(BLENDED_ACCURACY_SAMPLES)):
            sample = blended_model.sample(scene_model)
            sample_dict = _model_dump_compat(sample)

            if "collision" not in sample_dict:
                raise KeyError(
                    "BlendedModel.sample output has no `collision` field. "
                    f"Available fields: {list(sample_dict)}"
                )

            collision = parse_binary_value(sample_dict["collision"])
            collision_numeric = safe_numeric(pd.Series([collision])).iloc[0]
            if not np.isfinite(collision_numeric):
                raise ValueError(
                    f"Could not parse collision for {scene_name}, "
                    f"sample {accuracy_sample_idx}: "
                    f"{sample_dict['collision']!r}"
                )
            collision = int(float(collision_numeric))

            # The source notebook names mean(collision) as model accuracy.
            # Therefore each binary collision sample is the sample-level
            # correctness value; true_hit is retained only as a diagnostic.
            records.append({
                "scene_name": scene_name,
                "accuracy_sample_idx": int(accuracy_sample_idx),
                "predicted_hit": collision,
                "true_hit": true_hit,
                "model_correct": collision,
                "accuracy_definition": "collision_probability",
                "simulation_ticks": sample_dict.get(
                    "simulation_ticks",
                    np.nan,
                ),
                "trajectory_length": sample_dict.get(
                    "trajectory_length",
                    np.nan,
                ),
            })

        completed = scene_number + 1
        if (
            completed == 1
            or completed % 5 == 0
            or completed == len(exp2_scenes)
        ):
            print(
                f"Blended accuracy sampling: "
                f"{completed}/{len(exp2_scenes)} scenes"
            )

    sampled = pd.DataFrame(records)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    sampled.to_csv(cache_path, index=False)
    print("Saved blended collision cache:", cache_path)
    return sampled


if correct_column is not None:
    blended["_direct_correct"] = blended[correct_column].map(
        parse_binary_value
    )
    accuracy_per_sample = (
        blended
        .groupby(group_columns, as_index=False)
        .agg(model_correct=("_direct_correct", last_nonmissing))
    )
    accuracy_per_sample["accuracy_sample_idx"] = (
        accuracy_per_sample["sample_idx"]
    )
    accuracy_method = f"explicit column `{correct_column}`"

elif predicted_hit_column is not None or terminal_event_column is not None:
    selected_prediction_column = (
        predicted_hit_column
        if predicted_hit_column is not None
        else terminal_event_column
    )
    blended["_predicted_hit"] = blended[
        selected_prediction_column
    ].map(parse_binary_value)

    predicted_per_sample = (
        blended
        .groupby(group_columns, as_index=False)
        .agg(predicted_hit=("_predicted_hit", last_nonmissing))
    )

    if true_hit_column is not None:
        blended["_true_hit"] = blended[true_hit_column].map(
            parse_binary_value
        )
        true_per_sample = (
            blended
            .groupby(group_columns, as_index=False)
            .agg(true_hit=("_true_hit", last_nonmissing))
        )
    else:
        true_per_sample = (
            blended[group_columns]
            .drop_duplicates()
            .merge(
                truth_lookup,
                on="scene_name",
                how="left",
                validate="many_to_one",
            )
        )

    accuracy_per_sample = predicted_per_sample.merge(
        true_per_sample,
        on=group_columns,
        how="left",
        validate="one_to_one",
    )
    accuracy_per_sample["model_correct"] = np.where(
        np.isfinite(safe_numeric(accuracy_per_sample["predicted_hit"]))
        & np.isfinite(safe_numeric(accuracy_per_sample["true_hit"])),
        (
            safe_numeric(accuracy_per_sample["predicted_hit"])
            == safe_numeric(accuracy_per_sample["true_hit"])
        ).astype(float),
        np.nan,
    )
    accuracy_per_sample["accuracy_sample_idx"] = (
        accuracy_per_sample["sample_idx"]
    )
    accuracy_method = (
        f"predicted outcome column `{selected_prediction_column}` "
        "compared with true_hit"
    )

else:
    # This is the expected branch for the supplied model_predictions.csv.
    accuracy_per_sample = sample_blended_collision_outcomes()
    accuracy_method = (
        "50 embedded BlendedModel samples per Experiment 2 scene; "
        "mean `collision` probability, matching the source notebook"
    )

print("Blended correctness method:", accuracy_method)

accuracy_per_sample["model_correct"] = safe_numeric(
    accuracy_per_sample["model_correct"]
)
if "predicted_hit" in accuracy_per_sample.columns:
    accuracy_per_sample["predicted_hit"] = safe_numeric(
        accuracy_per_sample["predicted_hit"]
    )
if "true_hit" in accuracy_per_sample.columns:
    accuracy_per_sample["true_hit"] = safe_numeric(
        accuracy_per_sample["true_hit"]
    )

accuracy_scene_summary = (
    accuracy_per_sample
    .groupby("scene_name", as_index=False)
    .agg(
        model_accuracy=("model_correct", "mean"),
        n_accuracy_samples=("model_correct", "count"),
    )
)

# For the boxplot, binarize the source notebook's collision-probability
# accuracy: a scene is labeled model-correct when at least half of the
# stochastic samples collide with the goal.
accuracy_scene_summary["model_correct"] = np.where(
    np.isfinite(accuracy_scene_summary["model_accuracy"]),
    (
        accuracy_scene_summary["model_accuracy"] >= 0.5
    ).astype(float),
    np.nan,
)

if "predicted_hit" in accuracy_per_sample.columns:
    predicted_summary = (
        accuracy_per_sample
        .groupby("scene_name", as_index=False)
        .agg(
            predicted_hit_probability=("predicted_hit", "mean"),
            true_hit=("true_hit", last_nonmissing),
        )
    )
    predicted_summary["majority_predicted_hit"] = (
        predicted_summary["predicted_hit_probability"] >= 0.5
    ).astype(float)
    accuracy_scene_summary = accuracy_scene_summary.merge(
        predicted_summary,
        on="scene_name",
        how="left",
        validate="one_to_one",
    )

blended_scene_summary = (
    runtime_scene_summary
    .merge(
        accuracy_scene_summary,
        on="scene_name",
        how="outer",
        validate="one_to_one",
    )
)
blended_scene_summary["mean_sample_accuracy"] = (
    blended_scene_summary["model_accuracy"]
)
blended_scene_summary["n_samples"] = (
    blended_scene_summary["n_runtime_samples"]
)
blended_scene_summary["model"] = "blended"

blended_scene_summary = (
    blended_scene_summary
    .merge(
        common_scene_data,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )
)

runtime_per_sample.to_csv(
    MODEL_OUTPUTS["blended"]
    / "tables"
    / "blended_runtime_sample_level_summary.csv",
    index=False,
)
accuracy_per_sample.to_csv(
    MODEL_OUTPUTS["blended"]
    / "tables"
    / "blended_accuracy_sample_level_summary.csv",
    index=False,
)
blended_scene_summary.to_csv(
    MODEL_OUTPUTS["blended"]
    / "tables"
    / "blended_scene_level_behavioral_summary.csv",
    index=False,
)

print("Blended scenes:", len(blended_scene_summary))
print(
    "Blended scenes with accuracy:",
    int(blended_scene_summary["model_accuracy"].notna().sum()),
)
display(blended_scene_summary.head())

In [ ]:

# ============================================================
# 6. Behavioral-analysis plotting and statistical functions
# ============================================================
def prepare_model_analysis_df(scene_summary_df):
    data = scene_summary_df.copy()
    required = [
        "scene_name",
        "model_run_time_ms",
        "model_correct",
        "model_accuracy",
        "true_simulation_time_ms",
        "human_response_time_ms",
        "mean_accuracy",
        "straight_path",
        "path_group",
    ]
    missing = [column for column in required if column not in data.columns]
    if missing:
        raise KeyError(f"Behavioral dataset is missing {missing}.")

    for column in [
        "model_run_time_ms",
        "model_correct",
        "model_accuracy",
        "true_simulation_time_ms",
        "human_response_time_ms",
        "mean_accuracy",
        "straight_path",
    ]:
        data[column] = safe_numeric(data[column])

    data["path_group"] = pd.Categorical(
        data["path_group"],
        categories=PATH_GROUP_ORDER,
        ordered=True,
    )
    return data




In [ ]:

# ============================================================
# 8. Eye-gaze trace-matching helper functions
# ============================================================
def point_to_segment_distance(point_x, point_y, start_x, start_y, end_x, end_y):
    point_x = np.asarray(point_x, dtype=float)
    point_y = np.asarray(point_y, dtype=float)
    segment_x = float(end_x - start_x)
    segment_y = float(end_y - start_y)
    denominator = segment_x ** 2 + segment_y ** 2
    if np.isclose(denominator, 0.0):
        proportion = np.zeros_like(point_x, dtype=float)
    else:
        proportion = (
            (point_x - start_x) * segment_x
            + (point_y - start_y) * segment_y
        ) / denominator
        proportion = np.clip(proportion, 0.0, 1.0)
    closest_x = start_x + proportion * segment_x
    closest_y = start_y + proportion * segment_y
    return np.sqrt((point_x - closest_x) ** 2 + (point_y - closest_y) ** 2)


def compute_min_trace_distances(gaze_scene, segment_scene):
    output = gaze_scene.copy().reset_index(drop=True)
    gaze_x = output["x_aligned"].to_numpy(dtype=float)
    gaze_y = output["y_aligned"].to_numpy(dtype=float)
    minimum_simulation = np.full(len(output), np.inf, dtype=float)
    minimum_abstraction = np.full(len(output), np.inf, dtype=float)

    for segment in segment_scene.itertuples(index=False):
        start_x = float(segment.start_x)
        start_y = float(segment.start_y)
        end_x = float(segment.end_x)
        end_y = float(segment.end_y)

        if USE_STRICT_SEGMENT_Y_RANGE:
            lower_y = min(start_y, end_y)
            upper_y = max(start_y, end_y)
            candidate_mask = (gaze_y >= lower_y) & (gaze_y <= upper_y)
        else:
            candidate_mask = np.ones(len(output), dtype=bool)

        if not candidate_mask.any():
            continue

        candidate_indices = np.flatnonzero(candidate_mask)
        distances = point_to_segment_distance(
            point_x=gaze_x[candidate_mask],
            point_y=gaze_y[candidate_mask],
            start_x=start_x,
            start_y=start_y,
            end_x=end_x,
            end_y=end_y,
        )
        if str(segment.reasoning_trace) == "simulation":
            minimum_simulation[candidate_indices] = np.minimum(
                minimum_simulation[candidate_indices], distances
            )
        elif str(segment.reasoning_trace) == "abstraction":
            minimum_abstraction[candidate_indices] = np.minimum(
                minimum_abstraction[candidate_indices], distances
            )

    output["min_dist_simulation"] = minimum_simulation
    output["min_dist_abstraction"] = minimum_abstraction
    output.loc[
        ~np.isfinite(output["min_dist_simulation"]), "min_dist_simulation"
    ] = np.nan
    output.loc[
        ~np.isfinite(output["min_dist_abstraction"]), "min_dist_abstraction"
    ] = np.nan
    return output


def add_threshold_labels(distance_data, threshold):
    output = distance_data.copy()
    output["threshold"] = float(threshold)
    simulation_distance = safe_numeric(output["min_dist_simulation"])
    abstraction_distance = safe_numeric(output["min_dist_abstraction"])
    in_simulation = np.isfinite(simulation_distance) & (simulation_distance <= threshold)
    in_abstraction = np.isfinite(abstraction_distance) & (abstraction_distance <= threshold)
    output["in_simulation_area"] = in_simulation
    output["in_abstraction_area"] = in_abstraction
    output["gaze_area_label"] = np.select(
        [
            in_simulation & ~in_abstraction,
            in_abstraction & ~in_simulation,
            in_simulation & in_abstraction,
        ],
        ["simulation", "abstraction", "overlap"],
        default="unknown",
    )
    return output


def build_distance_table(human_data, model_segments, model_name, output_dir):
    cache_path = Path(output_dir) / "tables" / "gaze_min_distances_to_model_traces.csv"
    if cache_path.exists() and not REBUILD_DISTANCE_CACHE:
        print("Reading existing distance cache:", cache_path)
        return pd.read_csv(cache_path, low_memory=False)

    scene_names = sorted(
        set(human_data["scene_name"].astype(str))
        .intersection(set(model_segments["scene_name"].astype(str)))
    )
    parts = []
    for scene_index, scene_name in enumerate(scene_names, start=1):
        if scene_index == 1 or scene_index % 20 == 0 or scene_index == len(scene_names):
            print(f"{model_name}: distance matching scene {scene_index}/{len(scene_names)}: {scene_name}")
        gaze_scene = human_data.loc[human_data["scene_name"].eq(scene_name)].copy()
        segment_scene = model_segments.loc[
            model_segments["scene_name"].eq(scene_name)
        ].copy()
        if len(gaze_scene) == 0 or len(segment_scene) == 0:
            continue
        parts.append(compute_min_trace_distances(gaze_scene, segment_scene))

    if not parts:
        raise RuntimeError(f"No gaze/model distance rows were created for {model_name}.")

    distance_df = pd.concat(parts, ignore_index=True)
    distance_df.to_csv(cache_path, index=False)
    print("Saved distance cache:", cache_path)
    return distance_df


def run_threshold_sweep(distance_df, model_name, output_dir):
    participant_tables = []
    label_count_tables = []

    for threshold in THRESHOLDS:
        labeled = add_threshold_labels(distance_df, threshold)
        label_counts = (
            labeled
            .groupby(["threshold", "gaze_area_label"], as_index=False)
            .size()
            .rename(columns={"size": "n_gaze_points"})
        )
        label_count_tables.append(label_counts)

        if INCLUDE_OVERLAP_IN_BOTH_POOLS:
            abstraction_pool = labeled.loc[
                labeled["gaze_area_label"].isin(["abstraction", "overlap"])
            ].copy()
            simulation_pool = labeled.loc[
                labeled["gaze_area_label"].isin(["simulation", "overlap"])
            ].copy()
        else:
            abstraction_pool = labeled.loc[
                labeled["gaze_area_label"].eq("abstraction")
            ].copy()
            simulation_pool = labeled.loc[
                labeled["gaze_area_label"].eq("simulation")
            ].copy()

        abstraction_participant = (
            abstraction_pool
            .groupby("subject_id", as_index=False)
            .agg(
                n_abstraction_pool_points=("gaze_id", "size"),
                prop_saccade_abstraction=("is_saccade", "mean"),
                prop_smooth_pursuit_abstraction=("is_smooth_pursuit", "mean"),
            )
        )
        simulation_participant = (
            simulation_pool
            .groupby("subject_id", as_index=False)
            .agg(
                n_simulation_pool_points=("gaze_id", "size"),
                prop_saccade_simulation=("is_saccade", "mean"),
                prop_smooth_pursuit_simulation=("is_smooth_pursuit", "mean"),
            )
        )
        participant_threshold = abstraction_participant.merge(
            simulation_participant,
            on="subject_id",
            how="outer",
            validate="one_to_one",
        )
        participant_threshold.insert(0, "threshold", float(threshold))
        participant_threshold[
            "diff_saccade_abstraction_minus_simulation"
        ] = (
            participant_threshold["prop_saccade_abstraction"]
            - participant_threshold["prop_saccade_simulation"]
        )
        participant_threshold[
            "diff_smooth_pursuit_simulation_minus_abstraction"
        ] = (
            participant_threshold["prop_smooth_pursuit_simulation"]
            - participant_threshold["prop_smooth_pursuit_abstraction"]
        )
        participant_tables.append(participant_threshold)

    participant_metrics = pd.concat(participant_tables, ignore_index=True)
    label_counts = pd.concat(label_count_tables, ignore_index=True)

    analysis_specs = [
        {
            "analysis": "saccade_proportion_abstraction_minus_simulation",
            "condition_a": "Abstraction",
            "condition_b": "Simulation",
            "condition_a_column": "prop_saccade_abstraction",
            "condition_b_column": "prop_saccade_simulation",
            "difference_column": "diff_saccade_abstraction_minus_simulation",
            "difference_definition": "abstraction - simulation",
            "title": "Saccade proportion: abstraction − simulation",
            "file_stub": "08_saccade_proportion_abstraction_minus_simulation_threshold_sweep",
        },
        {
            "analysis": "smooth_pursuit_proportion_simulation_minus_abstraction",
            "condition_a": "Simulation",
            "condition_b": "Abstraction",
            "condition_a_column": "prop_smooth_pursuit_simulation",
            "condition_b_column": "prop_smooth_pursuit_abstraction",
            "difference_column": "diff_smooth_pursuit_simulation_minus_abstraction",
            "difference_definition": "simulation - abstraction",
            "title": "Smooth-pursuit proportion: simulation − abstraction",
            "file_stub": "09_smooth_pursuit_proportion_simulation_minus_abstraction_threshold_sweep",
        },
    ]

    rows = []
    for threshold in THRESHOLDS:
        threshold_data = participant_metrics.loc[
            participant_metrics["threshold"].eq(threshold)
        ].copy()
        for spec_index, specification in enumerate(analysis_specs):
            columns = [
                specification["condition_a_column"],
                specification["condition_b_column"],
                specification["difference_column"],
            ]
            paired = threshold_data.replace([np.inf, -np.inf], np.nan).dropna(
                subset=columns
            )
            test_summary = paired_wilcoxon_summary(
                paired[specification["difference_column"]]
            )
            bootstrap_summary = bootstrap_median_summary(
                paired[specification["difference_column"]],
                seed=BOOTSTRAP_SEED + int(threshold) + len(rows),
            )
            rows.append({
                "model": model_name,
                "threshold": float(threshold),
                "analysis": specification["analysis"],
                "condition_a": specification["condition_a"],
                "condition_b": specification["condition_b"],
                "difference_definition": specification["difference_definition"],
                "median_condition_a": (
                    float(paired[specification["condition_a_column"]].median())
                    if len(paired) else np.nan
                ),
                "median_condition_b": (
                    float(paired[specification["condition_b_column"]].median())
                    if len(paired) else np.nan
                ),
                **bootstrap_summary,
                **test_summary,
            })

    threshold_summary = pd.DataFrame(rows)
    threshold_summary["bootstrap_p_value_holm_across_thresholds"] = np.nan
    threshold_summary["wilcoxon_p_value_holm_across_thresholds"] = np.nan

    # Keep explicit Wilcoxon-prefixed aliases while retaining the original
    # columns for backward compatibility with earlier notebook outputs.
    threshold_summary["wilcoxon_p_value_raw"] = threshold_summary["p_value_raw"]
    threshold_summary["wilcoxon_significance_raw"] = threshold_summary[
        "wilcoxon_p_value_raw"
    ].map(significance_label)

    for analysis_name, indices in threshold_summary.groupby("analysis").groups.items():
        index_list = list(indices)
        for raw_column, adjusted_column in [
            (
                "bootstrap_p_value_two_sided",
                "bootstrap_p_value_holm_across_thresholds",
            ),
            (
                "wilcoxon_p_value_raw",
                "wilcoxon_p_value_holm_across_thresholds",
            ),
        ]:
            raw_p = threshold_summary.loc[index_list, raw_column]
            valid = raw_p.notna()
            if valid.any():
                adjusted = multipletests(
                    raw_p.loc[valid].to_numpy(dtype=float),
                    alpha=0.05,
                    method="holm",
                )[1]
                threshold_summary.loc[
                    raw_p.loc[valid].index,
                    adjusted_column,
                ] = adjusted

    threshold_summary["bootstrap_significance_holm"] = threshold_summary[
        "bootstrap_p_value_holm_across_thresholds"
    ].map(significance_label)
    threshold_summary["wilcoxon_significance_holm"] = threshold_summary[
        "wilcoxon_p_value_holm_across_thresholds"
    ].map(significance_label)

    # Legacy alias retained so older downstream code still finds this column.
    threshold_summary["p_value_holm_across_thresholds"] = threshold_summary[
        "wilcoxon_p_value_holm_across_thresholds"
    ]

    for specification in analysis_specs:
        plot_data = (
            threshold_summary.loc[
                threshold_summary["analysis"].eq(specification["analysis"])
            ]
            .sort_values("threshold")
            .reset_index(drop=True)
        )
        x_values = plot_data["threshold"].to_numpy(dtype=float)
        median_values = plot_data["median_difference"].to_numpy(dtype=float)
        lower_values = plot_data[
            "bootstrap_95_ci_lower_median_difference"
        ].to_numpy(dtype=float)
        upper_values = plot_data[
            "bootstrap_95_ci_upper_median_difference"
        ].to_numpy(dtype=float)

        fig, ax = plt.subplots(figsize=(8.4, 6.2))
        ax.fill_between(
            x_values,
            lower_values,
            upper_values,
            color=ACCURACY_COLOR,
            alpha=THRESHOLD_CI_ALPHA,
            linewidth=0,
            zorder=1,
        )
        ax.plot(
            x_values,
            median_values,
            marker="o",
            markersize=7,
            color=ACCURACY_COLOR,
            linewidth=2.6,
            zorder=3,
        )
        ax.axhline(
            0,
            color="black",
            linestyle="--",
            linewidth=1.1,
            alpha=0.65,
            zorder=0,
        )
        finite_values = np.concatenate([
            median_values[np.isfinite(median_values)],
            lower_values[np.isfinite(lower_values)],
            upper_values[np.isfinite(upper_values)],
        ])
        if len(finite_values):
            y_range = finite_values.max() - finite_values.min()
            offset = 0.05 * y_range if y_range > 0 else 0.01
        else:
            offset = 0.01

        for row in plot_data.itertuples(index=False):
            # Plot significance is based on the bootstrap analysis of the median,
            # not on the Wilcoxon signed-rank test.
            label = row.bootstrap_significance
            if label != "" and np.isfinite(row.median_difference):
                ax.text(
                    row.threshold,
                    row.median_difference + offset,
                    label,
                    ha="center",
                    va="bottom",
                    fontsize=12,
                    fontweight="bold",
                )

        ax.set_title(specification["title"], pad=12)
        ax.set_xlabel("Trace-area radius threshold (px)")
        ax.set_ylabel("Within-subject difference")
        ax.set_xticks(THRESHOLDS)
        ax.grid(alpha=0.22, zorder=0)
        fig.tight_layout()
        save_figure(fig, output_dir, specification["file_stub"])
        plt.show()

    participant_metrics.to_csv(
        Path(output_dir) / "tables" / "participant_eye_movement_metrics_by_threshold.csv",
        index=False,
    )
    label_counts.to_csv(
        Path(output_dir) / "tables" / "gaze_area_label_counts_by_threshold.csv",
        index=False,
    )
    threshold_summary.to_csv(
        Path(output_dir)
        / "tables"
        / "within_subject_bootstrap_and_wilcoxon_by_threshold.csv",
        index=False,
    )
    # Retain the earlier filename for compatibility; it now contains both tests.
    threshold_summary.to_csv(
        Path(output_dir) / "tables" / "within_subject_wilcoxon_by_threshold.csv",
        index=False,
    )
    display(threshold_summary)
    return participant_metrics, threshold_summary


In [ ]:

# ============================================================
# 9. Build HMC and blended trace segments
# ============================================================
def build_hmc_segments(prediction_dir):
    prediction_files = sorted(Path(prediction_dir).glob("*__hybrid_predictions.csv"))
    if not prediction_files:
        raise FileNotFoundError(f"No HMC prediction files found under {prediction_dir}")

    parts = []
    for prediction_path in prediction_files:
        prediction = pd.read_csv(prediction_path, low_memory=False)
        prediction["prediction_file"] = str(prediction_path)
        if "scene" in prediction.columns:
            prediction["scene_name"] = prediction["scene"].map(normalize_scene_name)
        else:
            prediction["scene_name"] = normalize_scene_name(
                prediction_path.name.replace("__hybrid_predictions.csv", "").split("__")[-1]
            )
        parts.append(prediction)

    raw = pd.concat(parts, ignore_index=True)
    required = {"scene_name", "source", "frame", "segment_id", "x", "y"}
    missing = required - set(raw.columns)
    if missing:
        raise KeyError(f"HMC prediction files are missing: {sorted(missing)}")

    raw["scene_name"] = raw["scene_name"].map(normalize_scene_name)
    raw["reasoning_trace"] = normalize_trace_series(raw["source"])
    raw["frame"] = safe_numeric(raw["frame"])
    raw["source_segment_id"] = safe_numeric(raw["segment_id"])
    raw["source_segment_key"] = (
        raw["source_segment_id"].astype("string").fillna("__NA__")
    )
    raw["x_model"] = safe_numeric(raw["x"]) + HMC_X_SHIFT
    raw["y_model"] = safe_numeric(raw["y"]) + HMC_Y_SHIFT
    raw["original_row_order"] = np.arange(len(raw), dtype=int)
    raw = raw.loc[
        raw["reasoning_trace"].isin(["simulation", "abstraction"])
        & np.isfinite(raw["frame"])
        & np.isfinite(raw["x_model"])
        & np.isfinite(raw["y_model"])
    ].copy()

    # -----------------------------
    # HMC abstraction lines
    # -----------------------------
    abstraction_rows = raw.loc[raw["reasoning_trace"].eq("abstraction")].copy()
    if "abstraction_line_role" in abstraction_rows.columns:
        abstraction_rows["line_role"] = (
            abstraction_rows["abstraction_line_role"]
            .astype("string")
            .fillna("")
            .str.strip()
            .str.lower()
        )
    else:
        abstraction_rows["line_role"] = ""

    abstraction_segment_rows = []
    for (scene_name, segment_key), group in abstraction_rows.groupby(
        ["scene_name", "source_segment_key"],
        sort=False,
    ):
        group = group.sort_values(
            ["frame", "original_row_order"],
            kind="mergesort",
        )
        start_candidates = group.loc[group["line_role"].eq("start")]
        end_candidates = group.loc[group["line_role"].eq("endpoint")]
        start_row = start_candidates.iloc[0] if len(start_candidates) else group.iloc[0]
        end_row = end_candidates.iloc[-1] if len(end_candidates) else group.iloc[-1]

        original_start_frame = float(start_row["frame"])
        end_frame = float(end_row["frame"])
        if (
            not np.isfinite(original_start_frame)
            or not np.isfinite(end_frame)
            or end_frame <= original_start_frame
            or end_frame < HMC_TRACE_START_FRAME
        ):
            continue

        original_start_x = float(start_row["x_model"])
        original_start_y = float(start_row["y_model"])
        end_x = float(end_row["x_model"])
        end_y = float(end_row["y_model"])
        retained_start_frame = float(max(original_start_frame, HMC_TRACE_START_FRAME))
        fraction = (
            (retained_start_frame - original_start_frame)
            / (end_frame - original_start_frame)
        )
        fraction = float(np.clip(fraction, 0.0, 1.0))
        retained_start_x = original_start_x + fraction * (end_x - original_start_x)
        retained_start_y = original_start_y + fraction * (end_y - original_start_y)

        if end_frame <= retained_start_frame:
            continue
        if not (
            in_display_bounds([retained_start_x], [retained_start_y])[0]
            and in_display_bounds([end_x], [end_y])[0]
        ):
            continue

        abstraction_segment_rows.append({
            "scene_name": scene_name,
            "sample_idx": "__single__",
            "reasoning_trace": "abstraction",
            "source_segment_key": str(segment_key),
            "start_frame": retained_start_frame,
            "end_frame": end_frame,
            "start_x": retained_start_x,
            "start_y": retained_start_y,
            "end_x": end_x,
            "end_y": end_y,
            "was_clipped_at_frame_15": bool(original_start_frame < HMC_TRACE_START_FRAME),
        })

    abstraction_segments = pd.DataFrame(abstraction_segment_rows)

    # -----------------------------
    # HMC consecutive simulation segments
    # -----------------------------
    simulation_points = (
        raw.loc[
            raw["reasoning_trace"].eq("simulation")
            & raw["frame"].ge(HMC_TRACE_START_FRAME)
        ]
        .sort_values(
            ["scene_name", "source_segment_key", "frame", "original_row_order"],
            kind="mergesort",
        )
        .copy()
    )
    group = simulation_points.groupby(
        ["scene_name", "source_segment_key"],
        sort=False,
    )
    simulation_points["start_frame"] = group["frame"].shift(1)
    simulation_points["start_x"] = group["x_model"].shift(1)
    simulation_points["start_y"] = group["y_model"].shift(1)
    simulation_points["end_frame"] = simulation_points["frame"]
    simulation_points["end_x"] = simulation_points["x_model"]
    simulation_points["end_y"] = simulation_points["y_model"]

    valid = (
        np.isfinite(simulation_points["start_frame"])
        & (simulation_points["end_frame"] - simulation_points["start_frame"]).eq(1)
        & in_display_bounds(simulation_points["start_x"], simulation_points["start_y"])
        & in_display_bounds(simulation_points["end_x"], simulation_points["end_y"])
    )
    simulation_segments = simulation_points.loc[
        valid,
        [
            "scene_name", "source_segment_key", "start_frame", "end_frame",
            "start_x", "start_y", "end_x", "end_y",
        ],
    ].copy()
    simulation_segments["sample_idx"] = "__single__"
    simulation_segments["reasoning_trace"] = "simulation"
    simulation_segments["was_clipped_at_frame_15"] = False

    segment_columns = [
        "scene_name", "sample_idx", "reasoning_trace", "source_segment_key",
        "start_frame", "end_frame", "start_x", "start_y", "end_x", "end_y",
        "was_clipped_at_frame_15",
    ]
    segments = pd.concat(
        [
            abstraction_segments.reindex(columns=segment_columns),
            simulation_segments.reindex(columns=segment_columns),
        ],
        ignore_index=True,
    )
    segments = segments.sort_values(
        ["scene_name", "start_frame", "end_frame", "reasoning_trace", "source_segment_key"],
        kind="mergesort",
    ).reset_index(drop=True)
    segments.insert(2, "segment_id", segments.groupby("scene_name").cumcount() + 1)
    if len(segments) == 0:
        raise RuntimeError("No valid HMC segments were created.")
    return segments


def build_blended_segments(blended_dataframe):
    data = blended_dataframe.copy()
    data["_file_order"] = np.arange(len(data), dtype=int)
    data["scene_name"] = data["scene_name"].map(normalize_scene_name)
    data["sample_idx"] = data["sample_idx"].astype("string").fillna("__single__").str.strip()
    data["reasoning_trace"] = normalize_trace_series(data["segment_class"])
    data["x_model"] = safe_numeric(data["x"]) + BLENDED_X_SHIFT
    data["y_model"] = safe_numeric(data["y"]) + BLENDED_Y_SHIFT

    data = data.sort_values(
        ["scene_name", "sample_idx", "_file_order"],
        kind="mergesort",
    ).reset_index(drop=True)
    raw_group = data.groupby(["scene_name", "sample_idx"], sort=False)
    data["raw_row_id"] = raw_group.cumcount() + 1

    data = data.loc[
        data["reasoning_trace"].isin(["simulation", "abstraction"])
        & in_display_bounds(data["x_model"], data["y_model"])
    ].copy()

    group = data.groupby(["scene_name", "sample_idx"], sort=False)
    data["previous_raw_row_id"] = group["raw_row_id"].shift(1)
    data["contiguous"] = data["previous_raw_row_id"].eq(data["raw_row_id"] - 1)
    data["previous_trace"] = group["reasoning_trace"].shift(1).where(data["contiguous"])
    data["start_x"] = group["x_model"].shift(1).where(data["contiguous"])
    data["start_y"] = group["y_model"].shift(1).where(data["contiguous"])
    data["end_x"] = data["x_model"]
    data["end_y"] = data["y_model"]

    # Approved 20% simulation / 80% abstraction split for sim -> abstraction transitions.
    split_fraction = 0.20
    data["is_split"] = (
        data["reasoning_trace"].eq("abstraction")
        & data["previous_trace"].eq("simulation")
        & data["start_x"].notna()
        & data["start_y"].notna()
    )
    data["mid_x"] = data["start_x"] + split_fraction * (data["end_x"] - data["start_x"])
    data["mid_y"] = data["start_y"] + split_fraction * (data["end_y"] - data["start_y"])

    unchanged = data.loc[~data["is_split"]].copy()
    unchanged["row_order_2"] = unchanged["raw_row_id"].astype(float)

    simulation_piece = data.loc[data["is_split"]].copy()
    simulation_piece["reasoning_trace"] = "simulation"
    simulation_piece["end_x"] = simulation_piece["mid_x"]
    simulation_piece["end_y"] = simulation_piece["mid_y"]
    simulation_piece["row_order_2"] = simulation_piece["raw_row_id"].astype(float) + 0.1

    abstraction_piece = data.loc[data["is_split"]].copy()
    abstraction_piece["start_x"] = abstraction_piece["mid_x"]
    abstraction_piece["start_y"] = abstraction_piece["mid_y"]
    abstraction_piece["row_order_2"] = abstraction_piece["raw_row_id"].astype(float) + 0.2

    segments = pd.concat(
        [unchanged, simulation_piece, abstraction_piece],
        ignore_index=True,
    )
    segments = segments.loc[
        segments["reasoning_trace"].isin(["simulation", "abstraction"])
        & np.isfinite(segments["start_x"])
        & np.isfinite(segments["start_y"])
        & np.isfinite(segments["end_x"])
        & np.isfinite(segments["end_y"])
    ].copy()
    segments = segments.sort_values(
        ["scene_name", "sample_idx", "row_order_2"],
        kind="mergesort",
    ).reset_index(drop=True)
    segments.insert(2, "segment_id", np.arange(1, len(segments) + 1))
    return segments[
        [
            "scene_name", "sample_idx", "segment_id", "reasoning_trace",
            "start_x", "start_y", "end_x", "end_y",
        ]
    ].copy()


hmc_segments = build_hmc_segments(HMC_PREDICTION_DIR)
blended_segments = build_blended_segments(blended_raw)

hmc_segments.to_csv(
    MODEL_OUTPUTS["HMC"] / "tables" / "model_trace_segments.csv",
    index=False,
)
blended_segments.to_csv(
    MODEL_OUTPUTS["blended"] / "tables" / "model_trace_segments.csv",
    index=False,
)

print("HMC segments:", len(hmc_segments), "scenes:", hmc_segments["scene_name"].nunique())
print("Blended segments:", len(blended_segments), "scenes:", blended_segments["scene_name"].nunique())
display(hmc_segments.groupby("reasoning_trace").size().rename("n_segments"))
display(blended_segments.groupby("reasoning_trace").size().rename("n_segments"))


In [ ]:

# ============================================================
# 10. Prepare human gaze once, using y - 40 for both models
# ============================================================
human = pd.read_csv(
    CLASSIFIED_TRIAL_PATH,
    dtype={
        "subject_id": "string",
        "scene_name": "string",
        "segment_class": "string",
    },
    low_memory=False,
)

if "subject_id" not in human.columns and "id" in human.columns:
    human["subject_id"] = human["id"].astype("string")

required_human_columns = ["subject_id", "scene_name", "x", "y", "pupil", "segment_class"]
missing_human = [column for column in required_human_columns if column not in human.columns]
if missing_human:
    raise KeyError(
        f"{CLASSIFIED_TRIAL_PATH} is missing {missing_human}.\n"
        f"Available columns: {list(human.columns)}"
    )

TIME_COLUMN = find_first_column(
    human,
    ["raw_time", "normalized_time", "time", "timestamp"],
    required=True,
    label="human gaze time column",
)

human["subject_id"] = (
    human["subject_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"^(\d+)\.0$", r"\1", regex=True)
)
human["scene_name"] = human["scene_name"].map(normalize_scene_name)
human["x_raw"] = safe_numeric(human["x"])
human["y_raw"] = safe_numeric(human["y"])
human["x_aligned"] = human["x_raw"] + HUMAN_X_SHIFT
human["y_aligned"] = human["y_raw"] + HUMAN_Y_SHIFT
human["pupil"] = safe_numeric(human["pupil"])
human[TIME_COLUMN] = safe_numeric(human[TIME_COLUMN])
human["log_pupil"] = np.where(human["pupil"] > 0, np.log(human["pupil"]), np.nan)

human["segment_class_clean"] = (
    human["segment_class"]
    .astype("string")
    .str.strip()
    .str.lower()
    .str.replace("_", " ", regex=False)
    .str.replace("-", " ", regex=False)
    .str.replace(r"\s+", " ", regex=True)
)
human["is_saccade"] = human["segment_class_clean"].str.contains("saccade", na=False)
human["is_smooth_pursuit"] = (
    human["segment_class_clean"].str.contains("pursuit", na=False)
    | human["segment_class_clean"].str.contains("smooth", na=False)
)

human = human.loc[
    human["subject_id"].notna()
    & np.isfinite(human["x_aligned"])
    & np.isfinite(human["y_aligned"])
    & np.isfinite(human[TIME_COLUMN])
    & in_display_bounds(human["x_aligned"], human["y_aligned"])
].copy()

human = human.sort_values(
    ["subject_id", "scene_name", TIME_COLUMN],
    kind="mergesort",
).reset_index(drop=True)
human["gaze_id"] = (
    human.groupby(["subject_id", "scene_name"], sort=False).cumcount() + 1
)

print("Human gaze y shift applied:", HUMAN_Y_SHIFT)
print("Participants:", human["subject_id"].nunique())
print("Prepared human scenes:", human["scene_name"].nunique())
print("HMC eye scenes available:", len(set(hmc_segments["scene_name"]).intersection(set(human["scene_name"]))))
print("Blended eye scenes available:", len(set(blended_segments["scene_name"]).intersection(set(human["scene_name"]))))
print("Prepared gaze rows:", len(human))

human.to_csv(
    OUTPUT_ROOT / "human_gaze_prepared_y_minus_40.csv",
    index=False,
)


In [ ]:
# ============================================================
# Build only the three source tables required by Part IIIB
# ============================================================

for model_name, scene_data in [
    ("HMC", hmc_scene_summary),
    ("blended", blended_scene_summary),
]:
    prepared_behavior = prepare_model_analysis_df(scene_data)
    prepared_behavior.to_csv(
        MODEL_OUTPUTS[model_name] / "tables" / "complete_behavioral_analysis_data.csv",
        index=False,
    )

distance_tables = {}
for model_name, segments in [
    ("HMC", hmc_segments),
    ("blended", blended_segments),
]:
    model_human = human.loc[
        human["scene_name"].isin(segments["scene_name"].unique())
    ].copy()
    distance_tables[model_name] = build_distance_table(
        human_data=model_human,
        model_segments=segments,
        model_name=model_name,
        output_dir=MODEL_OUTPUTS[model_name],
    )

shared_eye_scenes = sorted(
    set(hmc_segments["scene_name"].astype(str))
    .intersection(set(blended_segments["scene_name"].astype(str)))
    .intersection(set(human["scene_name"].astype(str)))
)
if not shared_eye_scenes:
    raise RuntimeError("No scenes are shared by HMC, blended, and human gaze data.")

def make_model_comparison_participant_metrics(distance_df, model_name):
    shared_distance = distance_df.loc[
        distance_df["scene_name"].astype(str).isin(shared_eye_scenes)
    ].copy()

    threshold_tables = []
    for threshold in THRESHOLDS:
        labeled = add_threshold_labels(shared_distance, threshold)

        if INCLUDE_OVERLAP_IN_BOTH_POOLS:
            abstraction_pool = labeled.loc[
                labeled["gaze_area_label"].isin(["abstraction", "overlap"])
            ].copy()
            simulation_pool = labeled.loc[
                labeled["gaze_area_label"].isin(["simulation", "overlap"])
            ].copy()
        else:
            abstraction_pool = labeled.loc[
                labeled["gaze_area_label"].eq("abstraction")
            ].copy()
            simulation_pool = labeled.loc[
                labeled["gaze_area_label"].eq("simulation")
            ].copy()

        abstraction_participant = (
            abstraction_pool
            .groupby("subject_id", as_index=False)
            .agg(
                n_abstraction_pool_points=("gaze_id", "size"),
                prop_saccade_predicted_abstraction=("is_saccade", "mean"),
            )
        )
        simulation_participant = (
            simulation_pool
            .groupby("subject_id", as_index=False)
            .agg(
                n_simulation_pool_points=("gaze_id", "size"),
                prop_smooth_pursuit_predicted_simulation=("is_smooth_pursuit", "mean"),
            )
        )

        participant_threshold = abstraction_participant.merge(
            simulation_participant,
            on="subject_id",
            how="outer",
            validate="one_to_one",
        )
        participant_threshold.insert(0, "model", model_name)
        participant_threshold.insert(0, "threshold", float(threshold))
        threshold_tables.append(participant_threshold)

    return pd.concat(threshold_tables, ignore_index=True)

comparison_participant_metrics = pd.concat(
    [
        make_model_comparison_participant_metrics(distance_tables["HMC"], "HMC"),
        make_model_comparison_participant_metrics(distance_tables["blended"], "blended"),
    ],
    ignore_index=True,
)

comparison_participant_metrics.to_csv(
    COMPARISON_OUTPUT
    / "tables"
    / "participant_eye_movement_proportions_shared_scenes_by_model_threshold.csv",
    index=False,
)

print("Self-contained comparison inputs prepared under:", OUTPUT_ROOT)
print("Shared eye-gaze scenes:", len(shared_eye_scenes))


# Part IIIB — Meta-control model vs Blended model figures

In [ ]:
set_figure_prefix("03_comparison")

# Meta-control model vs Blended model

Clean notebook containing only the requested model-comparison analyses.

**Figure standard**
- Title: 19 pt
- Axis labels: 17 pt
- Tick labels: 14 pt
- Significance labels: 14 pt
- Full boxed axes for numeric plots
- Original source-notebook color scheme
- No model-specific color encoding
- No figures are saved in this notebook

The data filters, statistical choices, colors, and visual encodings follow the supplied model-comparison notebooks. Statistical results are kept off the behavioral figures. For the eye-tracking threshold sweeps, the original bootstrap-based `ns`, `*`, `**`, and `***` significance labels are retained.


In [ ]:
# ============================================================
# 0. Imports, source tables, and shared visual standard
# ============================================================

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import statsmodels.api as sm

from scipy.stats import pearsonr, mannwhitneyu
from statsmodels.stats.multitest import multipletests
from IPython.display import display


HOME = Path.home()
BASE_DIR = HOME / "Downloads" / "Meta-control"
OUTPUT_ROOT = BASE_DIR / "final_figure_analysis" / "model_comparison"

# Source tables rebuilt by the self-contained preparation section above.
META_CONTROL_BEHAVIOR_PATH = (
    OUTPUT_ROOT
    / "HMC"
    / "tables"
    / "complete_behavioral_analysis_data.csv"
)

BLENDED_BEHAVIOR_PATH = (
    OUTPUT_ROOT
    / "blended"
    / "tables"
    / "complete_behavioral_analysis_data.csv"
)

EYE_COMPARISON_PARTICIPANT_PATH = (
    OUTPUT_ROOT
    / "HMC_vs_blended"
    / "tables"
    / "participant_eye_movement_proportions_shared_scenes_by_model_threshold.csv"
)


# ------------------------------------------------------------
# Analysis settings preserved from the source notebook
# ------------------------------------------------------------
THRESHOLDS = np.arange(50, 121, 10, dtype=float)

N_BOOTSTRAP = 5000
BOOTSTRAP_SEED = 20260729

CI_ALPHA = 0.16
THRESHOLD_CI_ALPHA = 0.18


# ------------------------------------------------------------
# Standard project colors
# ------------------------------------------------------------
NON_STRAIGHT_COLOR = "#1C77C3"
STRAIGHT_COLOR = "#F39237"
FIT_COLOR = "#4D4D4D"
ACCURACY_COLOR = "#1C77C3"

# Original source-notebook colors
NO_STRAIGHT_COLOR = "#1C77C3"
STRAIGHT_COLOR = "#F39237"
ACCURACY_COLOR = "#1C77C3"
FIT_COLOR = "#4D4D4D"


# ------------------------------------------------------------
# Larger figure-text standard
# ------------------------------------------------------------
TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
LEGEND_SIZE = 14
BASE_FONT_SIZE = 14

plt.rcParams.update({
    "font.size": BASE_FONT_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "xtick.labelsize": TICK_SIZE,
    "ytick.labelsize": TICK_SIZE,
    "legend.fontsize": LEGEND_SIZE,
    "legend.title_fontsize": LEGEND_SIZE,
    "figure.dpi": 120,
})

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)


# ------------------------------------------------------------
# Input checks
# ------------------------------------------------------------
for path, label in [
    (
        META_CONTROL_BEHAVIOR_PATH,
        "Meta-control behavioral analysis table",
    ),
    (
        BLENDED_BEHAVIOR_PATH,
        "Blended behavioral analysis table",
    ),
    (
        EYE_COMPARISON_PARTICIPANT_PATH,
        "Meta-control vs Blended eye-comparison participant table",
    ),
]:
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {label}: {path}\n"
            "Run the self-contained comparison-data preparation section above."
        )


def safe_numeric(values):
    return pd.to_numeric(
        values,
        errors="coerce",
    )


def style_numeric_axis(ax):
    """Use the same boxed numeric-axis style across final figures."""
    ax.tick_params(
        axis="both",
        labelsize=TICK_SIZE,
    )

    for spine in ax.spines.values():
        spine.set_visible(True)
        spine.set_linewidth(1.1)
        spine.set_color("black")


def significance_label(p_value):
    """Original source-notebook significance-label convention."""
    if pd.isna(p_value):
        return "not testable"
    if p_value < 0.001:
        return "***"
    if p_value < 0.01:
        return "**"
    if p_value < 0.05:
        return "*"
    return "ns"


def fit_line_with_95_ci(
    x,
    y,
    n_line=300,
):
    """
    Source-notebook OLS line fit with 95% confidence interval
    for the fitted mean.
    """
    x = np.asarray(
        x,
        dtype=float,
    )
    y = np.asarray(
        y,
        dtype=float,
    )

    valid = (
        np.isfinite(x)
        & np.isfinite(y)
    )

    x = x[valid]
    y = y[valid]

    design = sm.add_constant(
        x,
        has_constant="add",
    )

    result = sm.OLS(
        y,
        design,
    ).fit()

    x_line = np.linspace(
        np.min(x),
        np.max(x),
        int(n_line),
    )

    prediction_design = sm.add_constant(
        x_line,
        has_constant="add",
    )

    prediction = (
        result
        .get_prediction(
            prediction_design
        )
        .summary_frame(
            alpha=0.05
        )
    )

    return {
        "result": result,
        "x_line": x_line,
        "mean": prediction[
            "mean"
        ].to_numpy(dtype=float),
        "ci_lower": prediction[
            "mean_ci_lower"
        ].to_numpy(dtype=float),
        "ci_upper": prediction[
            "mean_ci_upper"
        ].to_numpy(dtype=float),
    }


def bootstrap_median_summary(
    values,
    seed,
    n_bootstrap=N_BOOTSTRAP,
):
    """
    Source-notebook bootstrap test for a participant-level
    median difference.

    The two-sided p-value is the doubled smaller empirical tail
    probability of bootstrapped medians relative to zero, with
    a +1 correction.
    """
    values = safe_numeric(
        pd.Series(values)
    )

    values = (
        values.loc[
            np.isfinite(values)
        ]
        .to_numpy(dtype=float)
    )

    n_values = int(
        len(values)
    )

    if n_values == 0:
        return {
            "bootstrap_n_participants": 0,
            "median_difference": np.nan,
            "bootstrap_95_ci_lower": np.nan,
            "bootstrap_95_ci_upper": np.nan,
            "bootstrap_p_value_two_sided": np.nan,
            "bootstrap_significance": "not testable",
            "bootstrap_95_ci_excludes_zero": False,
        }

    sample_median = float(
        np.median(values)
    )

    rng = np.random.default_rng(
        int(seed)
    )

    sampled_indices = rng.integers(
        low=0,
        high=n_values,
        size=(
            int(n_bootstrap),
            n_values,
        ),
    )

    bootstrap_medians = np.median(
        values[
            sampled_indices
        ],
        axis=1,
    )

    lower, upper = np.quantile(
        bootstrap_medians,
        [
            0.025,
            0.975,
        ],
    )

    lower_tail = (
        np.count_nonzero(
            bootstrap_medians <= 0.0
        )
        + 1
    ) / (
        len(bootstrap_medians)
        + 1
    )

    upper_tail = (
        np.count_nonzero(
            bootstrap_medians >= 0.0
        )
        + 1
    ) / (
        len(bootstrap_medians)
        + 1
    )

    p_value = float(
        min(
            1.0,
            2.0
            * min(
                lower_tail,
                upper_tail,
            ),
        )
    )

    return {
        "bootstrap_n_participants": n_values,
        "median_difference": sample_median,
        "bootstrap_95_ci_lower": float(lower),
        "bootstrap_95_ci_upper": float(upper),
        "bootstrap_p_value_two_sided": p_value,
        "bootstrap_significance": significance_label(p_value),
        "bootstrap_95_ci_excludes_zero": bool(
            (lower > 0.0)
            or (upper < 0.0)
        ),
    }


## 1. Log–log human response time vs model run time

Two figures are produced: one for the Meta-control model and one for the Blended model. The title is **“Human response time vs model run time”** for both. Statistics are displayed separately.


In [ ]:
from scipy.stats import t as student_t

# ============================================================
# 1. Log–log human response time vs model run time
#    Meta-control model and Blended model
#
# Original source-notebook visual encoding:
# - non-straight-path scenes = blue
# - straight-path scenes = orange
# - pooled OLS fit = gray
# The two models use the SAME color scheme.
# ============================================================

def load_behavior_table(
    path,
    model_label,
):
    data = pd.read_csv(
        path,
        low_memory=False,
    )

    required = [
        "scene_name",
        "model_run_time_ms",
        "human_response_time_ms",
        "model_correct",
        "mean_accuracy",
        "straight_path",
    ]

    missing = [
        column
        for column in required
        if column not in data.columns
    ]

    if missing:
        raise KeyError(
            f"{path} is missing {missing}."
        )

    data = data.copy()

    data["scene_name"] = (
        data["scene_name"]
        .astype(str)
        .str.strip()
    )

    for column in [
        "model_run_time_ms",
        "human_response_time_ms",
        "model_correct",
        "mean_accuracy",
        "straight_path",
    ]:
        data[column] = safe_numeric(
            data[column]
        )

    data["model_label"] = model_label

    return data


meta_control_behavior = load_behavior_table(
    META_CONTROL_BEHAVIOR_PATH,
    "Meta-control model",
)

blended_behavior = load_behavior_table(
    BLENDED_BEHAVIOR_PATH,
    "Blended model",
)


def plot_loglog_human_vs_model_time(
    data,
    model_label,
):
    # Match the source notebook:
    # Experiment 1 scenes do not begin with "scene".
    exp1 = data.loc[
        ~data["scene_name"]
        .astype(str)
        .str.startswith(
            "scene",
            na=False,
        )
    ].copy()

    exp1 = (
        exp1
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=[
                "model_run_time_ms",
                "human_response_time_ms",
                "straight_path",
            ]
        )
    )

    exp1 = exp1.loc[
        (
            exp1["model_run_time_ms"] > 0
        )
        & (
            exp1["human_response_time_ms"] > 0
        )
    ].copy()

    exp1["log10_model_run_time_ms"] = np.log10(
        exp1["model_run_time_ms"]
    )

    exp1["log10_human_response_time_ms"] = np.log10(
        exp1["human_response_time_ms"]
    )

    x = exp1[
        "log10_model_run_time_ms"
    ].to_numpy(dtype=float)

    y = exp1[
        "log10_human_response_time_ms"
    ].to_numpy(dtype=float)

    correlation = pearsonr(
        x,
        y,
    )

    fit = fit_line_with_95_ci(
        x,
        y,
    )

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(7.6, 5.9)
    )

    # Original: color by path type, not by model.
    for path_value, color in [
        (0, NO_STRAIGHT_COLOR),
        (1, STRAIGHT_COLOR),
    ]:
        group = exp1.loc[
            exp1["straight_path"].eq(
                path_value
            )
        ]

        ax.scatter(
            group["log10_model_run_time_ms"],
            group["log10_human_response_time_ms"],
            s=58,
            alpha=0.80,
            color=color,
            edgecolors="black",
            linewidths=0.35,
            zorder=3,
        )

    # Original pooled OLS fit.
    ax.fill_between(
        fit["x_line"],
        fit["ci_lower"],
        fit["ci_upper"],
        color=FIT_COLOR,
        alpha=CI_ALPHA,
        linewidth=0,
        zorder=1,
    )

    ax.plot(
        fit["x_line"],
        fit["mean"],
        color=FIT_COLOR,
        linewidth=2.8,
        zorder=4,
    )

    ax.set_title(
        "Human response time vs model run time",
        fontsize=TITLE_SIZE,
        pad=14,
    )

    ax.set_xlabel(
        "Log10 model run time (ms)",
        fontsize=AXIS_LABEL_SIZE,
        labelpad=10,
    )

    ax.set_ylabel(
        "Log10 human response time (ms)",
        fontsize=AXIS_LABEL_SIZE,
        labelpad=10,
    )

    # Same path-type legend for both models.
    # --------------------------------------------------------
    # Legend
    # --------------------------------------------------------
    legend_handles = [
        Line2D(
            [0], [0],
            color=NO_STRAIGHT_COLOR,
            marker="o",
            linestyle="None",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=0.4,
            label="Non-straight path",
        ),
        Line2D(
            [0], [0],
            color=STRAIGHT_COLOR,
            marker="o",
            linestyle="None",
            markersize=7,
            markeredgecolor="black",
            markeredgewidth=0.4,
            label="Straight path",
        ),
        Line2D(
            [0], [0],
            color=FIT_COLOR,
            linewidth=2.8,
            label="All scenes",
        ),
    ]

    ax.legend(
        handles=legend_handles,
        loc="upper left",
        frameon=True,
        framealpha=0.92,
        facecolor="white",
        edgecolor="black",
        fontsize=LEGEND_SIZE,
    )

    ax.grid(
        alpha=0.22,
        zorder=0,
    )

    style_numeric_axis(ax)

    fig.tight_layout()
    plt.show()

    return {
        "model": model_label,
        "n_scenes": len(exp1),
        "pearson_r": float(
            correlation.statistic
        ),
        "ols_slope": float(
            fit["result"].params[1]
        ),
        "ols_slope_p": float(
            fit["result"].pvalues[1]
        ),
    }


print("Meta-control model")
meta_control_time_stats = (
    plot_loglog_human_vs_model_time(
        meta_control_behavior,
        "Meta-control model",
    )
)

print("Blended model")
blended_time_stats = (
    plot_loglog_human_vs_model_time(
        blended_behavior,
        "Blended model",
    )
)

time_statistics = pd.DataFrame([
    meta_control_time_stats,
    blended_time_stats,
])

print(
    "Log–log human response time vs model run time statistics"
)

display(
    time_statistics.style.format({
        "pearson_r": "{:.5f}",
        "ols_slope": "{:.6f}",
        "ols_slope_p": "{:.6g}",
    })
)
# ============================================================
# Williams test: Meta-control vs Blended log–log correlations
# ============================================================

def williams_test_dependent_overlapping(
    r_jk,
    r_jh,
    r_kh,
    n,
):
    """
    Williams' t test for two dependent correlations sharing variable j.

    Here:
      j = human response time
      k = Meta-control model run time
      h = Blended-model run time

    Therefore, the tested difference is:
      r(Meta-control, human) - r(Blended, human)

    Formula: standard Williams (1959) test used by cocor's
    `williams1959` option.
    """
    r_jk = float(r_jk)
    r_jh = float(r_jh)
    r_kh = float(r_kh)
    n = int(n)

    if n <= 3:
        raise ValueError(
            "Williams' test requires n > 3."
        )

    correlation_matrix_determinant = (
        1
        + 2 * r_jk * r_jh * r_kh
        - r_jk**2
        - r_jh**2
        - r_kh**2
    )

    if (
        correlation_matrix_determinant < 0
        and correlation_matrix_determinant > -1e-12
    ):
        correlation_matrix_determinant = 0.0

    if correlation_matrix_determinant < 0:
        raise ValueError(
            "The three correlations do not form a valid "
            "correlation matrix. "
            f"Determinant = "
            f"{correlation_matrix_determinant:.8g}"
        )

    mean_compared_correlation = (
        r_jk + r_jh
    ) / 2

    denominator = (
        2
        * ((n - 1) / (n - 3))
        * correlation_matrix_determinant
        + mean_compared_correlation**2
        * (1 - r_kh) ** 3
    )

    if denominator <= 0:
        raise ValueError(
            "Williams-test denominator is not positive: "
            f"{denominator:.8g}"
        )

    t_statistic = (
        (r_jk - r_jh)
        * np.sqrt(
            ((n - 1) * (1 + r_kh))
            / denominator
        )
    )

    degrees_of_freedom = n - 3

    p_value = 2 * student_t.sf(
        np.abs(t_statistic),
        df=degrees_of_freedom,
    )

    return {
        "williams_t": float(
            t_statistic
        ),
        "williams_df": int(
            degrees_of_freedom
        ),
        "williams_p_two_sided": float(
            p_value
        ),
    }


def prepare_shared_exp1_time_data(
    meta_control_data,
    blended_data,
):
    """
    Align the same positive, complete Experiment 1 scenes
    for the two overlapping correlations.
    """
    def prepare_one(
        data,
        model_prefix,
    ):
        prepared = data.loc[
            ~data["scene_name"]
            .astype(str)
            .str.startswith(
                "scene",
                na=False,
            ),
            [
                "scene_name",
                "model_run_time_ms",
                "human_response_time_ms",
            ],
        ].copy()

        prepared["scene_name"] = (
            prepared["scene_name"]
            .astype(str)
            .str.strip()
        )

        if prepared[
            "scene_name"
        ].duplicated().any():
            raise ValueError(
                f"Duplicate Experiment 1 scene names "
                f"for {model_prefix}."
            )

        return prepared.rename(
            columns={
                "model_run_time_ms": (
                    f"{model_prefix}_model_run_time_ms"
                ),
                "human_response_time_ms": (
                    "human_response_time_ms_"
                    f"{model_prefix}"
                ),
            }
        )

    meta_control = prepare_one(
        meta_control_data,
        "meta_control",
    )

    blended = prepare_one(
        blended_data,
        "blended",
    )

    shared = meta_control.merge(
        blended,
        on="scene_name",
        how="inner",
        validate="one_to_one",
    )

    human_meta_control = shared[
        "human_response_time_ms_meta_control"
    ].to_numpy(dtype=float)

    human_blended = shared[
        "human_response_time_ms_blended"
    ].to_numpy(dtype=float)

    finite_human = (
        np.isfinite(human_meta_control)
        & np.isfinite(human_blended)
    )

    if (
        finite_human.any()
        and not np.allclose(
            human_meta_control[
                finite_human
            ],
            human_blended[
                finite_human
            ],
            rtol=0,
            atol=1e-9,
        )
    ):
        raise ValueError(
            "Human response times differ between "
            "the Meta-control and Blended tables."
        )

    shared[
        "human_response_time_ms"
    ] = shared[
        "human_response_time_ms_meta_control"
    ]

    shared = shared.drop(
        columns=[
            "human_response_time_ms_meta_control",
            "human_response_time_ms_blended",
        ]
    )

    analysis_columns = [
        "meta_control_model_run_time_ms",
        "blended_model_run_time_ms",
        "human_response_time_ms",
    ]

    shared = (
        shared
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna(
            subset=analysis_columns
        )
        .loc[
            lambda data:
            (
                data[
                    "meta_control_model_run_time_ms"
                ] > 0
            )
            & (
                data[
                    "blended_model_run_time_ms"
                ] > 0
            )
            & (
                data[
                    "human_response_time_ms"
                ] > 0
            )
        ]
        .sort_values(
            "scene_name"
        )
        .reset_index(
            drop=True
        )
    )

    if len(shared) < 4:
        raise ValueError(
            f"Only {len(shared)} complete shared "
            "Experiment 1 scenes remain."
        )

    return shared


shared_time_comparison = (
    prepare_shared_exp1_time_data(
        meta_control_behavior,
        blended_behavior,
    )
)

log_meta_control_time = np.log10(
    shared_time_comparison[
        "meta_control_model_run_time_ms"
    ].to_numpy(dtype=float)
)

log_blended_time = np.log10(
    shared_time_comparison[
        "blended_model_run_time_ms"
    ].to_numpy(dtype=float)
)

log_human_time = np.log10(
    shared_time_comparison[
        "human_response_time_ms"
    ].to_numpy(dtype=float)
)

r_meta_control_human = float(
    pearsonr(
        log_meta_control_time,
        log_human_time,
    ).statistic
)

r_blended_human = float(
    pearsonr(
        log_blended_time,
        log_human_time,
    ).statistic
)

r_meta_control_blended = float(
    pearsonr(
        log_meta_control_time,
        log_blended_time,
    ).statistic
)

williams = (
    williams_test_dependent_overlapping(
        r_jk=r_meta_control_human,
        r_jh=r_blended_human,
        r_kh=r_meta_control_blended,
        n=len(
            shared_time_comparison
        ),
    )
)

delta_r = (
    r_meta_control_human
    - r_blended_human
)

print(
    "\nWilliams test for dependent overlapping "
    "log–log correlations"
)

print(
    f"Meta-control r = "
    f"{r_meta_control_human:.5f}; "
    f"Blended r = "
    f"{r_blended_human:.5f}; "
    f"delta r = {delta_r:.5f}"
)

print(
    f"Williams: "
    f"t({williams['williams_df']}) = "
    f"{williams['williams_t']:.3f}, "
    f"p = "
    f"{williams['williams_p_two_sided']:.5f}"
)


## 2. Human accuracy by model correctness

The source notebook treats model-correct and model-incorrect scenes as two independent groups, so it uses a **two-sided Mann–Whitney U test** rather than a paired Wilcoxon signed-rank test. The test results are shown separately from the figures.


In [ ]:
# ============================================================
# 2. Human accuracy by model correctness
#    Meta-control model and Blended model
#
# Original source-notebook visual encoding:
# both model figures use the same ACCURACY_COLOR.
# ============================================================

def plot_human_accuracy_by_model_correctness(
    data,
    model_label,
    random_seed=20260802,
):
    # Match the source notebook:
    # Experiment 2 scenes begin with "scene".
    exp2 = data.loc[
        data["scene_name"]
        .astype(str)
        .str.startswith(
            "scene",
            na=False,
        ),
        [
            "scene_name",
            "model_correct",
            "mean_accuracy",
        ],
    ].copy()

    exp2 = (
        exp2
        .replace(
            [np.inf, -np.inf],
            np.nan,
        )
        .dropna()
    )

    incorrect = exp2.loc[
        exp2["model_correct"].eq(0),
        "mean_accuracy",
    ].to_numpy(dtype=float)

    correct = exp2.loc[
        exp2["model_correct"].eq(1),
        "mean_accuracy",
    ].to_numpy(dtype=float)

    if (
        len(incorrect) == 0
        or len(correct) == 0
    ):
        raise ValueError(
            "Both model-correctness groups must contain scenes."
        )

    # Original source test:
    # model-correct and model-incorrect scenes are independent groups.
    test = mannwhitneyu(
        incorrect,
        correct,
        alternative="two-sided",
        method="auto",
    )

    incorrect_median = float(
        np.median(incorrect)
    )

    correct_median = float(
        np.median(correct)
    )

    median_difference = (
        correct_median
        - incorrect_median
    )

    rank_biserial = -(
        2.0
        * float(test.statistic)
        / (
            len(incorrect)
            * len(correct)
        )
        - 1.0
    )

    # --------------------------------------------------------
    # Figure
    # --------------------------------------------------------
    fig, ax = plt.subplots(
        figsize=(7.2, 6.5)
    )

    boxplot = ax.boxplot(
        [
            incorrect,
            correct,
        ],
        positions=[
            1,
            2,
        ],
        widths=0.55,
        patch_artist=True,
        showfliers=False,
        medianprops={
            "color": "black",
            "linewidth": 2.2,
        },
        boxprops={
            "edgecolor": "black",
            "linewidth": 1.2,
        },
        whiskerprops={
            "color": "black",
            "linewidth": 1.2,
        },
        capprops={
            "color": "black",
            "linewidth": 1.2,
        },
    )

    # Original: same blue for both Meta-control and Blended figures.
    for box in boxplot["boxes"]:
        box.set_facecolor(
            ACCURACY_COLOR
        )
        box.set_alpha(
            0.68
        )

    rng = np.random.default_rng(
        random_seed
    )

    ax.scatter(
        np.ones(
            len(incorrect)
        )
        + rng.normal(
            0,
            0.055,
            len(incorrect),
        ),
        incorrect,
        s=48,
        alpha=0.72,
        color=ACCURACY_COLOR,
        edgecolors="black",
        linewidths=0.35,
        zorder=3,
    )

    ax.scatter(
        np.full(
            len(correct),
            2.0,
        )
        + rng.normal(
            0,
            0.055,
            len(correct),
        ),
        correct,
        s=48,
        alpha=0.72,
        color=ACCURACY_COLOR,
        edgecolors="black",
        linewidths=0.35,
        zorder=3,
    )

    ax.set_title(
        "Human accuracy by model correctness",
        fontsize=TITLE_SIZE,
        pad=14,
    )

    ax.set_ylabel(
        "Human mean accuracy",
        fontsize=AXIS_LABEL_SIZE,
        labelpad=10,
    )

    ax.set_xticks(
        [1, 2]
    )

    ax.set_xticklabels(
        [
            "Model Incorrect",
            "Model Correct",
        ],
        fontsize=17,
    )

    ax.set_ylim(
        -0.05,
        1.05,
    )

    ax.grid(
        axis="y",
        alpha=0.22,
    )

    style_numeric_axis(
        ax
    )

    # Re-apply x tick-label size after style_numeric_axis(),
    # in case that helper resets tick font sizes.
    ax.tick_params(
        axis="x",
        labelsize=17,
    )

    # No statistical annotation on the figure.
    fig.tight_layout()
    plt.show()

    return {
        "model": model_label,
        "n_model_incorrect": len(
            incorrect
        ),
        "n_model_correct": len(
            correct
        ),
        "median_human_accuracy_model_incorrect": (
            incorrect_median
        ),
        "median_human_accuracy_model_correct": (
            correct_median
        ),
        "median_difference_correct_minus_incorrect": (
            median_difference
        ),
        "mann_whitney_U": float(
            test.statistic
        ),
        "p_value_two_sided": float(
            test.pvalue
        ),
        "rank_biserial_r_positive_means_correct_higher": (
            rank_biserial
        ),
    }


print("Meta-control model")
meta_control_correctness_stats = (
    plot_human_accuracy_by_model_correctness(
        meta_control_behavior,
        "Meta-control model",
    )
)

print("Blended model")
blended_correctness_stats = (
    plot_human_accuracy_by_model_correctness(
        blended_behavior,
        "Blended model",
    )
)

correctness_statistics = pd.DataFrame([
    meta_control_correctness_stats,
    blended_correctness_stats,
])

print(
    "Human accuracy by model correctness: "
    "two-sided Mann–Whitney U tests"
)

display(
    correctness_statistics.style.format({
        "median_human_accuracy_model_incorrect": "{:.5f}",
        "median_human_accuracy_model_correct": "{:.5f}",
        "median_difference_correct_minus_incorrect": "{:.5f}",
        "mann_whitney_U": "{:.3f}",
        "p_value_two_sided": "{:.6g}",
        "rank_biserial_r_positive_means_correct_higher": "{:.5f}",
    })
)

## 3. Direct Meta-control vs Blended eye-movement comparison

The participant-level values are computed on the **same shared scenes** for both models, matching the source notebook. At each trace-area threshold, the plotted quantity is the within-participant Meta-control − Blended difference. Bootstrap test results are displayed in a separate table; no significance markers are placed on the figures.


In [ ]:
# ============================================================
# 3. Meta-control vs Blended eye-movement comparison
#
# Saccade proportion:
# Meta-control abstraction − Blended abstraction
#
# Smooth-pursuit proportion:
# Meta-control simulation − Blended simulation
# ============================================================

comparison_participant_metrics = pd.read_csv(
    EYE_COMPARISON_PARTICIPANT_PATH,
    low_memory=False,
)

required_eye_columns = [
    "threshold",
    "subject_id",
    "model",
    "prop_saccade_predicted_abstraction",
    "prop_smooth_pursuit_predicted_simulation",
]

missing_eye = [
    column
    for column in required_eye_columns
    if column not in comparison_participant_metrics.columns
]

if missing_eye:
    raise KeyError(
        f"{EYE_COMPARISON_PARTICIPANT_PATH} "
        f"is missing {missing_eye}."
    )

comparison_participant_metrics[
    "threshold"
] = safe_numeric(
    comparison_participant_metrics[
        "threshold"
    ]
)

# Source table uses internal labels "HMC" and "blended".
# Keep those only for indexing; all visible labels below use
# "Meta-control model" and "Blended model".


comparison_specs = [
    {
        "analysis": (
            "saccade_proportion_"
            "meta_control_abstraction_minus_"
            "blended_abstraction"
        ),
        "value_column": (
            "prop_saccade_predicted_abstraction"
        ),
        "title": (
            "Saccade proportion:\n"
            "Meta-control abstraction − Blended abstraction"
        ),
    },
    {
        "analysis": (
            "smooth_pursuit_proportion_"
            "meta_control_simulation_minus_"
            "blended_simulation"
        ),
        "value_column": (
            "prop_smooth_pursuit_predicted_simulation"
        ),
        "title": (
            "Smooth-pursuit proportion:\n"
            "Meta-control simulation − Blended simulation"
        ),
    },
]


bootstrap_rows = []

for specification_index, specification in enumerate(
    comparison_specs
):
    wide = (
        comparison_participant_metrics
        .pivot_table(
            index=[
                "threshold",
                "subject_id",
            ],
            columns="model",
            values=specification[
                "value_column"
            ],
            aggfunc="mean",
        )
        .reset_index()
    )

    for required_model in [
        "HMC",
        "blended",
    ]:
        if required_model not in wide.columns:
            wide[
                required_model
            ] = np.nan

    wide[
        "difference_meta_control_minus_blended"
    ] = (
        wide["HMC"]
        - wide["blended"]
    )

    for threshold_index, threshold in enumerate(
        THRESHOLDS
    ):
        paired = (
            wide.loc[
                wide[
                    "threshold"
                ].eq(
                    threshold
                )
            ]
            .replace(
                [np.inf, -np.inf],
                np.nan,
            )
            .dropna(
                subset=[
                    "HMC",
                    "blended",
                    "difference_meta_control_minus_blended",
                ]
            )
        )

        bootstrap = bootstrap_median_summary(
            paired[
                "difference_meta_control_minus_blended"
            ],
            seed=(
                BOOTSTRAP_SEED
                + 10000
                + specification_index * 1000
                + threshold_index
            ),
        )

        bootstrap_rows.append({
            "analysis": specification[
                "analysis"
            ],
            "threshold": float(
                threshold
            ),
            "n_paired_participants": len(
                paired
            ),
            "median_meta_control": (
                float(
                    paired[
                        "HMC"
                    ].median()
                )
                if len(paired)
                else np.nan
            ),
            "median_blended": (
                float(
                    paired[
                        "blended"
                    ].median()
                )
                if len(paired)
                else np.nan
            ),
            **bootstrap,
        })


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)

bootstrap_results[
    "bootstrap_p_value_holm_across_thresholds"
] = np.nan

for analysis_name, indices in (
    bootstrap_results
    .groupby(
        "analysis"
    )
    .groups
    .items()
):
    index_list = list(
        indices
    )

    raw_p = bootstrap_results.loc[
        index_list,
        "bootstrap_p_value_two_sided",
    ]

    valid = raw_p.notna()

    if valid.any():
        adjusted = multipletests(
            raw_p.loc[
                valid
            ].to_numpy(
                dtype=float
            ),
            alpha=0.05,
            method="holm",
        )[1]

        bootstrap_results.loc[
            raw_p.loc[
                valid
            ].index,
            "bootstrap_p_value_holm_across_thresholds",
        ] = adjusted


# ------------------------------------------------------------
# Figures
# ------------------------------------------------------------
for specification in comparison_specs:

    plot_data = (
        bootstrap_results.loc[
            bootstrap_results[
                "analysis"
            ].eq(
                specification[
                    "analysis"
                ]
            )
        ]
        .sort_values(
            "threshold"
        )
        .reset_index(
            drop=True
        )
    )

    x_values = (
        plot_data[
            "threshold"
        ].to_numpy(
            dtype=float
        )
    )

    median_values = (
        plot_data[
            "median_difference"
        ].to_numpy(
            dtype=float
        )
    )

    lower_values = (
        plot_data[
            "bootstrap_95_ci_lower"
        ].to_numpy(
            dtype=float
        )
    )

    upper_values = (
        plot_data[
            "bootstrap_95_ci_upper"
        ].to_numpy(
            dtype=float
        )
    )

    fig, ax = plt.subplots(
        figsize=(8.4, 6.2)
    )

    ax.fill_between(
        x_values,
        lower_values,
        upper_values,
        color=ACCURACY_COLOR,
        alpha=THRESHOLD_CI_ALPHA,
        linewidth=0,
        zorder=1,
    )

    ax.plot(
        x_values,
        median_values,
        marker="o",
        markersize=7,
        color=ACCURACY_COLOR,
        linewidth=2.6,
        zorder=3,
    )

    ax.axhline(
        0,
        color="black",
        linestyle="--",
        linewidth=1.1,
        alpha=0.65,
        zorder=0,
    )

    # --------------------------------------------------------
    # Original bootstrap-based ns / * / ** / *** labels
    # --------------------------------------------------------
    finite_values = np.concatenate([
        median_values[
            np.isfinite(median_values)
        ],
        lower_values[
            np.isfinite(lower_values)
        ],
        upper_values[
            np.isfinite(upper_values)
        ],
    ])

    if len(finite_values):
        y_range = (
            finite_values.max()
            - finite_values.min()
        )
        annotation_offset = (
            0.05 * y_range
            if y_range > 0
            else 0.01
        )
    else:
        annotation_offset = 0.01

    for row in plot_data.itertuples(
        index=False
    ):
        # Match the original source notebook:
        # significance is based on the raw two-sided bootstrap
        # analysis of the participant-level median difference.
        label = row.bootstrap_significance

        if (
            label != ""
            and np.isfinite(
                row.median_difference
            )
        ):
            ax.text(
                row.threshold,
                row.median_difference
                + annotation_offset,
                label,
                ha="center",
                va="bottom",
                fontsize=TICK_SIZE,
                fontweight="bold",
            )

    ax.set_title(
        specification[
            "title"
        ],
        fontsize=TITLE_SIZE,
        pad=14,
    )

    ax.set_xlabel(
        "Trace-area radius threshold (px)",
        fontsize=AXIS_LABEL_SIZE,
        labelpad=10,
    )

    ax.set_ylabel(
        "Within-subject difference",
        fontsize=AXIS_LABEL_SIZE,
        labelpad=10,
    )

    ax.set_xticks(
        THRESHOLDS
    )

    ax.grid(
        alpha=0.22,
        zorder=0,
    )

    style_numeric_axis(
        ax
    )

    # Original bootstrap significance labels are retained on the figure.
    fig.tight_layout()
    plt.show()


# ------------------------------------------------------------
# Bootstrap test-results table
# ------------------------------------------------------------
# Keep only the requested result columns in the displayed table.
# Analysis and threshold are retained as row indices so each
# result remains identifiable without adding extra result columns.
bootstrap_table = bootstrap_results[
    [
        "analysis",
        "threshold",
        "median_meta_control",
        "median_blended",
        "median_difference",
        "bootstrap_95_ci_lower",
        "bootstrap_95_ci_upper",
        "bootstrap_p_value_two_sided",
    ]
].copy()

bootstrap_table[
    "bootstrap_95_ci"
] = bootstrap_table.apply(
    lambda row: (
        f"[{row['bootstrap_95_ci_lower']:.5f}, "
        f"{row['bootstrap_95_ci_upper']:.5f}]"
    ),
    axis=1,
)

bootstrap_table = (
    bootstrap_table[
        [
            "analysis",
            "threshold",
            "median_meta_control",
            "median_blended",
            "median_difference",
            "bootstrap_95_ci",
            "bootstrap_p_value_two_sided",
        ]
    ]
    .rename(
        columns={
            "median_meta_control": "Median Meta-control",
            "median_blended": "Median Blended",
            "median_difference": "Median difference",
            "bootstrap_95_ci": "Bootstrap 95% CI",
            "bootstrap_p_value_two_sided": "p",
        }
    )
    .set_index(
        [
            "analysis",
            "threshold",
        ]
    )
)

print(
    "Meta-control vs Blended eye-movement "
    "bootstrap test results"
)

display(
    bootstrap_table.style.format({
        "Median Meta-control": "{:.5f}",
        "Median Blended": "{:.5f}",
        "Median difference": "{:.5f}",
        "p": "{:.6g}",
    })
)


In [ ]:
# ============================================================
# Final figure manifest
# ============================================================
figure_manifest = pd.DataFrame(_final_figure_manifest)
EXPECTED_FIGURE_COUNT = 19

print(f"Figures saved this run: {len(figure_manifest)} (expected {EXPECTED_FIGURE_COUNT})")
print("Figure folder:", FINAL_FIG_DIR)

if len(figure_manifest) != EXPECTED_FIGURE_COUNT:
    warnings.warn(
        f"Expected {EXPECTED_FIGURE_COUNT} figures from the three source notebooks, "
        f"but recorded {len(figure_manifest)}. Check whether all plotting cells ran."
    )

display(figure_manifest)


In [ ]:
# ============================================================
# Cross-validated maximum-likelihood comparison:
# Which model better predicts saccade vs smooth-pursuit?
#
# Outcome:
#   1 = saccade
#   0 = smooth pursuit
#
# Predictor for each model:
#   1 = model-predicted abstraction
#   0 = model-predicted simulation
#
# Observation model:
#   logit P(saccade) =
#       participant fixed effect
#       + beta * predicted_abstraction
#
# Model parameters are fit by maximum likelihood on TRAINING scenes.
# Predictive performance is evaluated on HELD-OUT scenes.
#
# Larger held-out log-likelihood = better prediction.
# Positive HMC - Blended log-likelihood = favors Meta-control.
#
# Primary threshold: 120 px
# Other thresholds: robustness analysis
# ============================================================

import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm

from statsmodels.stats.multitest import multipletests
from IPython.display import display


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

PRIMARY_THRESHOLD = 120.0

CV_N_FOLDS = 10
CV_SEED = 20260821

N_RESAMPLES = 20000
INFERENCE_SEED = 20260822

EPS = 1e-9


# ------------------------------------------------------------
# Helpers
# ------------------------------------------------------------

def _to_binary(series):
    """
    Robustly convert bool / 0-1 / True-False columns to numeric 0/1.
    """
    numeric = pd.to_numeric(
        series,
        errors="coerce",
    )

    text = (
        series
        .astype(str)
        .str.strip()
        .str.lower()
    )

    mapped = text.map({
        "true": 1.0,
        "false": 0.0,
        "yes": 1.0,
        "no": 0.0,
    })

    return (
        numeric.astype(float)
        .where(
            numeric.notna(),
            mapped,
        )
    )


def prepare_common_eye_rows(
    threshold,
):
    """
    Construct exactly the SAME gaze observations for HMC and Blended.

    Keep only:
    - scenes available to both models,
    - human samples that are either saccades or smooth pursuits,
    - samples uniquely assigned to abstraction or simulation
      by BOTH models.

    Overlap and unknown samples are excluded because they do not provide
    a unique binary abstraction/simulation predictor.
    """

    key_columns = [
        "subject_id",
        "scene_name",
        "gaze_id",
    ]

    prepared = {}

    for model_name in [
        "HMC",
        "blended",
    ]:

        data = (
            distance_tables[
                model_name
            ]
            .copy()
        )

        data = data.loc[
            data[
                "scene_name"
            ]
            .astype(str)
            .isin(
                shared_eye_scenes
            )
        ].copy()

        data = add_threshold_labels(
            data,
            threshold,
        )

        # Standardize merge keys.
        data[
            "subject_id"
        ] = (
            data[
                "subject_id"
            ]
            .astype(str)
            .str.strip()
        )

        data[
            "scene_name"
        ] = (
            data[
                "scene_name"
            ]
            .astype(str)
            .str.strip()
        )

        data[
            "gaze_id"
        ] = (
            pd.to_numeric(
                data[
                    "gaze_id"
                ],
                errors="coerce",
            )
            .astype("Int64")
        )

        data[
            "is_saccade_num"
        ] = _to_binary(
            data[
                "is_saccade"
            ]
        )

        data[
            "is_smooth_num"
        ] = _to_binary(
            data[
                "is_smooth_pursuit"
            ]
        )

        # Each gaze observation should occur once per model.
        if data.duplicated(
            key_columns
        ).any():
            raise RuntimeError(
                f"Duplicate gaze keys found for {model_name}."
            )

        prepared[
            model_name
        ] = data[
            key_columns
            + [
                "gaze_area_label",
                "is_saccade_num",
                "is_smooth_num",
            ]
        ].copy()


    # --------------------------------------------------------
    # Pair exactly the same human gaze rows across models
    # --------------------------------------------------------

    common = (
        prepared[
            "HMC"
        ]
        .merge(
            prepared[
                "blended"
            ],
            on=key_columns,
            how="inner",
            suffixes=(
                "_hmc",
                "_blended",
            ),
            validate="one_to_one",
        )
    )


    # --------------------------------------------------------
    # Verify that human eye-movement labels agree
    # --------------------------------------------------------

    same_outcome = (
        common[
            "is_saccade_num_hmc"
        ].eq(
            common[
                "is_saccade_num_blended"
            ]
        )
        &
        common[
            "is_smooth_num_hmc"
        ].eq(
            common[
                "is_smooth_num_blended"
            ]
        )
    )

    common = common.loc[
        same_outcome
    ].copy()


    # --------------------------------------------------------
    # Restrict outcome to saccade vs smooth pursuit
    # --------------------------------------------------------

    is_saccade_only = (
        common[
            "is_saccade_num_hmc"
        ].eq(1)
        &
        common[
            "is_smooth_num_hmc"
        ].eq(0)
    )

    is_smooth_only = (
        common[
            "is_saccade_num_hmc"
        ].eq(0)
        &
        common[
            "is_smooth_num_hmc"
        ].eq(1)
    )

    common = common.loc[
        is_saccade_only
        |
        is_smooth_only
    ].copy()


    # --------------------------------------------------------
    # Require a unique phase prediction from BOTH models
    # --------------------------------------------------------

    valid_phases = [
        "abstraction",
        "simulation",
    ]

    common = common.loc[
        common[
            "gaze_area_label_hmc"
        ].isin(
            valid_phases
        )
        &
        common[
            "gaze_area_label_blended"
        ].isin(
            valid_phases
        )
    ].copy()


    # 1 = saccade
    # 0 = smooth pursuit
    common[
        "y_saccade"
    ] = (
        common[
            "is_saccade_num_hmc"
        ]
        .astype(int)
    )

    return (
        common
        .reset_index(
            drop=True
        )
    )


def make_design_matrix(
    data,
    phase_column,
    subject_levels,
):
    """
    Design:
        intercept
        + abstraction indicator
        + participant fixed effects

    Same number of free parameters for HMC and Blended.
    """

    abstraction = (
        data[
            phase_column
        ]
        .eq(
            "abstraction"
        )
        .astype(float)
        .to_numpy()
    )

    columns = [
        np.ones(
            len(data),
            dtype=float,
        ),
        abstraction,
    ]

    # Drop the first participant as reference.
    for subject in subject_levels[
        1:
    ]:
        columns.append(
            data[
                "subject_id"
            ]
            .eq(subject)
            .astype(float)
            .to_numpy()
        )

    return np.column_stack(
        columns
    )


def fit_mle_and_predict(
    train,
    test,
    phase_column,
):
    """
    Bernoulli logistic regression fit by maximum likelihood.
    """

    subject_levels = sorted(
        train[
            "subject_id"
        ]
        .astype(str)
        .unique()
    )

    unseen_test_subjects = (
        set(
            test[
                "subject_id"
            ]
            .astype(str)
        )
        - set(
            subject_levels
        )
    )

    if unseen_test_subjects:
        raise RuntimeError(
            "Some held-out rows contain participants absent "
            "from the training data: "
            f"{sorted(unseen_test_subjects)}"
        )

    X_train = make_design_matrix(
        train,
        phase_column,
        subject_levels,
    )

    X_test = make_design_matrix(
        test,
        phase_column,
        subject_levels,
    )

    y_train = (
        train[
            "y_saccade"
        ]
        .to_numpy(
            dtype=float
        )
    )

    if len(
        np.unique(
            y_train
        )
    ) < 2:
        raise RuntimeError(
            "Training fold contains only one eye-movement class."
        )

    with warnings.catch_warnings():
        warnings.simplefilter(
            "ignore"
        )

        fit = sm.GLM(
            y_train,
            X_train,
            family=sm.families.Binomial(),
        ).fit(
            maxiter=200
        )

    probability_saccade = (
        fit.predict(
            X_test
        )
    )

    probability_saccade = np.clip(
        np.asarray(
            probability_saccade,
            dtype=float,
        ),
        EPS,
        1.0 - EPS,
    )

    return probability_saccade


def cross_validated_likelihood(
    data,
    n_folds=CV_N_FOLDS,
    seed=CV_SEED,
):
    """
    Scene-grouped cross-validation.

    Every gaze sample from a held-out scene stays out of training.
    HMC and Blended use exactly the same folds and observations.
    """

    scenes = np.array(
        sorted(
            data[
                "scene_name"
            ].unique()
        ),
        dtype=object,
    )

    if len(
        scenes
    ) < 2:
        raise RuntimeError(
            "Need at least two scenes for cross-validation."
        )

    n_folds = min(
        int(
            n_folds
        ),
        len(
            scenes
        ),
    )

    rng = np.random.default_rng(
        int(
            seed
        )
    )

    rng.shuffle(
        scenes
    )

    fold_scene_sets = np.array_split(
        scenes,
        n_folds,
    )

    prediction_parts = []


    for fold_index, test_scenes in enumerate(
        fold_scene_sets,
        start=1,
    ):

        test_mask = (
            data[
                "scene_name"
            ]
            .isin(
                test_scenes
            )
        )

        train = data.loc[
            ~test_mask
        ].copy()

        test = data.loc[
            test_mask
        ].copy()

        if (
            train[
                "gaze_area_label_hmc"
            ].nunique()
            < 2
            or
            train[
                "gaze_area_label_blended"
            ].nunique()
            < 2
        ):
            raise RuntimeError(
                f"Fold {fold_index} lacks both phases "
                "for at least one model."
            )


        # ----------------------------------------------------
        # Maximum-likelihood training
        # ----------------------------------------------------

        p_hmc = fit_mle_and_predict(
            train,
            test,
            phase_column=(
                "gaze_area_label_hmc"
            ),
        )

        p_blended = fit_mle_and_predict(
            train,
            test,
            phase_column=(
                "gaze_area_label_blended"
            ),
        )


        # ----------------------------------------------------
        # Held-out Bernoulli log-likelihood
        # ----------------------------------------------------

        y_test = (
            test[
                "y_saccade"
            ]
            .to_numpy(
                dtype=float
            )
        )

        loglik_hmc = (
            y_test
            * np.log(
                p_hmc
            )
            +
            (
                1.0
                - y_test
            )
            * np.log(
                1.0
                - p_hmc
            )
        )

        loglik_blended = (
            y_test
            * np.log(
                p_blended
            )
            +
            (
                1.0
                - y_test
            )
            * np.log(
                1.0
                - p_blended
            )
        )

        held_out = test[
            [
                "subject_id",
                "scene_name",
                "gaze_id",
                "y_saccade",
                "gaze_area_label_hmc",
                "gaze_area_label_blended",
            ]
        ].copy()

        held_out[
            "fold"
        ] = fold_index

        held_out[
            "p_saccade_hmc"
        ] = p_hmc

        held_out[
            "p_saccade_blended"
        ] = p_blended

        held_out[
            "loglik_hmc"
        ] = loglik_hmc

        held_out[
            "loglik_blended"
        ] = loglik_blended

        held_out[
            "delta_loglik_hmc_minus_blended"
        ] = (
            loglik_hmc
            - loglik_blended
        )

        prediction_parts.append(
            held_out
        )


    return pd.concat(
        prediction_parts,
        ignore_index=True,
    )


def scene_level_likelihood_inference(
    cv_predictions,
    seed,
    n_resamples=N_RESAMPLES,
):
    """
    Inference is done at the SCENE level rather than treating
    every gaze sample as independent.

    For each scene:
        mean held-out LL(HMC) - mean held-out LL(Blended)

    Then:
    - bootstrap scenes for a 95% CI
    - paired sign-flip randomization test for H0: difference = 0
    """

    scene_summary = (
        cv_predictions
        .groupby(
            "scene_name",
            as_index=False,
        )
        .agg(
            mean_loglik_hmc=(
                "loglik_hmc",
                "mean",
            ),
            mean_loglik_blended=(
                "loglik_blended",
                "mean",
            ),
            mean_delta_loglik=(
                "delta_loglik_hmc_minus_blended",
                "mean",
            ),
            n_gaze_samples=(
                "gaze_id",
                "size",
            ),
        )
    )

    differences = (
        scene_summary[
            "mean_delta_loglik"
        ]
        .to_numpy(
            dtype=float
        )
    )

    observed = float(
        np.mean(
            differences
        )
    )

    rng = np.random.default_rng(
        int(
            seed
        )
    )


    # --------------------------------------------------------
    # Scene bootstrap CI
    # --------------------------------------------------------

    bootstrap_indices = rng.integers(
        0,
        len(
            differences
        ),
        size=(
            int(
                n_resamples
            ),
            len(
                differences
            ),
        ),
    )

    bootstrap_means = (
        differences[
            bootstrap_indices
        ]
        .mean(
            axis=1
        )
    )

    ci_lower, ci_upper = np.quantile(
        bootstrap_means,
        [
            0.025,
            0.975,
        ],
    )


    # --------------------------------------------------------
    # Paired scene-level sign-flip test
    # --------------------------------------------------------

    random_signs = rng.choice(
        np.array(
            [
                -1.0,
                1.0,
            ]
        ),
        size=(
            int(
                n_resamples
            ),
            len(
                differences
            ),
        ),
    )

    null_means = (
        differences[
            None,
            :
        ]
        * random_signs
    ).mean(
        axis=1
    )

    p_value = (
        np.count_nonzero(
            np.abs(
                null_means
            )
            >=
            abs(
                observed
            )
        )
        + 1
    ) / (
        len(
            null_means
        )
        + 1
    )


    return {
        "n_scenes": int(
            len(
                differences
            )
        ),
        "scene_mean_delta_loglik": observed,
        "scene_bootstrap_95_ci_lower": float(
            ci_lower
        ),
        "scene_bootstrap_95_ci_upper": float(
            ci_upper
        ),
        "scene_signflip_p": float(
            p_value
        ),
    }


# ============================================================
# Run analysis across thresholds
# ============================================================

summary_rows = []
all_cv_predictions = {}

for threshold_index, threshold in enumerate(
    THRESHOLDS
):

    common_eye = prepare_common_eye_rows(
        threshold
    )

    if len(
        common_eye
    ) == 0:
        continue

    cv_predictions = cross_validated_likelihood(
        common_eye,
        n_folds=CV_N_FOLDS,
        seed=CV_SEED,
    )

    all_cv_predictions[
        float(
            threshold
        )
    ] = cv_predictions

    inference = scene_level_likelihood_inference(
        cv_predictions,
        seed=(
            INFERENCE_SEED
            + threshold_index
        ),
    )


    # --------------------------------------------------------
    # Standard predictive scores
    # --------------------------------------------------------

    mean_loglik_hmc = float(
        cv_predictions[
            "loglik_hmc"
        ].mean()
    )

    mean_loglik_blended = float(
        cv_predictions[
            "loglik_blended"
        ].mean()
    )

    mean_delta_loglik = (
        mean_loglik_hmc
        - mean_loglik_blended
    )

    # Negative mean log-likelihood = log loss.
    # LOWER log loss is better.
    log_loss_hmc = -mean_loglik_hmc
    log_loss_blended = -mean_loglik_blended


    summary_rows.append({
        "threshold": float(
            threshold
        ),

        "n_gaze_samples": int(
            len(
                cv_predictions
            )
        ),

        "n_scenes": int(
            cv_predictions[
                "scene_name"
            ].nunique()
        ),

        "mean_cv_loglik_hmc": (
            mean_loglik_hmc
        ),

        "mean_cv_loglik_blended": (
            mean_loglik_blended
        ),

        "delta_cv_loglik_hmc_minus_blended": (
            mean_delta_loglik
        ),

        "cv_log_loss_hmc": (
            log_loss_hmc
        ),

        "cv_log_loss_blended": (
            log_loss_blended
        ),

        **inference,
    })


cv_predictive_comparison = pd.DataFrame(
    summary_rows
).sort_values(
    "threshold"
).reset_index(
    drop=True
)


# ============================================================
# Holm correction across threshold sweep
# ============================================================

if len(
    cv_predictive_comparison
):
    cv_predictive_comparison[
        "scene_signflip_p_holm"
    ] = multipletests(
        cv_predictive_comparison[
            "scene_signflip_p"
        ].to_numpy(
            dtype=float
        ),
        alpha=0.05,
        method="holm",
    )[1]


# ============================================================
# Winner
# ============================================================

def model_winner(
    delta,
):
    if delta > 0:
        return "Meta-control"
    if delta < 0:
        return "Blended"
    return "Tie"


cv_predictive_comparison[
    "better_predictive_model"
] = (
    cv_predictive_comparison[
        "scene_mean_delta_loglik"
    ]
    .map(
        model_winner
    )
)


# ============================================================
# Display compact result table
# ============================================================

display_columns = [
    "threshold",
    "n_gaze_samples",
    "n_scenes",
    "mean_cv_loglik_hmc",
    "mean_cv_loglik_blended",
    "delta_cv_loglik_hmc_minus_blended",
    "scene_mean_delta_loglik",
    "scene_bootstrap_95_ci_lower",
    "scene_bootstrap_95_ci_upper",
    "scene_signflip_p",
    "scene_signflip_p_holm",
    "better_predictive_model",
]

print(
    "Cross-validated maximum-likelihood comparison"
)
print(
    "Higher log-likelihood is better; "
    "positive HMC − Blended values favor Meta-control."
)

display(
    cv_predictive_comparison[
        display_columns
    ].style.format({
        "threshold": "{:.0f}",
        "mean_cv_loglik_hmc": "{:.6f}",
        "mean_cv_loglik_blended": "{:.6f}",
        "delta_cv_loglik_hmc_minus_blended": "{:.6f}",
        "scene_mean_delta_loglik": "{:.6f}",
        "scene_bootstrap_95_ci_lower": "{:.6f}",
        "scene_bootstrap_95_ci_upper": "{:.6f}",
        "scene_signflip_p": "{:.6g}",
        "scene_signflip_p_holm": "{:.6g}",
    })
)


# ============================================================
# Primary 120-px result
# ============================================================

primary_result = (
    cv_predictive_comparison
    .loc[
        cv_predictive_comparison[
            "threshold"
        ].eq(
            PRIMARY_THRESHOLD
        )
    ]
)

if len(
    primary_result
) != 1:
    raise RuntimeError(
        f"Expected exactly one result for "
        f"{PRIMARY_THRESHOLD:.0f}px."
    )

primary_result = primary_result.iloc[
    0
]

print(
    f"\nPRIMARY ANALYSIS ({PRIMARY_THRESHOLD:.0f}px)"
)

print(
    "Meta-control mean held-out log-likelihood per gaze: "
    f"{primary_result['mean_cv_loglik_hmc']:.6f}"
)

print(
    "Blended mean held-out log-likelihood per gaze: "
    f"{primary_result['mean_cv_loglik_blended']:.6f}"
)

print(
    "HMC − Blended scene-mean held-out log-likelihood: "
    f"{primary_result['scene_mean_delta_loglik']:.6f}"
)

print(
    "95% scene-bootstrap CI: "
    f"[{primary_result['scene_bootstrap_95_ci_lower']:.6f}, "
    f"{primary_result['scene_bootstrap_95_ci_upper']:.6f}]"
)

print(
    "Scene-level sign-flip p = "
    f"{primary_result['scene_signflip_p']:.6g}"
)

print(
    "Better predictive model: "
    f"{primary_result['better_predictive_model']}"
)

In [ ]:
# ============================================================
# Threshold sweep:
# Meta-control − Blended predictive performance
# with significance stars
# ============================================================

plot_data = (
    cv_predictive_comparison
    .sort_values("threshold")
    .reset_index(drop=True)
)

x_values = (
    plot_data["threshold"]
    .to_numpy(dtype=float)
)

mean_values = (
    plot_data["scene_mean_delta_loglik"]
    .to_numpy(dtype=float)
)

lower_values = (
    plot_data["scene_bootstrap_95_ci_lower"]
    .to_numpy(dtype=float)
)

upper_values = (
    plot_data["scene_bootstrap_95_ci_upper"]
    .to_numpy(dtype=float)
)


# ------------------------------------------------------------
# Significance helper
# ------------------------------------------------------------

def significance_stars(p):
    if not np.isfinite(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    return "ns"


# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(8.4, 6.2)
)


# 95% bootstrap CI
ax.fill_between(
    x_values,
    lower_values,
    upper_values,
    color=ACCURACY_COLOR,
    alpha=THRESHOLD_CI_ALPHA,
    linewidth=0,
    zorder=1,
)


# Mean HMC − Blended predictive advantage
ax.plot(
    x_values,
    mean_values,
    marker="o",
    markersize=7,
    color=ACCURACY_COLOR,
    linewidth=2.6,
    zorder=3,
)


# Equal predictive performance
ax.axhline(
    0,
    color="black",
    linestyle="--",
    linewidth=1.1,
    alpha=0.65,
    zorder=0,
)


# ------------------------------------------------------------
# Significance stars
# ------------------------------------------------------------

finite_values = np.concatenate([
    mean_values[np.isfinite(mean_values)],
    lower_values[np.isfinite(lower_values)],
    upper_values[np.isfinite(upper_values)],
])

if len(finite_values):
    y_range = (
        finite_values.max()
        - finite_values.min()
    )
    offset = (
        0.05 * y_range
        if y_range > 0
        else 0.001
    )
else:
    offset = 0.001


for row in plot_data.itertuples(index=False):

    stars = significance_stars(
        row.scene_signflip_p
    )

    if (
        stars != ""
        and np.isfinite(
            row.scene_mean_delta_loglik
        )
    ):
        ax.text(
            row.threshold,
            row.scene_mean_delta_loglik + offset,
            stars,
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            zorder=5,
        )


# ------------------------------------------------------------
# Labels and styling
# ------------------------------------------------------------

ax.set_title(
    "Meta-control vs blended predictive performance",
    pad=12,
)

ax.set_xlabel(
    "Trace-area radius threshold (px)"
)

ax.set_ylabel(
    "Held-out log-likelihood difference\n"
    "(Meta-control − blended)"
)

ax.set_xticks(
    THRESHOLDS
)

ax.grid(
    alpha=0.22,
    zorder=0,
)

add_standard_text_sizes(
    ax
)

fig.tight_layout()

plt.show()

In [ ]:
save_figure(
    fig,
    BASE_DIR / "final_figures",
    "meta_control_vs_blended_cv_loglik_threshold_sweep",
)

In [ ]:
# ============================================================
# Observed vs predicted saccade probability
# Threshold = 120 px
#
# Left half:  Meta-control
# Right half: Blended
#
# Visual grammar:
#   Meta-control observed  = light blue, solid
#   Meta-control predicted = dark blue, dashed
#
#   Blended observed       = light gray, solid
#   Blended predicted      = dark gray, dashed
#
# No figure saving
# ============================================================

PRIMARY_THRESHOLD = 120.0


# ============================================================
# Figure formatting
# ============================================================
FIGSIZE = (9.5, 6.5)

FIGURE_TITLE_SIZE = 19
AXIS_LABEL_SIZE = 17
TICK_SIZE = 14
MODEL_LABEL_SIZE = 16
LEGEND_TEXT_SIZE = 14

TITLE_PAD = 12

MAIN_LINE_WIDTH = 3.4
ERROR_LINE_WIDTH = 1.7
MARKER_SIZE = 9


# ============================================================
# Colors
# ============================================================

# Meta-control = blue family
HMC_OBSERVED_COLOR = "#9CC9EE"
HMC_PREDICTED_COLOR = "#1C77C3"

# Blended = gray family
BLENDED_OBSERVED_COLOR = "#BDBDBD"
BLENDED_PREDICTED_COLOR = "#4D4D4D"


# ============================================================
# Held-out predictions at threshold = 120
# ============================================================

cv_primary = (
    all_cv_predictions[
        float(PRIMARY_THRESHOLD)
    ]
    .copy()
)


# ============================================================
# Helper
# Participant-level values first,
# then mean ± SEM across participants
# ============================================================

def prepare_actual_predicted_summary(
    cv_data,
    phase_column,
    probability_column,
):

    data = cv_data[
        [
            "subject_id",
            "y_saccade",
            phase_column,
            probability_column,
        ]
    ].copy()

    data = data.rename(
        columns={
            phase_column: "phase",
            probability_column: "predicted_p_saccade",
        }
    )


    # --------------------------------------------------------
    # Participant-level summary
    # --------------------------------------------------------

    participant_summary = (
        data
        .groupby(
            [
                "subject_id",
                "phase",
            ],
            as_index=False,
        )
        .agg(
            observed=(
                "y_saccade",
                "mean",
            ),
            predicted=(
                "predicted_p_saccade",
                "mean",
            ),
        )
    )


    # --------------------------------------------------------
    # Across-participant summary
    # --------------------------------------------------------

    summary = (
        participant_summary
        .groupby(
            "phase",
            as_index=False,
        )
        .agg(
            observed_mean=(
                "observed",
                "mean",
            ),
            observed_sd=(
                "observed",
                "std",
            ),
            predicted_mean=(
                "predicted",
                "mean",
            ),
            predicted_sd=(
                "predicted",
                "std",
            ),
            n_participants=(
                "subject_id",
                "nunique",
            ),
        )
    )


    summary["observed_sem"] = (
        summary["observed_sd"]
        /
        np.sqrt(
            summary["n_participants"]
        )
    )

    summary["predicted_sem"] = (
        summary["predicted_sd"]
        /
        np.sqrt(
            summary["n_participants"]
        )
    )

    return summary


# ============================================================
# Summaries
# ============================================================

hmc_summary = prepare_actual_predicted_summary(
    cv_primary,
    phase_column="gaze_area_label_hmc",
    probability_column="p_saccade_hmc",
)

blended_summary = prepare_actual_predicted_summary(
    cv_primary,
    phase_column="gaze_area_label_blended",
    probability_column="p_saccade_blended",
)


# ============================================================
# Fixed phase order
# ============================================================

phase_order = [
    "simulation",
    "abstraction",
]


def order_phases(df):

    return (
        df
        .set_index("phase")
        .reindex(phase_order)
        .reset_index()
    )


hmc_plot = order_phases(
    hmc_summary
)

blended_plot = order_phases(
    blended_summary
)


# ============================================================
# Extract values
# ============================================================

hmc_observed_mean = (
    hmc_plot["observed_mean"].to_numpy()
)

hmc_observed_sem = (
    hmc_plot["observed_sem"].to_numpy()
)

hmc_predicted_mean = (
    hmc_plot["predicted_mean"].to_numpy()
)

hmc_predicted_sem = (
    hmc_plot["predicted_sem"].to_numpy()
)


blended_observed_mean = (
    blended_plot["observed_mean"].to_numpy()
)

blended_observed_sem = (
    blended_plot["observed_sem"].to_numpy()
)

blended_predicted_mean = (
    blended_plot["predicted_mean"].to_numpy()
)

blended_predicted_sem = (
    blended_plot["predicted_sem"].to_numpy()
)


# ============================================================
# X positions
#
# Equal model halves:
#
# Meta-control:
#   0.5 = Simulation
#   1.5 = Abstraction
#
# Blended:
#   2.5 = Simulation
#   3.5 = Abstraction
# ============================================================

HMC_X = np.array([
    0.5,
    1.5,
])

BLENDED_X = np.array([
    2.5,
    3.5,
])

HMC_CENTER = 1.0
BLENDED_CENTER = 3.0

DIVIDER_X = 2.0


# ============================================================
# Figure
# ============================================================

fig = plt.figure(
    figsize=FIGSIZE
)

ax = fig.add_axes(
    AXES_POSITION
)


# ============================================================
# Meta-control observed
# light blue + SOLID
# ============================================================

ax.errorbar(
    HMC_X,
    hmc_observed_mean,
    yerr=hmc_observed_sem,

    color=HMC_OBSERVED_COLOR,

    linestyle="-",
    linewidth=MAIN_LINE_WIDTH,

    marker="o",
    markersize=MARKER_SIZE,
    markerfacecolor=HMC_OBSERVED_COLOR,
    markeredgecolor=HMC_OBSERVED_COLOR,

    capsize=5,
    elinewidth=ERROR_LINE_WIDTH,
    capthick=ERROR_LINE_WIDTH,

    zorder=5,
)


# ============================================================
# Meta-control predicted
# dark blue + DASHED
# ============================================================

ax.errorbar(
    HMC_X,
    hmc_predicted_mean,
    yerr=hmc_predicted_sem,

    color=HMC_PREDICTED_COLOR,

    linestyle="--",
    linewidth=MAIN_LINE_WIDTH,

    marker="o",
    markersize=MARKER_SIZE,
    markerfacecolor=HMC_PREDICTED_COLOR,
    markeredgecolor=HMC_PREDICTED_COLOR,

    capsize=5,
    elinewidth=ERROR_LINE_WIDTH,
    capthick=ERROR_LINE_WIDTH,

    zorder=6,
)


# ============================================================
# Blended observed
# light gray + SOLID
# ============================================================

ax.errorbar(
    BLENDED_X,
    blended_observed_mean,
    yerr=blended_observed_sem,

    color=BLENDED_OBSERVED_COLOR,

    linestyle="-",
    linewidth=MAIN_LINE_WIDTH,

    marker="o",
    markersize=MARKER_SIZE,
    markerfacecolor=BLENDED_OBSERVED_COLOR,
    markeredgecolor=BLENDED_OBSERVED_COLOR,

    capsize=5,
    elinewidth=ERROR_LINE_WIDTH,
    capthick=ERROR_LINE_WIDTH,

    zorder=5,
)


# ============================================================
# Blended predicted
# dark gray + DASHED
# ============================================================

ax.errorbar(
    BLENDED_X,
    blended_predicted_mean,
    yerr=blended_predicted_sem,

    color=BLENDED_PREDICTED_COLOR,

    linestyle="--",
    linewidth=MAIN_LINE_WIDTH,

    marker="o",
    markersize=MARKER_SIZE,
    markerfacecolor=BLENDED_PREDICTED_COLOR,
    markeredgecolor=BLENDED_PREDICTED_COLOR,

    capsize=5,
    elinewidth=ERROR_LINE_WIDTH,
    capthick=ERROR_LINE_WIDTH,

    zorder=6,
)


# ============================================================
# X axis
# ============================================================

ax.set_xlim(
    0.0,
    4.0,
)

ax.set_xticks(
    [
        0.5,
        1.5,
        2.5,
        3.5,
    ]
)

ax.set_xticklabels(
    [
        "Simulation",
        "Abstraction",
        "Simulation",
        "Abstraction",
    ],
    fontsize=TICK_SIZE,
)

ax.tick_params(
    axis="x",
    length=0,
    pad=7,
)


# ============================================================
# Model labels centered within each half
# ============================================================

ax.text(
    HMC_CENTER,
    -0.115,
    "Meta-control",
    transform=ax.get_xaxis_transform(),

    ha="center",
    va="top",

    fontsize=MODEL_LABEL_SIZE,
    fontweight="bold",

    color=HMC_PREDICTED_COLOR,
)

ax.text(
    BLENDED_CENTER,
    -0.115,
    "Blended",
    transform=ax.get_xaxis_transform(),

    ha="center",
    va="top",

    fontsize=MODEL_LABEL_SIZE,
    fontweight="bold",

    color=BLENDED_PREDICTED_COLOR,
)


# ============================================================
# Divider between model halves
# ============================================================

ax.axvline(
    DIVIDER_X,

    color="black",
    linewidth=1.0,
    alpha=0.18,

    zorder=1,
)


# ============================================================
# Title + Y label
# ============================================================

ax.set_title(
    "Observed vs predicted saccade probability",
    fontsize=FIGURE_TITLE_SIZE,
    pad=TITLE_PAD,
)

ax.set_ylabel(
    "Saccade probability",
    fontsize=AXIS_LABEL_SIZE,
)

ax.yaxis.set_label_coords(
    -0.09,
    0.5,
)


# ============================================================
# Y axis
# Tick every 0.05
# ============================================================

ax.set_ylim(
    0.65,
    0.85,
)

ax.set_yticks(
    np.arange(
        0.65,
        0.851,
        0.05,
    )
)

ax.tick_params(
    axis="y",
    labelsize=TICK_SIZE,
)


# ============================================================
# Legend
#
# Use long, thick line samples so solid vs dashed
# is immediately visible.
# ============================================================

from matplotlib.lines import Line2D


legend_handles = [

    Line2D(
        [0],
        [0],

        color="#A9BED0",
        linestyle="-",
        linewidth=3.6,

        marker="o",
        markersize=8,

        markerfacecolor="#A9BED0",
        markeredgecolor="#A9BED0",

        label="Observed",
    ),

    Line2D(
        [0],
        [0],

        color="#364F63",
        linestyle="--",
        linewidth=3.6,

        marker="o",
        markersize=8,

        markerfacecolor="#364F63",
        markeredgecolor="#364F63",

        label="Predicted",
    ),
]


ax.legend(
    handles=legend_handles,

    loc="upper left",

    # Longer legend line samples make dashes much clearer
    handlelength=4.2,
    handletextpad=0.9,

    frameon=True,
    framealpha=0.92,
    facecolor="white",
    edgecolor="black",

    fontsize=LEGEND_TEXT_SIZE,
)


# ============================================================
# Full boxed frame
# ============================================================

for spine in ax.spines.values():

    spine.set_visible(True)
    spine.set_linewidth(1.1)
    spine.set_color("black")


# ============================================================
# Grid
# ============================================================

ax.grid(
    axis="y",
    alpha=0.22,
    zorder=0,
)


plt.show()


# ============================================================
# Numerical summaries
# ============================================================

print("Meta-control")
display(
    hmc_summary
)

print("Blended")
display(
    blended_summary
)